# Creating df for research paper code

In [19]:
# pip install imblearn

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import tools.helpers as hh  
import tools.visualisers_ as vv
import tools.filter_helpers as fh

DATA_ROOT = Path("/Users/gnaanikko.pa/Documents/Academic /MIMIC/model_building")
PARQ_DIR = DATA_ROOT / "parq"
OUTPUT_DIR = DATA_ROOT / "outputs"

In [2]:
import importlib
importlib.reload(vv)

<module 'tools.visualisers_' from '/Users/gnaanikko.pa/Documents/Academic /MIMIC/model_building/tools/visualisers_.py'>

In [3]:
from sqlalchemy import create_engine

In [4]:
user = "postgres"
password = "mimic"
host = "localhost"   
port = "5432"
database = "mimiciv"

In [5]:
engine = create_engine(f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{database}")


# Load Cohorts

In [6]:
# Load core datasets for the respiratory infection ICU cohort
resp_inf_cohort = pd.read_parquet(PARQ_DIR / "resp_inf_cohort_23Jan26_1953.parquet")
df_resp_inf_icu_final = pd.read_parquet(PARQ_DIR / "df_resp_inf_icu_final.parq")

# Microbiology, medications, prescriptions, inputs, and labs
# resp_micro = pd.read_parquet(PARQ_DIR / "resp_micro_hosp_icu_inf_df_23Jan26_2050.parquet")
resp_micro = pd.read_parquet(PARQ_DIR / "micro_df_26Feb26_0415.parquet")

resp_emar = pd.read_parquet(PARQ_DIR / "resp_emar_hosp_icu_inf_results_df_24Jan26_0507.parquet")
resp_pres = pd.read_parquet(PARQ_DIR / "resp_pres_hosp_icu_inf_results_df24Jan26_0526.parquet")
resp_input = pd.read_parquet(PARQ_DIR / "resp_inputevents_icu_inf_results_df.parquet")
resp_lab = pd.read_parquet(PARQ_DIR / "resp_lab_test_icu_inf_df.parq")


In [7]:
hh.dxx(resp_inf_cohort)

11.9k Unique Patient IDs (11905)
13.6k Unique Admission IDs (13611)
16.4k Unique ICU Stay IDs (16438)
18.1k Rows, shape: (18109, 19)



,subject_id,gender,anchor_age,anchor_year,hadm_id,hospital_admit_time,hospital_discharge_time,hospital_death_time,stay_id,icu_admit_time,icu_discharge_time,icu_los_days,admission_type,admission_location,discharge_location,race,seq_num,icd_version,icd_code
dtype,int64,object,int64,int64,int64,datetime64[ns],datetime64[ns],datetime64[ns],int64,datetime64[ns],datetime64[ns],float64,object,object,object,object,int64,int64,object
NotNA | NA,18109 | 0,18109 | 0,18109 | 0,18109 | 0,18109 | 0,18109 | 0,18109 | 0,4055 | 14054,18109 | 0,18109 | 0,18099 | 10,18099 | 10,18109 | 0,18109 | 0,18074 | 35,18109 | 0,18109 | 0,18109 | 0,18109 | 0
nunique,11905,2,73,96,13611,13607,13598,2831,16438,16438,16430,16246,9,11,14,33,38,2,142
0,10003637,M,57,2145,28317408,2150-05-14 19:51:00,2150-05-22 16:25:00,2150-05-22 16:25:00,32824762,2150-05-16 07:30:28,2150-05-22 18:38:55,6.464201,EW EMER.,WALK-IN/SELF REFERRAL,DIED,PORTUGUESE,3,10,J1289
1,10010888,M,43,2174,20162667,2174-01-09 22:19:00,2174-01-27 16:00:00,NaT,33318955,2174-01-09 00:21:00,2174-01-18 00:48:41,9.019225,EW EMER.,TRANSFER FROM HOSPITAL,HOME HEALTH CARE,WHITE,2,10,J1282
2,10011938,F,57,2121,23798746,2133-08-13 09:48:00,2133-10-05 19:27:00,NaT,31780787,2133-08-13 09:48:53,2133-09-01 18:22:42,19.356817,EW EMER.,TRANSFER FROM SKILLED NURSING FACILITY,REHAB,WHITE - OTHER EUROPEAN,2,10,J15212


In [8]:
hh.dxx(df_resp_inf_icu_final)

11.9k Unique Patient IDs (11905)
13.6k Unique Admission IDs (13611)
16.4k Unique ICU Stay IDs (16438)
18.1k Rows, shape: (18109, 6)



,subject_id,hadm_id,stay_id,seq_num,icd_version,icd_code
dtype,int64,int64,int64,int64,int64,object
NotNA | NA,18109 | 0,18109 | 0,18109 | 0,18109 | 0,18109 | 0,18109 | 0
nunique,11905,13611,16438,38,2,142
0,10001843,26133978,39698942,4,10,J189
1,10001884,26184834,37510196,19,10,J0190
2,10002155,20345487,32358465,2,9,486


## Get Target Time 
Target time = AST sample time + 24 hours 

In [9]:
df_resp_inf_icu_hadm_ids=tuple(df_resp_inf_icu_final.hadm_id.unique().tolist())


In [10]:
df_resp_inf_icu_sub_ids=tuple(df_resp_inf_icu_final.subject_id.unique().tolist())


In [11]:
# query=f"""SELECT 
#     p.subject_id,
#     p.gender,
#     p.anchor_age,
#     p.anchor_year,
#     a.hadm_id,
#     a.admittime AS hospital_admit_time,
#     a.dischtime AS hospital_discharge_time,
#     a.deathtime AS hospital_death_time,
#     i.stay_id,
#     i.intime AS icu_admit_time,
#     i.outtime AS icu_discharge_time,
#     i.los AS icu_los_days,
#     a.admission_type,
#     a.admission_location,
#     a.discharge_location,
#     a.race
# FROM mimiciv_hosp.patients p
# JOIN mimiciv_hosp.admissions a
#   ON p.subject_id = a.subject_id
# LEFT JOIN mimiciv_icu.icustays i
#   ON a.hadm_id = i.hadm_id
#  where 
#  a.hadm_id IS NOT NULL 
# AND  p.subject_id in {df_resp_inf_icu_sub_ids};  """

# all_adm_df = pd.read_sql_query(sql=query, con=engine)

In [12]:
# hh.dxx(all_adm_df)

In [13]:
# hh.parq(all_adm_df,'all_adm_df_')

In [14]:
all_adm_df= hh.load_data('./parq/all_adm_df_27Feb26_1738.parquet')

In [15]:
# hh.dxx(df_admissions_new)

In [16]:
# df_resp_inf_icu_final

In [17]:
# query = f"""
# -- Get respiratory infection ICU patients with any microbiology record during ICU stay (raw data)
# SELECT 
#     icu.subject_id,
#     icu.hadm_id,
#     icu.stay_id,
#     icu.first_careunit,
#     icu.last_careunit,
#     icu.intime,
#     icu.outtime,
#     icu.los,
#     me.microevent_id,
#     me.chartdate,
#     me.charttime,
#     me.spec_type_desc,
#     me.storetime,
#     me.test_name,
#     me.org_name,
#     me.ab_name,
#     me.interpretation,
#     me.dilution_value,
#     me.comments
# FROM mimiciv_icu.icustays icu
# INNER JOIN mimiciv_hosp.microbiologyevents me 
#     ON icu.subject_id = me.subject_id 
#     AND icu.hadm_id = me.hadm_id
# WHERE icu.subject_id IN {df_resp_inf_icu_sub_ids}
#      AND icu.hadm_id IS NOT NULL 
# ;



# """


# all_micro_icu_df = pd.read_sql_query(query, con=engine)

In [19]:
# hh.dx(all_micro_icu_df)

In [20]:
# hh.parq(all_micro_icu_df, 'all_micro_df_')

In [21]:
all_micro_icu_df= hh.load_data('./parq/all_micro_df_26Feb26_0625.parquet')

In [22]:
hh.dx(all_micro_icu_df)

11.5k Unique Patient IDs (11462)
18.9k Unique Admission IDs (18878)
22.4k Unique ICU Stay IDs (22440)
519.5k Rows, shape: (519525, 19)



### VITALS

In [23]:
# query = f"""SELECT
#     icu.subject_id,
#     icu.hadm_id,
#     icu.stay_id,
#     icu.intime,
#     icu.outtime,
#     ce.itemid,
#     ce.charttime,
#     ce.valuenum

# FROM mimiciv_icu.icustays icu
# JOIN mimiciv_icu.chartevents ce
#     ON icu.stay_id = ce.stay_id

# WHERE
#     icu.hadm_id IN {df_resp_inf_icu_hadm_ids}
#     AND icu.hadm_id IS NOT NULL
#     AND ce.valuenum IS NOT NULL
#     AND ce.itemid IN (
#         -- Heart Rate
#         220045,

#         -- Blood Pressure (Non-Invasive)
#         220179,  -- Non Invasive BP systolic
#         220180,  -- Non Invasive BP diastolic
#         220181,  -- Non Invasive BP mean

#         -- Blood Pressure (Arterial / Invasive)
#         220050,  -- Arterial BP systolic
#         220051,  -- Arterial BP diastolic
#         220052,  -- Arterial BP mean
#         225309,  -- ART BP Systolic
#         225310,  -- ART BP Diastolic
#         225312,  -- ART BP Mean

#         -- Respiratory Rate
#         220210,  -- Respiratory Rate
#         224690,  -- Respiratory Rate (Total)

#         -- Temperature
#         223762,  -- Temperature Celsius
#         223761,  -- Temperature Fahrenheit

#         -- SpO2
#         220277,  -- SpO2 peripheral

#         -- Glucose
#         225664,  -- Glucose finger stick
#         220621,  -- Glucose (serum)
#         226537   -- Glucose (whole blood)
#     )
#     -- Sanity bounds (from official MIMIC-IV vitalsign.sql)
#     AND (
#         (ce.itemid = 220045  AND ce.valuenum > 0   AND ce.valuenum < 300)   -- heart_rate
#         OR (ce.itemid IN (220179, 220050, 225309) AND ce.valuenum > 0 AND ce.valuenum < 400)  -- sbp
#         OR (ce.itemid IN (220180, 220051, 225310) AND ce.valuenum > 0 AND ce.valuenum < 300)  -- dbp
#         OR (ce.itemid IN (220052, 220181, 225312) AND ce.valuenum > 0 AND ce.valuenum < 300)  -- mbp
#         OR (ce.itemid IN (220210, 224690) AND ce.valuenum > 0 AND ce.valuenum < 70)   -- resp_rate
#         OR (ce.itemid = 223762 AND ce.valuenum > 10  AND ce.valuenum < 50)   -- temp celsius
#         OR (ce.itemid = 223761 AND ce.valuenum > 70  AND ce.valuenum < 120)  -- temp fahrenheit
#         OR (ce.itemid = 220277  AND ce.valuenum > 0  AND ce.valuenum <= 100) -- spo2
#         OR (ce.itemid IN (225664, 220621, 226537) AND ce.valuenum > 0)       -- glucose
#     )
# ;
# """

# vitals_df = pd.read_sql_query(query, con=engine)
# print(f"Vitals query returned: {vitals_df.shape}")


In [29]:
hh.dxx(vitals_df)

11.9k Unique Patient IDs (11905)
13.6k Unique Admission IDs (13611)
16.4k Unique ICU Stay IDs (16437)
17.2M Rows, shape: (17209316, 9)



,subject_id,hadm_id,stay_id,intime,outtime,itemid,charttime,valuenum,d_vital_name
dtype,int64,int64,int64,datetime64[ns],datetime64[ns],int64,datetime64[ns],float64,object
NotNA | NA,17209316 | 0,17209316 | 0,17209316 | 0,17209316 | 0,17190513 | 18803,17209316 | 0,17209316 | 0,17209316 | 0,17209316 | 0
nunique,11905,13611,16437,16437,16429,18,2163846,1452,8
0,10008924,23676183,30244392,2139-07-09 23:42:19,2139-07-11 17:04:07,220045,2139-07-09 23:42:00,97.000000,heart_rate
1,10001843,26133978,39698942,2134-12-05 18:50:03,2134-12-06 14:38:26,220045,2134-12-06 07:00:00,129.000000,heart_rate
2,10001843,26133978,39698942,2134-12-05 18:50:03,2134-12-06 14:38:26,220210,2134-12-06 07:00:00,18.000000,respiratory_rate


In [25]:
hh.dxx(vitals_df)

NameError: name 'vitals_df' is not defined

In [28]:
vital_item_map = {
    # Heart Rate
    220045: "heart_rate",

    # Blood Pressure — all variants map to same canonical name
    220179: "systolic_bp",   # Non-Invasive
    220050: "systolic_bp",   # Arterial
    225309: "systolic_bp",   # ART

    220180: "diastolic_bp",  # Non-Invasive
    220051: "diastolic_bp",  # Arterial
    225310: "diastolic_bp",  # ART

    220181: "mean_arterial_pressure",  # Non-Invasive
    220052: "mean_arterial_pressure",  # Arterial
    225312: "mean_arterial_pressure",  # ART

    # Respiratory Rate
    220210: "respiratory_rate",
    224690: "respiratory_rate",  # Total

    # Temperature — both map to same name; Fahrenheit converted below
    223762: "temperature_c",  # already Celsius
    223761: "temperature_c",  # Fahrenheit — convert below

    # SpO2
    220277: "spo2",

    # Glucose
    225664: "glucose",  # finger stick
    220621: "glucose",  # serum
    226537: "glucose",  # whole blood
}

vitals_df["d_vital_name"] = vitals_df["itemid"].map(vital_item_map)

# Convert Fahrenheit (itemid 223761) to Celsius
f_mask = vitals_df["itemid"] == 223761
vitals_df.loc[f_mask, "valuenum"] = (
    (vitals_df.loc[f_mask, "valuenum"] - 32) / 1.8
)

print(f"Vitals shape: {vitals_df.shape}")
print(f"\nPer-vital counts:")
print(vitals_df["d_vital_name"].value_counts())
print(f"\nPer-vital unique hadm_ids:")
print(vitals_df.groupby("d_vital_name")["hadm_id"].nunique())


Vitals shape: (17209316, 9)

Per-vital counts:
d_vital_name
respiratory_rate          2876196
mean_arterial_pressure    2645310
systolic_bp               2644994
diastolic_bp              2644439
heart_rate                2599310
spo2                      2565369
temperature_c              729325
glucose                    504373
Name: count, dtype: int64

Per-vital unique hadm_ids:
d_vital_name
diastolic_bp              13602
glucose                   13439
heart_rate                13611
mean_arterial_pressure    13600
respiratory_rate          13607
spo2                      13601
systolic_bp               13602
temperature_c             13573
Name: hadm_id, dtype: int64


In [26]:
# hh.parq(vitals_df, 'vitals_df')

In [27]:
vitals_df=hh.load_data('./parq/vitals_df29Mar26_2344.parquet')
# vitals_df=hh.load_data('./parq/vitals_df27Feb26_1808.parquet')


In [30]:
vitals_df[vitals_df['d_vital_name']=='spo2'].valuenum.median()

np.float64(97.0)

In [31]:
# hh.dxx(resp_micro_hosp_icu_inf_df)

## AST results

In [32]:
all_ast_df=all_micro_icu_df.dropna(subset=["org_name","ab_name","interpretation"])

In [33]:
# hh.parq(all_ast_df,'all_ast_df')

In [34]:
all_ast_df=hh.load_data('./parq/all_ast_df27Feb26_1751.parquet')

In [35]:
hh.dx(resp_micro)

11.3k Unique Patient IDs (11348)
12.9k Unique Admission IDs (12922)
15.7k Unique ICU Stay IDs (15723)
421.7k Rows, shape: (421739, 19)



In [36]:
hh.dx(all_ast_df)

4.7k Unique Patient IDs (4680)
6.0k Unique Admission IDs (5971)
7.8k Unique ICU Stay IDs (7838)
160.4k Rows, shape: (160393, 20)



## Index culture timing (`time_df`)

**Authoritative definition:** run **`02_03_Feature_space_clean.ipynb`** and use its output parquet (`time_df_after_first_icu_*.parquet`).

That pipeline:

- Drops a **hospital admission** if **any** respiratory AST has `charttime` **before** `first_icu_admit_time` (earliest ICU `intime` for that `hadm_id`).
- Sets **`first_ast_time`** to the **earliest** remaining culture with `charttime ≥ first_icu_admit_time`.
- Aligns **`stay_id`** / ICU window when the sample falls inside an ICU interval; otherwise flags `ast_inside_icu_stay` (see clean notebook).

**This notebook** only **loads** the newest matching file into `time_df` so later cells that still reference `time_df` keep working. Re-run the clean notebook after changing cohort or timing rules.



In [37]:
hh.dx(all_micro_icu_df)

11.5k Unique Patient IDs (11462)
18.9k Unique Admission IDs (18878)
22.4k Unique ICU Stay IDs (22440)
519.5k Rows, shape: (519525, 19)



In [38]:
all_ast_df.isna().sum()

subject_id            0
hadm_id               0
stay_id               0
first_careunit        0
last_careunit         0
intime                0
outtime               0
los                   0
microevent_id         0
chartdate             0
charttime             0
spec_type_desc        0
storetime            92
test_name             0
org_name              0
ab_name               0
interpretation        0
dilution_value     7884
comments          68521
combined_col          0
dtype: int64

In [39]:
hh.dx(all_ast_df)

4.7k Unique Patient IDs (4680)
6.0k Unique Admission IDs (5971)
7.8k Unique ICU Stay IDs (7838)
160.4k Rows, shape: (160393, 20)



In [40]:
resp_ast_df= hh.df_subset(all_ast_df,resp_inf_cohort,by_col='hadm_id')

In [41]:
hh.dxx(resp_ast_df)

4.1k Unique Patient IDs (4079)
4.4k Unique Admission IDs (4426)
5.9k Unique ICU Stay IDs (5914)
127.9k Rows, shape: (127899, 20)



,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los,microevent_id,chartdate,charttime,spec_type_desc,storetime,test_name,org_name,ab_name,interpretation,dilution_value,comments,combined_col
dtype,int64,int64,int64,object,object,datetime64[ns],datetime64[ns],float64,int64,datetime64[ns],datetime64[ns],object,datetime64[ns],object,object,object,object,float64,object,object
NotNA | NA,127899 | 0,127899 | 0,127899 | 0,127899 | 0,127899 | 0,127899 | 0,127899 | 0,127899 | 0,127899 | 0,127899 | 0,127899 | 0,127899 | 0,127807 | 92,127899 | 0,127899 | 0,127899 | 0,127899 | 0,121517 | 6382,78969 | 48930,127899 | 0
nunique,4079,4426,5914,14,14,5914,5914,5900,84584,6889,8419,34,8508,14,136,46,5,26,47,1855
14,10005817,28661809,31316840,Medical/Surgical Intensive Care Unit (MICU/SICU),Medical/Surgical Intensive Care Unit (MICU/SICU),2135-01-03 21:55:32,2135-01-19 21:16:23,15.972812,2605,2135-01-08 00:00:00,2135-01-08 15:30:00,MRSA SCREEN,2135-01-12 10:47:00,MRSA SCREEN,STAPH AUREUS COAG +,ERYTHROMYCIN,R,nan,None,"['STAPH AUREUS COAG +', 'ERYTHROMYCIN', 'R']"
15,10005817,28661809,31316840,Medical/Surgical Intensive Care Unit (MICU/SICU),Medical/Surgical Intensive Care Unit (MICU/SICU),2135-01-03 21:55:32,2135-01-19 21:16:23,15.972812,2606,2135-01-08 00:00:00,2135-01-08 15:30:00,MRSA SCREEN,2135-01-12 10:47:00,MRSA SCREEN,STAPH AUREUS COAG +,CLINDAMYCIN,R,nan,None,"['STAPH AUREUS COAG +', 'CLINDAMYCIN', 'R']"
16,10005817,28661809,31316840,Medical/Surgical Intensive Care Unit (MICU/SICU),Medical/Surgical Intensive Care Unit (MICU/SICU),2135-01-03 21:55:32,2135-01-19 21:16:23,15.972812,2607,2135-01-08 00:00:00,2135-01-08 15:30:00,MRSA SCREEN,2135-01-12 10:47:00,MRSA SCREEN,STAPH AUREUS COAG +,TRIMETHOPRIM/SULFA,S,0.500000,None,"['STAPH AUREUS COAG +', 'TRIMETHOPRIM/SULFA', 'S']"


In [42]:
# Load canonical timing table (newest stamp wins)
_after_first = sorted(PARQ_DIR.glob("time_df_after_first_icu*.parquet"), key=lambda p: p.stat().st_mtime, reverse=True)
_fallback = sorted(PARQ_DIR.glob("time_df*.parquet"), key=lambda p: p.stat().st_mtime, reverse=True)
_paths = _after_first if _after_first else _fallback
if not _paths:
    raise FileNotFoundError(
        "No time_df parquet in PARQ_DIR. Run 02_03_Feature_space_clean.ipynb first "
        "(or place a time_df_after_first_icu_*.parquet under parq/)."
    )
_time_path = _paths[0]
time_df = pd.read_parquet(_time_path)
print("time_df loaded:", _time_path.name, "| shape:", time_df.shape)
hh.dxx(time_df)


time_df loaded: time_df_after_first_icu12Apr26_1615.parquet | shape: (3882, 14)
3.6k Unique Patient IDs (3588)
3.9k Unique Admission IDs (3882)
3.3k Unique ICU Stay IDs (3311)
3.9k Rows, shape: (3882, 14)



,subject_id,hadm_id,stay_id,hospital_admit_time,hospital_discharge_time,hospital_death_time,icu_admit_time,icu_discharge_time,first_icu_admit_time,first_ast_time,ast_inside_icu_stay,hours_adm_to_ast,hours_icu_to_ast,hours_first_icu_admit_to_ast
dtype,int64,int64,float64,datetime64[ns],datetime64[ns],datetime64[ns],datetime64[ns],datetime64[ns],datetime64[ns],datetime64[ns],bool,float64,float64,float64
NotNA | NA,3882 | 0,3882 | 0,3311 | 571,3882 | 0,3882 | 0,915 | 2967,3311 | 571,3311 | 571,3882 | 0,3882 | 0,3882 | 0,3882 | 0,3882 | 0,3882 | 0
nunique,3588,3882,3312,3880,3880,916,3312,3312,3882,3882,2,3264,3655,3657
0,18172155,20009330,36841282.000000,2144-01-01 00:33:00,2144-01-09 21:07:00,NaT,2144-01-01 04:26:36,2144-01-06 22:46:18,2144-01-01 04:26:36,2144-01-01 11:53:00,True,11.333333,7.440000,7.440000
1,19318312,20013496,32840324.000000,2161-11-04 04:33:00,2161-12-01 17:15:00,NaT,2161-11-04 06:58:00,2161-11-22 18:27:35,2161-11-04 06:58:00,2161-11-04 22:30:00,True,17.950000,15.533333,15.533333
2,17333498,20015712,nan,2162-05-14 03:54:00,2162-06-11 13:55:00,NaT,NaT,NaT,2162-05-14 04:28:00,2162-05-22 16:00:00,False,204.100000,203.533333,203.533333


## Demographics

In [43]:
# query = f"""
# SELECT
#     icu.subject_id,
#     icu.hadm_id,
#     icu.stay_id,

#     -- Patient level
#     p.gender,
#     p.anchor_age,
#     p.anchor_year,
#     p.anchor_year_group,

#     -- Admission level
#     a.admittime AS hospital_admit_time,
#     a.dischtime AS hospital_discharge_time,
#     a.admission_type,
#     a.admission_location,
#     a.discharge_location,
#     a.insurance,
#     a.race,
#     a.marital_status,
#     a.language,

#     -- ICU level
#     icu.first_careunit,
#     icu.last_careunit,
#     icu.intime,
#     icu.outtime,
#     icu.los

# FROM mimiciv_icu.icustays icu
# LEFT JOIN mimiciv_hosp.admissions a
#     ON icu.hadm_id = a.hadm_id
# LEFT JOIN mimiciv_hosp.patients p
#     ON icu.subject_id = p.subject_id
# WHERE icu.hadm_id IN {df_resp_inf_icu_hadm_ids}
#      AND icu.hadm_id IS NOT NULL 
# ;



# """


# demo_df = pd.read_sql_query(query, con=engine)

In [44]:
# hh.dx(demo_df)

In [45]:
demo_df= hh.load_data('./parq/demo_df_26Feb26_1629.parquet')

In [46]:
# hh.df_sample(demo_df,by_col='subject_id',item=11281568)

In [47]:
# # ------------------------------------------------------------
# # STEP 1: Extract admission year
# # ------------------------------------------------------------
# demo_df["admission_year"] = (
#     demo_df["hospital_admit_time"].dt.year
# )

# # ------------------------------------------------------------
# # STEP 2: Calculate age at admission
# # ------------------------------------------------------------
# demo_df["age_at_admission"] = (
#     demo_df["anchor_age"] +
#     (demo_df["admission_year"] -
#      demo_df["anchor_year"])
# )

In [48]:
hh.df_sample(demo_df,by_col='subject_id',item=11281568)

,subject_id,hadm_id,stay_id,gender,age_at_admission,anchor_age,anchor_year,anchor_year_group,hospital_admit_time,hospital_discharge_time,...,insurance,race,marital_status,language,first_careunit,last_careunit,intime,outtime,los,admission_year
2045,11281568,20006409,39966506,M,54,50,2122,2011 - 2013,2126-10-28 00:28:00,2126-11-04 19:00:00,...,Medicare,WHITE,SINGLE,English,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2126-10-28 02:24:00,2126-10-30 19:06:19,2.696053,2126
2046,11281568,20006409,37402901,M,54,50,2122,2011 - 2013,2126-10-28 00:28:00,2126-11-04 19:00:00,...,Medicare,WHITE,SINGLE,English,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2126-10-31 03:15:33,2126-11-04 21:14:17,4.749120,2126
2047,11281568,20090713,37599276,M,51,50,2122,2011 - 2013,2123-06-14 17:38:00,2123-06-23 17:23:00,...,Medicare,WHITE,SINGLE,English,Medical/Surgical Intensive Care Unit (MICU/SICU),Medical/Surgical Intensive Care Unit (MICU/SICU),2123-06-14 19:45:00,2123-06-18 20:08:18,4.016181,2123
2048,11281568,20479007,39430734,M,55,50,2122,2011 - 2013,2127-09-04 14:04:00,2127-09-10 15:03:00,...,Medicare,WHITE,SINGLE,English,Medical/Surgical Intensive Care Unit (MICU/SICU),Medical/Surgical Intensive Care Unit (MICU/SICU),2127-09-04 19:31:00,2127-09-08 23:24:41,4.162280,2127
2049,11281568,21090555,31254034,M,53,50,2122,2011 - 2013,2125-02-25 08:32:00,2125-03-05 18:41:00,...,Medicare,WHITE,SINGLE,English,Medical/Surgical Intensive Care Unit (MICU/SICU),Medical/Surgical Intensive Care Unit (MICU/SICU),2125-02-25 10:54:41,2125-02-25 19:38:29,0.363750,2125
2050,11281568,21098084,39450375,M,53,50,2122,2011 - 2013,2125-09-03 15:08:00,2125-09-13 11:45:00,...,Medicare,WHITE,SINGLE,English,Medical/Surgical Intensive Care Unit (MICU/SICU),Medical/Surgical Intensive Care Unit (MICU/SICU),2125-09-03 17:30:00,2125-09-04 17:13:00,0.988194,2125
2051,11281568,21098084,36029370,M,53,50,2122,2011 - 2013,2125-09-03 15:08:00,2125-09-13 11:45:00,...,Medicare,WHITE,SINGLE,English,Medical/Surgical Intensive Care Unit (MICU/SICU),Medical/Surgical Intensive Care Unit (MICU/SICU),2125-09-06 21:38:50,2125-09-13 12:29:14,6.618333,2125
2052,11281568,21354243,33669622,M,54,50,2122,2011 - 2013,2126-08-09 16:51:00,2126-08-14 15:05:00,...,Medicare,WHITE,SINGLE,English,Medical/Surgical Intensive Care Unit (MICU/SICU),Medical/Surgical Intensive Care Unit (MICU/SICU),2126-08-09 18:04:00,2126-08-14 15:08:00,4.877778,2126
2053,11281568,21803174,30195896,M,54,50,2122,2011 - 2013,2126-04-28 00:35:00,2126-05-01 14:44:00,...,Medicare,WHITE,SINGLE,English,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2126-04-28 02:28:00,2126-04-28 15:54:41,0.560197,2126
2054,11281568,22329565,36989710,M,55,50,2122,2011 - 2013,2127-05-30 22:06:00,2127-06-09 13:20:00,...,Medicare,WHITE,SINGLE,English,Medical/Surgical Intensive Care Unit (MICU/SICU),Medical/Surgical Intensive Care Unit (MICU/SICU),2127-05-30 23:30:41,2127-05-31 17:37:06,0.754456,2127


In [49]:
demo_df["subject_id"].value_counts()

subject_id
11281568    20
17295976    18
15573773    17
17585185    12
10253349    10
            ..
13877014     1
13877180     1
13877262     1
13878385     1
19999625     1
Name: count, Length: 11905, dtype: int64

In [50]:
demo_df.columns

Index(['subject_id', 'hadm_id', 'stay_id', 'gender', 'age_at_admission',
       'anchor_age', 'anchor_year', 'anchor_year_group', 'hospital_admit_time',
       'hospital_discharge_time', 'admission_type', 'admission_location',
       'discharge_location', 'insurance', 'race', 'marital_status', 'language',
       'first_careunit', 'last_careunit', 'intime', 'outtime', 'los',
       'admission_year'],
      dtype='object')

In [51]:
# demo_df=demo_df[['subject_id', 'hadm_id', 'stay_id', 'gender','age_at_admission', 'anchor_age',
#        'anchor_year', 'anchor_year_group', 'hospital_admit_time', 'hospital_discharge_time',
#        'admission_type', 'admission_location', 'discharge_location',
#        'insurance', 'race', 'marital_status', 'language', 'first_careunit',
#        'last_careunit', 'intime', 'outtime', 'los', 'admission_year',
#       ]]

In [52]:
# hh.parq(demo_df,'demo_df_')

In [53]:
# demo_df= hh.load_data('./parq/demo_df_26Feb26_1629.parquet')

## Target MICRO AST DF

In [58]:
hh.dxx(all_micro_icu_df)

11.5k Unique Patient IDs (11462)


18.9k Unique Admission IDs (18878)
22.4k Unique ICU Stay IDs (22440)
519.5k Rows, shape: (519525, 19)



,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los,microevent_id,chartdate,charttime,spec_type_desc,storetime,test_name,org_name,ab_name,interpretation,dilution_value,comments
dtype,int64,int64,int64,object,object,datetime64[ns],datetime64[ns],float64,int64,datetime64[ns],datetime64[ns],object,datetime64[ns],object,object,object,object,float64,object
NotNA | NA,519525 | 0,519525 | 0,519525 | 0,519525 | 0,519525 | 0,519525 | 0,519525 | 0,519525 | 0,519525 | 0,519525 | 0,519525 | 0,519525 | 0,514796 | 4729,519525 | 0,195856 | 323669,160393 | 359132,160393 | 359132,152509 | 367016,429900 | 89625
nunique,11462,18878,22440,14,14,22439,22440,22073,364102,30702,154248,75,225294,131,332,48,7,26,265
0,10005817,28661809,31316840,Medical/Surgical Intensive Care Unit (MICU/SICU),Medical/Surgical Intensive Care Unit (MICU/SICU),2135-01-03 21:55:32,2135-01-19 21:16:23,15.972812,2602,2135-01-04 00:00:00,2135-01-04 13:41:00,PLEURAL FLUID,2135-01-04 19:19:00,GRAM STAIN,None,None,None,nan,NO POLYMORPHONUCLEAR LEUKOCYTES SEEN. NO MICROORGANISMS SEEN.
1,10005817,20626031,32604416,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2132-12-15 09:29:01,2132-12-17 18:06:07,2.359097,2591,2132-12-12 00:00:00,2132-12-12 02:08:00,URINE,2132-12-17 14:25:00,URINE CULTURE,"STAPHYLOCOCCUS, COAGULASE NEGATIVE",ERYTHROMYCIN,R,8.000000,None
2,10005817,20626031,32604416,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2132-12-15 09:29:01,2132-12-17 18:06:07,2.359097,2592,2132-12-12 00:00:00,2132-12-12 02:08:00,URINE,2132-12-17 14:25:00,URINE CULTURE,"STAPHYLOCOCCUS, COAGULASE NEGATIVE",NITROFURANTOIN,S,16.000000,None


In [55]:
hh.dx(all_ast_df)

4.7k Unique Patient IDs (4680)
6.0k Unique Admission IDs (5971)
7.8k Unique ICU Stay IDs (7838)
160.4k Rows, shape: (160393, 20)



In [56]:
all_ast_df.org_name.nunique()

153

In [61]:
hh.dxx(time_df)

3.6k Unique Patient IDs (3588)
3.9k Unique Admission IDs (3882)
3.3k Unique ICU Stay IDs (3311)
3.9k Rows, shape: (3882, 14)



,subject_id,hadm_id,stay_id,hospital_admit_time,hospital_discharge_time,hospital_death_time,icu_admit_time,icu_discharge_time,first_icu_admit_time,first_ast_time,ast_inside_icu_stay,hours_adm_to_ast,hours_icu_to_ast,hours_first_icu_admit_to_ast
dtype,int64,int64,float64,datetime64[ns],datetime64[ns],datetime64[ns],datetime64[ns],datetime64[ns],datetime64[ns],datetime64[ns],bool,float64,float64,float64
NotNA | NA,3882 | 0,3882 | 0,3311 | 571,3882 | 0,3882 | 0,915 | 2967,3311 | 571,3311 | 571,3882 | 0,3882 | 0,3882 | 0,3882 | 0,3882 | 0,3882 | 0
nunique,3588,3882,3312,3880,3880,916,3312,3312,3882,3882,2,3264,3655,3657
0,18172155,20009330,36841282.000000,2144-01-01 00:33:00,2144-01-09 21:07:00,NaT,2144-01-01 04:26:36,2144-01-06 22:46:18,2144-01-01 04:26:36,2144-01-01 11:53:00,True,11.333333,7.440000,7.440000
1,19318312,20013496,32840324.000000,2161-11-04 04:33:00,2161-12-01 17:15:00,NaT,2161-11-04 06:58:00,2161-11-22 18:27:35,2161-11-04 06:58:00,2161-11-04 22:30:00,True,17.950000,15.533333,15.533333
2,17333498,20015712,nan,2162-05-14 03:54:00,2162-06-11 13:55:00,NaT,NaT,NaT,2162-05-14 04:28:00,2162-05-22 16:00:00,False,204.100000,203.533333,203.533333


### AST restricted to `time_df` cohort (`hadm_id`)

- **`res`** — all micro AST rows (`all_ast_df`) for admissions that appear in **`time_df`** (same as before, with explicit before/after counts).
- **`resp_ast_cohort`** — respiratory AST (`resp_ast_df`) intersected with **`time_df`** (addresses the larger pre-cohort table, e.g. ~5k admissions → ~3.9k).
- **`index_ast_rows`** — rows where **`charttime`** equals **`time_df.first_ast_time`** (index culture; does not require `stay_id`).


In [126]:
# --- Before: cohort sizes ---
print("=== Before restricting to time_df cohort ===")
print(
    "resp_ast_df  rows:", len(resp_ast_df),
    "| hadm_id nunique:", resp_ast_df["hadm_id"].nunique(),
)
print(
    "all_ast_df   rows:", len(all_ast_df),
    "| hadm_id nunique:", all_ast_df["hadm_id"].nunique(),
)
print("time_df rows:", len(time_df), "| hadm_id nunique:", time_df["hadm_id"].nunique())

# All micro AST lines for final admissions (used downstream as `res`)
res = hh.df_subset(all_ast_df, isin_df=time_df, by_col="hadm_id")

# Respiratory AST only, same admissions
resp_ast_cohort = hh.df_subset(resp_ast_df, isin_df=time_df, by_col="hadm_id")

print("=== After: intersected with time_df (hadm_id) ===")
print(
    "res (all_ast ∩ time_df) rows:", len(res),
    "| hadm_id nunique:", res["hadm_id"].nunique(),
)
print(
    "resp_ast_cohort rows:", len(resp_ast_cohort),
    "| hadm_id nunique:", resp_ast_cohort["hadm_id"].nunique(),
)

# Index culture: match time_df anchor to charttime (not stay_id)
_tm = time_df[["hadm_id", "first_ast_time"]].drop_duplicates("hadm_id")
_idx_base = resp_ast_cohort.merge(_tm, on="hadm_id", how="inner")
_idx_base["charttime"] = pd.to_datetime(_idx_base["charttime"])
_idx_base["first_ast_time"] = pd.to_datetime(_idx_base["first_ast_time"])
index_ast_rows = _idx_base[_idx_base["charttime"] == _idx_base["first_ast_time"]].copy()

print("=== Index culture (resp_ast, charttime == first_ast_time) ===")
print(
    "index_ast_rows rows:", len(index_ast_rows),
    "| hadm_id nunique:", index_ast_rows["hadm_id"].nunique(),
)
_missing = set(time_df["hadm_id"]) - set(index_ast_rows["hadm_id"])
print("time_df hadm_id with no index row in resp_ast_cohort:", len(_missing))
if len(_missing) and len(_missing) <= 15:
    print("  hadm_id:", sorted(_missing))


=== Before restricting to time_df cohort ===
resp_ast_df  rows: 127899 | hadm_id nunique: 4426
all_ast_df   rows: 160393 | hadm_id nunique: 5971
time_df rows: 3882 | hadm_id nunique: 3882
=== After: intersected with time_df (hadm_id) ===
res (all_ast ∩ time_df) rows: 111403 | hadm_id nunique: 3882
resp_ast_cohort rows: 111403 | hadm_id nunique: 3882
=== Index culture (resp_ast, charttime == first_ast_time) ===
index_ast_rows rows: 51688 | hadm_id nunique: 3882
time_df hadm_id with no index row in resp_ast_cohort: 0


In [69]:
len(index_ast_rows.org_name.unique())

96

In [127]:
hh.dx(index_ast_rows)

3.6k Unique Patient IDs (3588)
3.9k Unique Admission IDs (3882)
5.2k Unique ICU Stay IDs (5213)
51.7k Rows, shape: (51688, 21)



In [ ]:
# hh.parq(index_ast_rows, 'index_ast_df')

File saved at: index_ast_df13Apr26_0531.parquet


In [76]:
hh.dx(res)

3.6k Unique Patient IDs (3588)
3.9k Unique Admission IDs (3882)
5.2k Unique ICU Stay IDs (5213)
111.4k Rows, shape: (111403, 20)



In [78]:
hh.dx(resp_ast_cohort)

3.6k Unique Patient IDs (3588)
3.9k Unique Admission IDs (3882)
5.2k Unique ICU Stay IDs (5213)
111.4k Rows, shape: (111403, 20)



In [75]:
index_ast_df= hh.load_data('./parq/index_ast_df13Apr26_0531.parquet')
hh.dx(index_ast_df)

3.6k Unique Patient IDs (3588)
3.9k Unique Admission IDs (3882)
5.2k Unique ICU Stay IDs (5213)
51.7k Rows, shape: (51688, 21)



In [63]:
hh.dxx(res)

3.6k Unique Patient IDs (3588)
3.9k Unique Admission IDs (3882)
5.2k Unique ICU Stay IDs (5213)
111.4k Rows, shape: (111403, 20)



,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los,microevent_id,chartdate,charttime,spec_type_desc,storetime,test_name,org_name,ab_name,interpretation,dilution_value,comments,combined_col
dtype,int64,int64,int64,object,object,datetime64[ns],datetime64[ns],float64,int64,datetime64[ns],datetime64[ns],object,datetime64[ns],object,object,object,object,float64,object,object
NotNA | NA,111403 | 0,111403 | 0,111403 | 0,111403 | 0,111403 | 0,111403 | 0,111403 | 0,111403 | 0,111403 | 0,111403 | 0,111403 | 0,111403 | 0,111311 | 92,111403 | 0,111403 | 0,111403 | 0,111403 | 0,105885 | 5518,72431 | 38972,111403 | 0
nunique,3588,3882,5213,14,14,5213,5213,5203,73425,6083,7283,34,7354,14,115,45,5,26,47,1695
14,10005817,28661809,31316840,Medical/Surgical Intensive Care Unit (MICU/SICU),Medical/Surgical Intensive Care Unit (MICU/SICU),2135-01-03 21:55:32,2135-01-19 21:16:23,15.972812,2605,2135-01-08 00:00:00,2135-01-08 15:30:00,MRSA SCREEN,2135-01-12 10:47:00,MRSA SCREEN,STAPH AUREUS COAG +,ERYTHROMYCIN,R,nan,None,"['STAPH AUREUS COAG +', 'ERYTHROMYCIN', 'R']"
15,10005817,28661809,31316840,Medical/Surgical Intensive Care Unit (MICU/SICU),Medical/Surgical Intensive Care Unit (MICU/SICU),2135-01-03 21:55:32,2135-01-19 21:16:23,15.972812,2606,2135-01-08 00:00:00,2135-01-08 15:30:00,MRSA SCREEN,2135-01-12 10:47:00,MRSA SCREEN,STAPH AUREUS COAG +,CLINDAMYCIN,R,nan,None,"['STAPH AUREUS COAG +', 'CLINDAMYCIN', 'R']"
16,10005817,28661809,31316840,Medical/Surgical Intensive Care Unit (MICU/SICU),Medical/Surgical Intensive Care Unit (MICU/SICU),2135-01-03 21:55:32,2135-01-19 21:16:23,15.972812,2607,2135-01-08 00:00:00,2135-01-08 15:30:00,MRSA SCREEN,2135-01-12 10:47:00,MRSA SCREEN,STAPH AUREUS COAG +,TRIMETHOPRIM/SULFA,S,0.500000,None,"['STAPH AUREUS COAG +', 'TRIMETHOPRIM/SULFA', 'S']"


In [65]:
len(res.org_name.unique())

115

In [67]:
len(all_ast_df.org_name.unique())

153

In [247]:
all_ast_df.groupby('org_name').stay_id.nunique().sort_values(ascending= False)[0:50]


org_name
STAPH AUREUS COAG +                                2694
PSEUDOMONAS AERUGINOSA                             1453
ESCHERICHIA COLI                                   1431
ENTEROCOCCUS SP.                                   1205
KLEBSIELLA PNEUMONIAE                              1040
ENTEROCOCCUS FAECIUM                                532
STENOTROPHOMONAS MALTOPHILIA                        495
STAPHYLOCOCCUS, COAGULASE NEGATIVE                  458
PROTEUS MIRABILIS                                   318
SERRATIA MARCESCENS                                 310
ENTEROBACTER CLOACAE COMPLEX                        256
STREPTOCOCCUS PNEUMONIAE                            202
ENTEROCOCCUS FAECALIS                               197
ACINETOBACTER BAUMANNII COMPLEX                     192
STAPHYLOCOCCUS EPIDERMIDIS                          177
KLEBSIELLA OXYTOCA                                  166
ENTEROBACTER AEROGENES                              149
POSITIVE FOR METHICILLIN RESISTANT STAP

### df to map

In [68]:
all_micro_icu_df['combined_col']= all_micro_icu_df[['org_name', 'ab_name', 'interpretation']].values.tolist()

In [72]:
all_micro_icu_df['combined_col'] = all_micro_icu_df['combined_col'].astype(str)

In [306]:
all_ast_df= all_micro_icu_df.dropna(subset=['org_name', 'ab_name', 'interpretation'])

In [307]:
hh.dx(all_ast_df)

4.7k Unique Patient IDs (4680)
6.0k Unique Admission IDs (5971)
7.8k Unique ICU Stay IDs (7838)
160.4k Rows, shape: (160393, 20)



In [159]:
hh.dx(all_ast_df)

4.7k Unique Patient IDs (4680)
6.0k Unique Admission IDs (5971)
7.8k Unique ICU Stay IDs (7838)
160.4k Rows, shape: (160393, 20)



In [305]:
all_ast_df.test_name.value_counts()

test_name
RESPIRATORY CULTURE                      79969
URINE CULTURE                            26678
Blood Culture, Routine                   17937
WOUND CULTURE                            10895
FLUID CULTURE                             9423
REFLEX URINE CULTURE                      6418
TISSUE                                    5636
MRSA SCREEN                               1374
Fluid Culture in Bottles                   900
Staph aureus Screen                        738
R/O VANCOMYCIN RESISTANT ENTEROCOCCUS      228
BLOOD/FUNGAL CULTURE                       129
ISOLATE FOR MIC                             44
ANAEROBIC CULTURE                           16
FECAL CULTURE                                8
Name: count, dtype: int64

In [160]:
mapper_df=all_ast_df.groupby(['stay_id','org_name','storetime']).apply(lambda x:''.join(x['combined_col'])).reset_index(name ='AST_PATTERN')

/var/folders/83/nr7h56f573z_vnkxbhjqb9tc0000gr/T/ipykernel_22823/3488151554.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mapper_df=all_ast_df.groupby(['stay_id','org_name','storetime']).apply(lambda x:''.join(x['combined_col'])).reset_index(name ='AST_PATTERN')


In [1]:
mapper_df

NameError: name 'mapper_df' is not defined

In [164]:
mapper_df=mapper_df.reset_index()

In [165]:
mapper_df.loc[107]

index                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  107
stay_id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      

In [178]:
mapper_df.to_json('mapper_df_pid1.json')

In [166]:
mapper_dff=mapper_df[['index','org_name','AST_PATTERN']]

In [167]:
mapper_dff.loc[0]

index                                                                                                                                                                                                                                                                                                                                                                                                                        0
org_name                                                                                                                                                                                                                                                                                                                                                                                                  ENTEROBACTER CLOACAE
AST_PATTERN    ['ENTEROBACTER CLOACAE', 'TRIMETHOPRIM/SULFA', 'S']['ENTEROBACTER CLOACAE', 'GENTAMICIN', 'S']['ENTEROBACTER CLOACAE', 'TOBRAMYCIN', 'S']['ENTEROBACTER CLO

In [123]:
# mapper_dff.to_json('mapper_df_1.json')

In [168]:
# mapper_dff=pd.read_json('mapper_df_1.json')

In [169]:
mapper_dff.shape

(18549, 3)

In [170]:
mapper_dff.loc[153]

index                                                                                                                                                                                                                                    153
org_name                                                                                                                                                                                                               ENTEROCOCCUS FAECALIS
AST_PATTERN    ['ENTEROCOCCUS FAECALIS', 'PENICILLIN G', 'S']['ENTEROCOCCUS FAECALIS', 'AMPICILLIN', 'S']['ENTEROCOCCUS FAECALIS', 'VANCOMYCIN', 'R']['ENTEROCOCCUS FAECALIS', 'DAPTOMYCIN', 'S']['ENTEROCOCCUS FAECALIS', 'LINEZOLID', 'S']
Name: 153, dtype: object

In [171]:
# df['mapped'] = pd.Series('a;b;a;f;d'.split(';'))

In [172]:
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)

In [173]:
mapper_dff

,index,org_name,AST_PATTERN
0,0,ENTEROBACTER CLOACAE,"['ENTEROBACTER CLOACAE', 'TRIMETHOPRIM/SULFA', 'S']['ENTEROBACTER CLOACAE', 'GENTAMICIN', 'S']['ENTEROBACTER CLOACAE', 'TOBRAMYCIN', 'S']['ENTEROBACTER CLOACAE', 'CEFTAZIDIME', 'S']['ENTEROBACTER CLOACAE', 'CEFTRIAXONE', 'S']['ENTEROBACTER CLOACAE', 'CIPROFLOXACIN', 'S']['ENTEROBACTER CLOACAE', 'PIPERACILLIN', 'S']['ENTEROBACTER CLOACAE', 'CEFEPIME', 'S']['ENTEROBACTER CLOACAE', 'MEROPENEM', 'S']"
1,1,KLEBSIELLA PNEUMONIAE,"['KLEBSIELLA PNEUMONIAE', 'CEFAZOLIN', 'S']['KLEBSIELLA PNEUMONIAE', 'TRIMETHOPRIM/SULFA', 'S']['KLEBSIELLA PNEUMONIAE', 'GENTAMICIN', 'S']['KLEBSIELLA PNEUMONIAE', 'TOBRAMYCIN', 'S']['KLEBSIELLA PNEUMONIAE', 'CEFTAZIDIME', 'S']['KLEBSIELLA PNEUMONIAE', 'CEFTRIAXONE', 'S']['KLEBSIELLA PNEUMONIAE', 'CIPROFLOXACIN', 'S']['KLEBSIELLA PNEUMONIAE', 'AMPICILLIN/SULBACTAM', 'S']['KLEBSIELLA PNEUMONIAE', 'CEFUROXIME', 'S']['KLEBSIELLA PNEUMONIAE', 'PIPERACILLIN/TAZO', 'S']['KLEBSIELLA PNEUMONIAE', 'CEFEPIME', 'S']['KLEBSIELLA PNEUMONIAE', 'MEROPENEM', 'S']"
2,2,ENTEROCOCCUS SP.,"['ENTEROCOCCUS SP.', 'PENICILLIN G', 'R']['ENTEROCOCCUS SP.', 'AMPICILLIN', 'R']['ENTEROCOCCUS SP.', 'VANCOMYCIN', 'R']['ENTEROCOCCUS SP.', 'LINEZOLID', 'S']"
3,3,KLEBSIELLA PNEUMONIAE,"['KLEBSIELLA PNEUMONIAE', 'CEFAZOLIN', 'R']['KLEBSIELLA PNEUMONIAE', 'TRIMETHOPRIM/SULFA', 'R']['KLEBSIELLA PNEUMONIAE', 'NITROFURANTOIN', 'R']['KLEBSIELLA PNEUMONIAE', 'GENTAMICIN', 'S']['KLEBSIELLA PNEUMONIAE', 'TOBRAMYCIN', 'S']['KLEBSIELLA PNEUMONIAE', 'CEFTAZIDIME', 'R']['KLEBSIELLA PNEUMONIAE', 'CEFTRIAXONE', 'R']['KLEBSIELLA PNEUMONIAE', 'CIPROFLOXACIN', 'R']['KLEBSIELLA PNEUMONIAE', 'AMPICILLIN/SULBACTAM', 'R']['KLEBSIELLA PNEUMONIAE', 'CEFUROXIME', 'R']['KLEBSIELLA PNEUMONIAE', 'PIPERACILLIN/TAZO', 'R']['KLEBSIELLA PNEUMONIAE', 'CEFEPIME', 'R']['KLEBSIELLA PNEUMONIAE', 'MEROPENEM', 'S']"
4,4,KLEBSIELLA PNEUMONIAE,"['KLEBSIELLA PNEUMONIAE', 'CEFAZOLIN', 'R']['KLEBSIELLA PNEUMONIAE', 'TRIMETHOPRIM/SULFA', 'R']['KLEBSIELLA PNEUMONIAE', 'NITROFURANTOIN', 'R']['KLEBSIELLA PNEUMONIAE', 'GENTAMICIN', 'S']['KLEBSIELLA PNEUMONIAE', 'TOBRAMYCIN', 'S']['KLEBSIELLA PNEUMONIAE', 'CEFTAZIDIME', 'R']['KLEBSIELLA PNEUMONIAE', 'CEFTRIAXONE', 'R']['KLEBSIELLA PNEUMONIAE', 'CIPROFLOXACIN', 'R']['KLEBSIELLA PNEUMONIAE', 'AMPICILLIN/SULBACTAM', 'R']['KLEBSIELLA PNEUMONIAE', 'CEFUROXIME', 'R']['KLEBSIELLA PNEUMONIAE', 'PIPERACILLIN/TAZO', 'R']['KLEBSIELLA PNEUMONIAE', 'CEFEPIME', 'R']['KLEBSIELLA PNEUMONIAE', 'MEROPENEM', 'S']"
...,...,...,...
18544,18544,PSEUDOMONAS AERUGINOSA,"['PSEUDOMONAS AERUGINOSA', 'GENTAMICIN', 'S']['PSEUDOMONAS AERUGINOSA', 'TOBRAMYCIN', 'S']['PSEUDOMONAS AERUGINOSA', 'CEFTAZIDIME', 'S']['PSEUDOMONAS AERUGINOSA', 'CIPROFLOXACIN', 'R']['PSEUDOMONAS AERUGINOSA', 'PIPERACILLIN/TAZO', 'S']['PSEUDOMONAS AERUGINOSA', 'CEFEPIME', 'S']['PSEUDOMONAS AERUGINOSA', 'MEROPENEM', 'S']"
18545,18545,PSEUDOMONAS AERUGINOSA,"['PSEUDOMONAS AERUGINOSA', 'GENTAMICIN', 'S']['PSEUDOMONAS AERUGINOSA', 'TOBRAMYCIN', 'S']['PSEUDOMONAS AERUGINOSA', 'CEFTAZIDIME', 'S']['PSEUDOMONAS AERUGINOSA', 'CIPROFLOXACIN', 'R']['PSEUDOMONAS AERUGINOSA', 'PIPERACILLIN/TAZO', 'S']['PSEUDOMONAS AERUGINOSA', 'CEFEPIME', 'S']['PSEUDOMONAS AERUGINOSA', 'MEROPENEM', 'S']"
18546,18546,STAPH AUREUS COAG +,"['STAPH AUREUS COAG +', 'ERYTHROMYCIN', 'R']['STAPH AUREUS COAG +', 'CLINDAMYCIN', 'R']['STAPH AUREUS COAG +', 'TRIMETHOPRIM/SULFA', 'S']['STAPH AUREUS COAG +', 'TETRACYCLINE', 'S']['STAPH AUREUS COAG +', 'GENTAMICIN', 'S']['STAPH AUREUS COAG +', 'OXACILLIN', 'S']['STAPH AUREUS COAG +', 'LEVOFLOXACIN', 'S']"
18547,18547,STAPH AUREUS COAG +,"['STAPH AUREUS COAG +', 'ERYTHROMYCIN', 'R']['STAPH AUREUS COAG +', 'CLINDAMYCIN', 'R']['STAPH AUREUS COAG +', 'TRIMETHOPRIM/SULFA', 'S']['STAPH AUREUS COAG +', 'TETRACYCLINE', 'S']['STAPH AUREUS COAG +', 'GENTAMICIN', 'S']['STAPH AUREUS COAG +', 'OXACILLIN', 'S']['STAPH AUREUS COAG +', 'LEVOFLOXACIN', 'S']"


In [139]:
# len(pd.Series('A;B;C;D;D;D;B;E;E;A;C;C;B;F;B;F;D;D;B;F;G;B;H;B;B;I;E;E;F;F;B;F;F;F;H;H;D;D;D;D;D;D;D;D;D;E;E;E;E;E;E;E;H;I;I;J;E;I;K;B;B;L;I;B;A;C;B;B;F;F;C;I;C;B;B;B;F;I;I;B;E;H;C;L;E;E;E;C;C;I;B;I;K;K;D;D;B;L;L;H;H;H;C;A;A;B;B;B;I;F;F;B;A;L;L;L;E;A;G;B;I;I;A;A;A;M;D;E;I;E;I;L;F;B;A;A;B;H;L;L;A;B;B;I;I;I;C;B;L;I;I;J;A;C;C;B;H;D;L;L;L;L;L;L;E;E;H;H;G;E;C;B;A;A;H;B;F;F;F;F;I;I;I;F;F;F;B;B;C;L;I;L;L;L;B;I;I;I;F;H;H;C;C;C;C;B;B;F;I;B;B;E;A;F;B;H;B;E;A;C;C;B;I;I;K;B;E;E;B;F;F;C;J;J;B;L;B;D;L;H;G;I;C;B;B;E;F;C;C;L;L;L;C;C;C;F;F;E;E;B;G;F;F;C;B;L;C;C;B;F;A;B;L;F;F;F;H;F;F;B;B;F;F;A;C;C;C;D;D;D;D;D;D;D;B;L;L;E;E;F;H;E;B;B;A;I;K;K;K;K;K;K;C;E;B;L;L;L;L;L;L;A;E;H;B;B;A;A;I;J;E;G;J;C;E;D;B;D;I;E;B;L;G;I;C;C;C;C;B;B;M;H;H;H;H;B;B;B;I;I;B;F;I;A;B;L;I;I;L;E;I;E;E;L;G;G;A;G;D;C;F;F;D;A;I;F;B;B;L;L;L;E;C;B;B;B;H;H;H;B;B;B;C;C;D;D;D;D;D;M;D;D;D;D;F;F;H;F;E;B;B;F;K;K;K;K;K;K;C;E;B;L;L;L;L;L;L;A;E;H;G;I;A;A;A;B;E;D;C;I;C;E;E;E;E;E;B;E;E;E;E;A;B;B;B;I;I;F;F;F;I;D;D;H;I;J;I;I;I;F;A;A;B;H;B;C;D;B;B;B;B;B;G;C;C;C;B;E;E;E;J;L;B;I;I;I;J;C;E;K;K;K;B;E;G;A;I;B;H;C;G;B;H;G;F;H;L;L;L;L;L;L;L;F;F;C;C;D;L;I;I;B;E;E;L;L;E;E;E;E;E;B;L;I;K;F;C;I;I;I;I;I;I;I;I;I;I;H;E;L;L;C;C;B;C;L;G;D;L;L;I;B;G;C;J;H;H;H;B;G;G;F;F;F;A;A;G;G;B;B;L;E;C;C;C;C;B;L;D;B;J;J;B;C;B;L;L;L;B;B;J;B;C;C;C;C;I;I;I;A;C;D;B;B;B;B;B;B;I;I;F;G;D;L;B;H;L;L;E;A;E;L;B;I;I;H;H;E;E;E;E;E;E;F;L;H;A;L;E;E;E;E;H;C;E;G;A;C;I;C;B;I;F;F;A;B;B;F;B;A;E;G;C;D;E;I;K;I;I;D;F;L;F;L;B;I;L;L;L;L;L;L;L;L;L;L;L;L;L;L;L;L;L;L;K;C;B;B;L;E;E;E;E;E;L;A;A;B;G;L;I;I;I;I;A;A;D;A;A;D;F;K;D;L;C;L;E;E;H;H;H;H;A;B;M;L;L;H;H;G;I;I;C;L;L;L;E;E;D;D;C;L;B;L;E;J;B;B;B;B;B;L;F;E;F;H;I;I;F;E;E;E;E;E;C;E;A;A;C;I;F;J;J;A;C;C;C;D;D;D;D;D;D;D;B;L;L;E;E;F;H;L;L;L;L;L;B;L;I;F;D;F;I;I;C;C;D;D;F;F;H;B;J;G;B;C;C;B;B;D;I;C;F;E;I;B;H;H;H;H;L;L;L;B;D;I;E;H;D;C;K;C;D;L;L;L;E;G;F;I;C;D;B;B;H;H;H;I;L;L;A;A;L;D;L;C;G;L;E;E;I;F;D;F;I;A;L;H;A;E;I;A;C;G;J;L;G;B;E;G;B;I;F;B;F;I;I;B;B;B;B;B;E;E;E;E;E;A;I;D;B;B;B;B;I;I;I;I;J;C;B;L;B;B;L;D;D;D;F;B;H;B;D;I;L;B;B;E;B;B;B;B;I;G;C;B;B;B;H;B;B;L;I;I;I;I;D;I;C;J;J;A;B;G;C;I;I;F;L;G;F;B;B;L;B;E;I;J;B;C;B;E;B;E;C;B;L;E;B;K;C;B;L;B;B;F;A;A;I;I;B;C;F;C;B;L;A;C;B;G;I;H;E;E;D;E;G;C;E;L;C;E;E;E;I;A;A;B;I;I;I;I;F;B;A;D;D;I;I;F;D;L;G;C;L;H;B;E;A;G;B;E;I;E;E;A;B;C;C;C;F;C;C;E;E;E;F;I;H;H;B;I;L;E;B;C;L;G;B;I;F;B;B;B;D;B;L;L;L;B;I;I;K;B;B;I;C;G;A;D;B;I;L;L;L;L;L;L;A;H;H;H;H;H;K;K;K;K;E;L;H;J;K;K;D;D;F;B;B;L;H;F;F;I;I;I;G;F;F;F;F;F;E;E;F;B;B;B;C;E;L;B;B;B;A;M;M;K;L;E;A;A;A;C;C;E;I;I;L;D;B;E;F;F;F;G;G;B;G;L;B;B;G;I;L;C;H;F;L;L;E;E;E;L;L;L;L;L;L;L;E;E;B;F;B;G;L;B;A;F;F;B;B;I;B;D;B;F;F;I;D;B;I;A;I;J;D;F;C;D;L;L;L;L;L;I;I;I;I;H;B;F;F;B;L;L;L;L;G;E;A;B;A;E;B;B;H;H;B;E;E;C;G;E;F;E;A;C;C;C;B;E;D;B;E;C;C;C;C;D;K;K;L;L;L;E;E;E;B;A;I;H;H;H;C;B;I;C;B;B;D;C;L;A;B;B;L;L;E;E;E;H;B;L;E;C;D;E;E;L;I;D;L;L;I;B;F;H;A;A;B;K;A;F;B;B;L;A;L;L;B;E;D;B;I;M;I;I;I;I;F;M;C;E;E;E;E;H;H;H;A;A;B;D;I;I;L;L;I;I;H;D;D;D;E;M;L;E;G;D;D;D;I;E;J;L;K;K;I;A;L;E;F;F;H;L;A;B;C;C;B;B;M;H;H;H;H;K;K;K;K;K;K;F;C;C;C;B;B;B;B;B;B;B;I;I;I;I;I;F;H;B;L;G;L;L;L;L;L;L;B;I;I;I;F;I;B;B;L;L;L;C;F;F;F;E;G;G;K;C;C;L;L;M;M;B;I;I;F;D;L;C;B;I;A;C;C;I;H;E;J;I;B;F;G;B;B;D;L;L;L;I;F;F;B;F;B;D;D;B;B;B;E;I;B;B;B;G;G;D;L;L;L;H;I;C;G;C;B;B;E;L;G;B;C;C;B;L;L;E;E;F;F;B;E;E;B;B;D;D;L;C;C;C;B;A;A;A;A;B;B;A;C;B;B;L;L;L;H;H;H;H;I;G;L;L;L;E;E;B;B;J;E;F;I;I;I;I;E;E;E;E;E;L;I;M;L;E;D;L;L;E;B;G;I;H;H;H;H;G;L;I;G;B;L;L;G;I;I;C;C;C;C;C;C;D;D;D;D;D;M;D;D;G;A;B;G;K;K;K;K;G;G;L;L;L;L;L;L;L;F;L;L;H;L;H;H;C;C;C;C;B;B;F;B;B;B;A;G;C;K;L;I;I;C;C;C;C;B;I;F;L;C;L;L;L;L;L;L;L;F;F;C;B;B;C;D;D;D;D;H;B;E;E;C;C;E;B;D;B;L;I;L;L;E;E;E;E;G;L;B;B;B;B;E;E;G;G;C;D;L;L;L;A;A;H;L;B;B;B;L;E;H;H;L;I;I;I;I;I;L;L;A;L;L;E;E;I;L;F;F;E;E;E;I;I;I;C;C;G;I;I;C;M;A;B;B;C;D;D;B;B;B;B;I;E;E;E;E;G;G;D;B;I;E;F;F;F;F;B;B;F;F;D;E;E;L;I;H;B;I;C;D;D;D;B;I;G;I;F;F;F;F;I;L;E;F;F;J;B;B;B;F;F;G;G;G;A;A;C;C;G;I;I;I;I;B;D;B;B;B;I;H;B;B;J;C;B;B;B;F;D;D;F;G;H;H;I;J;D;D;I;F;G;L;A;A;A;A;I;I;G;B;C;C;C;C;B;H;H;I;I;L;E;B;A;C;L;L;L;L;I;I;C;I;H;E;J;E;E;L;L;B;E;E;E;D;L;H;G;D;B;B;B;B;E;E;E;B;L;L;F;B;G;F;F;L;L;E;I;I;J;G;B;I;E;E;E;E;F;J;B;I;A;A;I;I;B;B;I;A;C;L;B;I;L;A;H;C;C;B;B;B;B;B;B;L;L;L;L;L;L;I;I;G;B;B;I;F;I;B;L;D;E;C;B;B;B;D;B;A;A;E;E;I;B;F;I;L;G;E;E;E;G;B;B;B;B;H;H;H;F;H;H;H;H;F;B;L;L;L;L;L;L;A;H;H;H;H;H;B;C;E;H;B;L;L;I;B;E;G;D;D;A;A;A;H;E;D;B;A;A;L;I;J;H;I;L;B;L;L;L;I;F;F;G;K;C;C;B;E;E;E;E;E;E;H;I;C;B;B;E;C;B;B;L;A;I;B;B;B;L;E;E;E;H;E;D;L;L;L;L;L;L;L;L;L;B;I;H;H;H;H;H;I;B;B;B;B;A;A;I;A;B;B;I;I;C;E;F;D;B;E;G;L;L;L;L;L;C;H;L;L;H;H;G;E;A;G;B;B;F;E;A;I;C;D;G;B;I;I;C;B;L;C;B;E;B;A;A;I;I;K;A;H;L;D;C;F;F;C;F;F;H;B;A;A;B;I;A;E;L;A;A;I;F;F;D;L;B;E;E;E;B;B;D;L;L;L;L;I;B;E;E;E;B;H;H;H;C;C;A;L;L;L;H;H;H;E;F;E;B;E;E;E;E;D;L;L;G;L;L;H;B;B;C;F;I;I;H;H;A;B;D;C;B;E;E;F;B;L;L;L;E;C;B;D;D;E;E;E;E;G;C;L;L;I;I;I;D;E;J;L;I;G;B;B;E;B;B;L;F;A;F;B;F;F;B;E;E;E;D;I;G;B;D;B;B;I;B;I;E;L;L;L;L;L;L;I;E;L;B;I;C;B;L;E;L;L;I;I;I;C;C;I;I;I;F;C;C;L;L;E;H;C;C;B;B;B;B;B;B;L;L;L;L;L;L;I;I;E;D;M;M;L;E;B;B;B;B;B;E;E;B;A;E;D;H;H;H;H;H;H;H;H;H;H;H;H;H;H;G;C;C;C;C;D;D;D;D;D;D;D;D;D;D;L;L;F;B;J;D;G;F;F;D;L;B;B;B;C;B;E;E;F;B;F;F;J;B;A;A;A;D;D;D;L;A;A;K;L;L;C;F;G;B;L;F;J;A;A;F;C;B;G;L;L;E;E;E;F;A;I;K;K;D;B;B;B;I;I;C;E;E;E;E;L;D;A;C;C;D;E;C;D;E;E;D;D;B;D;C;C;L;L;L;L;L;L;F;E;D;D;B;L;F;C;B;B;E;F;D;B;B;L;A;E;E;C;C;C;C;E;E;E;C;E;B;B;G;B;F;B;H;F;F;I;L;F;E;C;C;C;G;I;G;C;L;B;G;C;I;L;C;B;B;B;A;A;H;H;B;B;L;K;C;C;I;H;H;C;I;D;A;B;C;L;L;L;L;K;D;E;E;E;E;E;E;E;E;E;E;E;H;I;I;I;F;A;A;F;B;B;I;L;L;F;F;G;G;G;C;D;D;B;H;F;B;F;I;B;C;C;B;B;H;K;L;J;L;L;E;E;E;E;E;F;L;C;L;L;L;L;L;L;L;F;B;C;E;E;G;B;B;F;F;B;K;K;B;H;H;K;F;C;I;I;I;I;I;I;I;I;I;I;K;G;G;H;A;A;A;E;H;H;B;G;H;E;L;E;E;E;E;E;G;F;F;I;I;I;F;B;B;B;B;F;J;F;F;A;A;A;B;B;D;C;I;F;G;C;C;D;A;G;G;C;C;G;G;F;F;A;G;I;I;C;B;G;B;G;L;B;J;A;I;I;C;C;B;I;L;L;L;E;E;E;E;B;F;L;E;B;E;B;L;E;B;B;I;K;D;D;D;G;G;B;B;B;G;B;L;L;A;I;C;G;B;B;I;F;H;C;L;L;L;F;F;I;B;L;C;C;L;L;L;I;I;I;D;D;D;L;L;L;L;F;B;B;L;L;E;C;B;B;K;K;C;D;D;D;D;M;L;L;L;L;L;L;L;D;B;E;I;E;E;J;B;L;L;L;L;F;F;F;F;F;H;D;D;D;D;D;E;E;L;E;E;E;E;G;D;B;I;B;B;F;L;A;C;B;I;C;L;L;L;L;L;L;E;E;F;F;F;B;I;C;C;C;E;E;C;D;C;F;G;G;F;B;K;G;C;A;F;B;B;B;L;L;F;F;F;B;L;H;B;E;E;C;B;B;I;D;K;B;E;B;C;B;B;H;F;E;E;B;D;D;C;C;C;G;G;B;B;I;I;B;I;J;J;F;L;E;C;E;L;H;G;F;F;G;D;I;A;L;L;H;E;E;E;B;G;I;M;C;C;G;I;I;E;B;I;A;B;B;B;L;E;D;B;C;C;C;C;D;E;B;B;C;B;D;B;B;D;F;G;B;I;B;E;E;F;B;D;D;B;A;B;B;D;A;D;L;F;I;K;K;C;B;L;L;G;G;F;F;F;B;I;F;A;E;B;A;G;E;E;I;L;D;A;A;A;A;A;A;A;K;K;D;E;B;A;B;D;I;I;I;I;L;E;L;L;L;L;A;B;E;F;B;B;L;L;E;B;E;E;E;E;B;M;G;L;B;B;G;I;I;E;L;L;A;A;G;G;I;E;E;E;B;L;B;L;B;F;F;F;C;C;B;F;H;I;K;K;K;I;I;L;I;B;B;G;F;B;I;A;I;I;I;I;I;B;E;K;K;E;D;B;B;B;F;F;E;E;C;L;E;E;E;E;E;A;E;E;E;I;D;B;B;C;L;C;C;B;B;L;I;E;I;A;B;B;C;D;L;L;L;A;J;G;B;B;B;L;L;I;B;B;H;I;F;F;K;K;D;D;L;L;L;H;I;I;J;E;A;G;B;B;F;C;B;L;L;L;F;F;B;B;G;D;D;C;D;L;L;L;L;L;L;E;E;A;F;L;I;C;E;E;E;E;B;B;B;B;E;E;H;C;D;J;L;L;B;B;L;L;H;J;B;B;B;F;F;D;D;B;A;C;C;I;H;D;L;L;C;L;L;B;H;E;F;F;E;H;H;E;K;K;A;B;L;L;G;A;E;F;I;I;I;C;C;C;B;B;D;K;E;C;B;I;I;A;C;C;C;F;A;B;B;F;A;L;K;E;I;G;I;I;I;I;C;E;G;B;B;I;F;A;G;G;G;I;E;E;E;E;F;F;C;B;L;C;B;J;L;I;I;L;I;B;I;F;A;A;A;A;A;G;D;B;F;F;F;F;F;C;C;A;I;C;D;D;K;A;L;A;E;A;A;D;L;B;E;E;D;L;H;B;L;L;A;J;B;I;E;E;F;F;B;L;L;L;L;G;C;I;F;I;L;B;L;L;L;I;C;I;B;B;L;H;C;C;D;E;D;C;C;B;G;E;H;H;A;L;K;L;L;L;L;L;A;A;A;A;D;I;G;L;A;B;D;B;A;E;E;C;B;G;E;E;J;K;H;E;G;G;C;B;D;E;E;E;K;B;I;D;B;A;L;F;D;G;C;L;F;F;E;I;E;B;E;I;I;H;E;H;B;B;B;B;B;B;I;I;F;I;E;B;D;E;I;L;F;L;H;L;L;L;L;H;B;G;J;B;E;I;E;C;B;D;B;B;B;A;B;I;B;D;D;B;E;E;E;E;D;D;C;C;D;B;C;L;L;B;B;D;B;D;L;L;L;L;G;A;A;L;G;I;H;I;H;I;I;E;B;I;G;H;E;E;A;B;B;B;I;I;I;E;E;F;D;H;I;A;A;A;A;A;A;C;B;D;B;I;F;A;A;D;D;C;L;L;L;L;D;C;F;F;B;B;B;B;B;B;B;K;B;D;D;B;I;I;I;I;H;H;H;H;B;F;F;F;B;L;L;L;L;L;C;B;L;I;J;D;B;L;B;L;L;L;L;I;H;H;B;F;G;B;C;E;E;E;B;E;E;I;L;B;F;D;F;B;B;B;L;L;L;E;H;C;C;B;B;B;B;B;B;L;L;L;L;L;L;I;I;G;D;C;F;I;I;L;I;B;I;F;F;F;F;G;C;C;B;H;C;C;F;B;B;L;I;B;F;B;B;I;F;I;G;C;C;C;B;E;L;D;B;B;L;L;L;H;E;E;A;A;A;A;A;G;G;E;E;E;B;F;C;B;B;B;E;I;F;B;L;B;D;A;A;A;C;B;B;B;I;H;F;I;C;C;L;L;L;L;L;H;B;A;L;B;I;I;L;C;D;A;C;C;C;G;G;B;B;I;D;F;C;B;I;I;I;I;B;D;D;H;H;L;L;I;I;L;K;K;K;A;A;L;L;E;E;F;F;F;F;F;B;A;A;E;L;I;I;F;I;I;I;I;C;G;L;L;L;L;L;L;B;J;B;B;B;I;H;E;E;I;L;H;A;A;A;A;G;B;B;B;B;C;B;A;G;B;C;B;B;B;A;A;A;M;B;K;G;B;L;F;I;E;L;L;F;L;I;I;E;E;E;E;E;E;B;B;I;F;F;L;L;A;D;E;E;F;B;L;L;I;B;B;B;L;F;L;I;H;B;B;B;B;I;F;C;L;L;E;L;L;J;A;A;A;A;C;M;G;F;F;K;I;H;E;E;D;D;E;B;I;B;B;D;L;A;D;H;B;G;B;B;L;G;B;B;E;I;G;D;B;E;E;E;I;B;L;L;D;A;I;B;I;I;G;C;C;C;C;C;B;B;E;E;E;E;E;I;A;A;E;L;L;L;L;F;F;I;E;E;B;A;H;L;C;I;F;F;B;C;C;K;L;B;B;L;F;G;B;E;B;G;F;F;A;C;C;C;C;C;C;C;C;C;F;I;I;L;E;L;L;F;F;F;F;E;G;B;I;G;B;B;I;F;I;I;E;B;D;A;D;D;I;I;E;H;E;E;E;A;A;G;I;A;I;B;L;L;L;F;F;L;E;G;I;I;I;F;F;G;G;E;L;L;E;G;G;L;L;A;I;I;F;B;E;E;E;J;I;J;E;E;C;D;D;E;F;G;F;E;E;E;E;E;J;C;E;E;E;E;E;B;I;L;L;L;L;I;D;B;B;B;I;I;I;L;B;G;D;D;A;A;A;B;L;L;A;A;A;A;E;C;F;F;B;B;B;L;I;B;I;C;D;C;L;L;L;L;H;B;B;E;D;D;L;E;A;A;C;E;F;L;L;E;C;F;F;A;C;C;C;C;C;C;C;C;C;I;I;I;I;A;I;I;B;B;A;L;E;E;E;B;I;I;L;L;H;H;D;B;L;L;L;F;C;E;H;H;H;L;I;I;I;I;D;D;I;B;K;K;E;L;I;F;D;F;F;F;F;B;B;B;B;B;I;E;H;D;F;E;I;G;B;B;B;F;F;H;C;L;L;I;H;I;A;J;D;B;C;F;H;G;L;B;A;B;B;B;I;I;I;H;H;D;I;E;I;G;I;A;L;L;F;A;G;B;L;B;B;E;E;E;B;B;L;B;B;K;K;C;B;L;L;B;I;I;E;L;I;F;F;L;B;J;J;B;D;B;L;L;L;L;E;B;A;A;A;A;A;E;I;B;B;B;B;L;L;G;F;B;B;B;A;B;E;I;C;E;C;K;G;B;A;I;B;B;L;D;B;E;B;B;F;E;L;A;I;J;L;I;I;J;I;H;B;E;B;L;G;A;B;B;B;A;A;C;B;F;B;I;B;L;B;B;C;C;D;D;D;D;D;M;D;D;D;D;F;F;H;G;K;B;L;C;D;D;G;D;E;E;E;E;C;C;C;C;C;B;B;C;F;B;I;I;D;I;D;C;C;B;E;A;G;L;F;L;F;I;F;G;D;D;E;E;E;I;K;K;K;K;K;K;K;K;K;K;C;D;D;D;B;B;L;L;L;L;L;L;L;L;H;B;A;L;H;C;C;G;D;D;D;B;C;K;L;C;B;L;L;L;L;L;L;L;L;F;B;A;G;B;B;E;E;A;A;E;E;B;B;B;A;A;A;A;F;G;F;F;F;I;A;I;A;B;C;C;I;M;L;I;B;E;E;A;G;G;F;I;I;L;L;L;L;L;E;E;E;E;E;E;G;B;E;E;E;C;C;L;L;L;L;L;L;L;F;F;L;E;H;I;F;F;B;B;C;L;L;L;L;L;L;F;C;A;G;F;H;H;A;A;A;A;F;A;F;F;F;B;D;H;C;C;H;C;L;L;L;E;I;I;I;A;B;H;L;H;A;B;J;D;F;F;M;B;F;G;E;C;C;C;L;D;D;D;L;L;L;L;D;B;F;I;C;B;D;B;I;I;G;I;B;B;B;B;E;E;A;B;A;B;D;L;I;I;I;L;C;E;I;I;I;L;L;F;D;E;E;K;K;A;H;I;I;J;L;L;C;D;L;L;I;H;L;L;L;L;L;I;G;F;C;L;L;L;L;L;L;F;E;I;E;E;D;B;B;I;J;B;I;A;E;I;D;D;H;G;A;L;L;L;L;L;H;B;B;I;H;E;I;I;H;C;C;B;B;B;B;B;B;L;L;L;L;L;L;I;I;C;F;B;I;I;D;A;B;B;I;I;I;I;I;F;H;H;H;K;B;E;E;I;C;L;E;F;B;B;B;B;I;C;B;L;L;L;I;L;F;F;F;F;F;H;F;L;A;A;G;B;L;I;I;E;L;L;A;H;A;A;C;B;E;C;B;B;L;A;A;A;C;C;I;L;B;K;C;C;D;D;L;E;I;I;H;B;I;G;G;C;C;G;G;K;C;D;D;D;D;M;L;L;L;L;L;L;L;F;F;C;C;C;I;I;F;H;H;H;D;L;L;L;I;C;C;B;L;L;L;L;L;L;L;F;F;K;I;I;I;D;D;C;C;I;D;A;F;H;H;H;H;H;F;I;F;F;J;C;B;G;G;D;L;L;L;L;L;L;L;E;E;I;C;B;E;E;I;F;B;B;B;F;I;I;I;K;B;I;E;I;B;B;B;G;I;I;I;I;D;I;F;D;B;E;G;B;L;L;F;I;I;J;E;E;I;D;L;L;L;C;B;B;B;D;A;H;H;A;B;E;E;E;D;D;D;G;B;L;L;L;L;E;E;A;D;C;I;H;L;L;E;A;L;L;A;A;C;C;C;C;C;I;B;C;D;E;C;B;B;E;E;E;E;E;E;C;A;A;A;M;L;B;E;B;A;C;B;B;I;I;I;J;G;L;L;E;H;C;I;I;A;C;B;K;K;K;D;D;E;E;L;A;E;G;B;C;C;C;C;C;F;C;C;C;G;I;C;B;G;I;B;D;G;B;C;A;A;A;D;A;A;A;A;A;G;B;B;B;B;B;B;B;B;B;B;B;L;K;C;D;L;L;L;E;A;C;C;C;L;C;C;G;G;I;I;F;F;I;C;C;C;I;I;F;H;H;H;D;E;H;H;C;C;C;C;B;B;F;G;L;D;G;B;B;L;L;L;L;E;E;H;D;D;D;E;E;J;C;D;F;C;I;D;D;D;D;C;L;A;A;B;B;L;L;L;L;L;E;E;A;A;B;B;C;G;H;H;H;H;H;H;H;H;H;H;H;H;H;H;I;I;D;G;G;I;E;C;E;E;I;F;F;B;L;L;L;L;C;A;B;I;G;B;C;J;J;B;A;C;C;B;D;D;L;B;B;B;L;I;B;B;B;A;H;H;I;I;B;D;B;B;C;F;F;L;D;D;D;E;E;E;H;C;D;C;G;D;B;B;B;B;I;I;I;G;C;C;C;B;E;B;A;I;I;L;F;I;B;B;E;L;I;B;D;D;B;E;L;I;E;K;K;K;K;I;I;B;B;L;L;H;D;G;A;A;B;E;A;L;E;H;I;L;C;L;B;C;C;H;H;L;L;E;B;F;B;E;L;I;E;J;J;G;A;B;D;D;D;E;E;E;G;G;E;H;B;L;L;H;H;L;A;F;F;B;F;A;D;D;E;E;I;I;I;I;I;G;I;K;F;C;I;I;I;I;I;I;I;I;I;I;C;B;I;I;L;B;B;I;J;A;A;I;J;B;L;C;B;B;L;B;I;C;C;C;C;I;H;H;H;B;L;G;I;I;I;G;H;I;E;C;C;F;F;C;D;M;G;K;K;K;K;G;G;L;L;L;L;L;L;L;F;L;L;L;L;L;L;A;H;H;H;H;H;A;F;G;A;A;A;A;F;C;B;E;E;F;C;B;K;L;H;H;C;C;C;C;B;B;F;K;C;I;D;L;C;L;L;B;B;C;F;C;B;E;E;J;C;E;H;H;J;B;E;H;I;I;H;L;L;L;L;D;L;B;B;B;E;E;I;K;G;B;L;L;L;L;I;I;I;I;B;C;B;B;D;K;A;E;D;D;E;E;E;E;C;B;B;L;F;G;G;B;B;B;L;L;L;L;E;E;H;C;D;D;D;D;D;D;B;L;L;L;L;L;L;L;L;L;I;D;A;I;I;E;I;B;B;L;L;L;L;A;E;A;I;I;E;G;A;G;B;B;E;K;K;E;I;I;B;L;I;L;L;L;L;I;I;B;I;K;B;H;C;H;B;H;H;G;D;D;D;D;H;L;L;L;E;H;C;B;I;D;B;B;D;A;D;B;A;D;L;L;F;B;I;L;A;B;A;C;A;B;A;C;C;C;F;C;B;G;B;A;G;H;D;H;C;D;B;B;G;B;I;L;E;I;H;H;F;I;I;C;C;F;F;C;A;E;C;C;B;D;D;H;H;B;I;C;L;D;B;B;E;A;A;G;I;L;M;M;G;I;I;I;E;C;C;B;L;L;E;E;L;E;E;E;B;F;G;C;B;G;G;E;F;B;E;L;B;D;H;J;D;L;I;A;A;G;G;C;C;G;G;F;F;G;D;D;D;D;D;E;E;L;E;E;B;I;A;D;L;L;L;E;E;B;L;D;B;B;F;E;E;L;L;G;H;A;A;A;A;A;C;C;C;C;C;B;I;I;I;I;F;H;L;I;A;C;C;I;E;H;E;L;E;E;E;E;I;F;F;F;F;B;I;I;F;C;L;K;E;L;E;E;E;E;H;K;H;B;B;A;G;L;L;I;C;B;G;L;L;E;E;H;L;D;I;K;H;E;E;H;G;A;L;L;E;E;F;I;D;E;E;E;E;E;A;A;C;L;J;G;I;L;L;H;G;I;E;F;C;B;B;E;E;E;E;E;E;E;E;E;J;I;B;B;J;E;B;I;B;B;C;D;A;E;H;H;A;I;L;G;I;E;E;E;E;H;B;B;A;D;D;I;I;H;I;I;B;C;C;B;E;E;E;A;G;G;B;H;H;B;G;E;H;H;E;L;E;C;B;L;L;E;D;L;H;A;A;B;B;E;E;E;B;E;F;B;I;E;E;E;C;C;G;K;C;I;C;D;D;L;A;A;B;B;K;L;E;E;B;A;E;L;E;I;B;C;M;M;L;I;G;B;C;L;L;L;L;L;L;L;L;L;L;D;L;A;D;G;C;A;I;K;K;K;K;C;L;L;L;L;E;L;L;D;B;B;I;I;A;L;H;L;B;D;D;D;B;E;G;G;A;A;A;A;A;A;B;I;I;I;F;G;E;F;G;G;D;E;I;F;E;I;I;L;I;L;L;L;J;D;B;B;B;B;B;B;B;I;J;J;E;B;B;C;I;K;K;K;I;I;I;B;C;B;B;I;F;L;G;G;G;B;L;L;B;L;I;G;D;L;E;L;E;I;E;F;F;B;L;L;L;L;B;C;A;A;C;D;L;L;C;C;E;B;L;L;L;L;B;K;I;L;B;B;B;B;L;L;F;I;D;B;E;E;E;C;C;A;A;I;I;L;L;L;C;L;L;L;H;H;I;A;C;C;C;C;C;C;C;E;A;L;L;L;I;H;A;A;A;A;A;D;F;B;E;E;E;C;B;L;I;F;F;F;F;E;D;D;B;E;E;E;F;B;L;L;L;C;B;F;G;F;I;E;D;B;L;L;L;G;M;B;B;L;E;E;F;F;B;D;L;E;F;F;B;L;L;L;L;E;E;E;E;E;A;B;B;L;F;I;I;I;B;F;F;F;F;D;D;L;E;A;B;I;H;J;L;L;D;E;E;E;E;E;E;F;F;E;E;A;A;A;F;L;F;F;F;F;F;H;F;F;C;L;F;G;E;E;L;L;L;L;L;I;I;I;I;L;I;F;F;F;F;I;K;K;B;K;C;C;I;H;H;B;I;B;B;G;D;D;A;B;B;B;B;B;B;I;L;L;L;L;F;A;G;G;G;B;E;E;A;A;C;I;F;E;K;D;H;A;L;D;B;A;A;I;A;E;I;B;B;C;H;E;B;A;A;M;G;G;G;C;C;C;C;C;G;C;G;B;B;C;G;D;D;A;M;B;B;F;E;G;I;I;I;A;A;I;I;L;E;D;L;L;I;H;H;E;E;I;G;B;B;H;I;L;D;D;D;C;A;I;E;E;J;C;B;E;C;B;B;L;L;K;K;F;A;A;B;D;D;I;I;I;I;B;G;D;D;D;G;I;I;I;I;I;A;G;B;B;B;B;G;C;C;C;C;D;D;D;D;D;D;D;D;D;D;L;L;F;D;D;D;L;L;L;C;D;D;D;D;D;D;D;L;L;L;L;L;L;L;D;A;A;A;A;A;A;G;A;A;C;C;B;B;B;B;B;B;B;B;F;B;G;L;I;I;F;F;L;H;B;E;I;I;B;B;I;A;I;L;L;E;F;C;L;C;C;C;B;K;H;E;G;C;D;F;B;B;B;E;E;A;B;B;E;B;A;L;L;L;L;L;K;B;B;H;H;H;C;B;E;D;L;K;K;D;D;L;L;L;H;E;E;E;E;F;L;L;L;F;B;B;I;B;I;A;A;A;M;G;G;G;B;G;A;I;F;F;A;L;A;A;A;A;A;A;I;I;G;G;B;B;I;I;I;F;C;L;C;B;B;D;B;D;L;B;E;C;F;B;C;H;F;J;G;H;F;G;G;B;B;I;I;I;F;A;A;I;B;B;B;B;F;A;A;A;M;E;E;B;B;D;C;C;C;C;E;E;E;D;L;I;G;J;E;I;H;H;E;E;A;A;A;B;G;F;F;B;C;B;A;A;I;I;G;B;B;I;L;L;E;B;A;E;I;D;L;L;G;D;D;C;B;A;H;I;E;E;F;F;F;H;K;K;G;G;G;C;L;E;L;L;A;L;C;C;D;E;L;L;L;L;E;F;A;A;B;I;C;C;C;C;L;L;L;L;L;L;E;E;H;D;I;F;F;E;E;E;E;E;E;E;E;E;B;B;L;I;D;E;E;I;E;B;B;L;E;E;E;L;G;B;B;D;L;L;L;L;I;I;A;B;C;C;B;B;B;A;B;A;A;I;B;J;L;E;C;B;I;I;I;L;C;B;B;I;I;I;L;J;D;L;L;E;B;B;F;F;F;F;F;D;D;B;H;A;A;F;L;H;H;H;E;B;L;L;H;A;A;I;C;B;B;D;D;K;E;H;J;A;B;L;B;B;I;I;I;C;G;B;B;L;L;L;G;F;B;L;C;B;L;L;L;B;I;F;F;A;I;D;H;H;C;B;B;I;D;E;E;A;I;I;E;L;B;E;E;E;B;B;C;E;E;E;E;B;L;G;A;K;K;K;K;I;E;E;E;B;G;L;H;H;L;L;L;L;L;L;L;H;H;E;A;A;A;E;L;L;E;E;I;L;C;E;E;E;E;J;J;E;F;M;L;L;L;F;F;J;D;F;F;F;K;G;G;H;C;E;A;B;I;I;C;E;E;E;E;B;B;H;I;I;I;L;E;E;C;I;B;B;L;L;L;E;B;A;D;D;I;I;A;K;E;B;F;E;E;I;A;D;C;I;I;C;C;L;E;B;F;G;D;D;B;I;I;I;J;A;D;B;B;B;B;H;B;I;D;B;D;L;I;I;I;I;K;K;K;K;I;H;H;I;F;G;I;B;B;B;B;H;H;H;I;B;E;A;A;C;B;B;E;E;L;I;D;D;D;B;L;E;E;E;E;E;B;E;E;E;E;B;E;D;B;B;B;F;B;M;M;L;F;F;B;E;C;B;D;B;L;C;E;E;E;D;B;I;I;I;I;I;I;H;B;A;C;E;E;L;F;B;L;K;K;B;I;I;I;A;E;E;C;L;B;C;B;B;E;B;L;L;L;H;C;I;B;B;L;B;K;I;E;E;E;E;C;G;L;E;E;D;D;B;B;L;K;K;K;K;K;K;K;K;K;K;C;D;D;D;B;B;L;L;L;L;L;L;L;L;H;B;E;E;E;E;E;E;I;I;J;L;L;H;H;H;H;H;H;G;G;D;L;A;A;B;D;A;B;C;C;C;C;C;C;C;D;L;E;L;F;F;F;F;B;A;E;I;K;K;K;K;E;E;H;H;H;E;E;E;F;C;E;C;C;L;L;L;L;L;H;H;E;E;E;C;I;C;C;L;L;L;L;L;H;H;G;C;A;A;B;L;L;L;H;H;I;B;F;H;B;E;I;A;L;E;D;A;D;B;I;F;F;G;C;C;C;C;D;D;D;D;D;D;D;D;D;D;L;L;F;C;B;I;B;L;A;L;L;E;L;C;G;G;C;C;C;C;C;C;L;H;H;L;D;D;E;I;F;I;E;E;B;B;G;B;B;B;F;F;C;H;F;G;G;C;C;G;G;C;D;M;B;B;L;L;L;A;G;G;D;H;H;L;C;C;C;C;C;G;C;B;K;D;L;F;B;F;I;I;B;F;B;E;E;E;E;I;D;D;C;B;B;L;L;L;L;I;E;E;E;E;E;C;B;I;B;E;C;L;B;B;B;C;B;B;B;D;L;C;C;C;F;B;B;I;C;B;B;I;I;I;F;I;I;I;F;F;B;B;E;E;E;E;E;H;E;A;C;C;D;C;C;C;L;B;A;B;B;B;B;C;C;A;G;I;K;B;E;E;G;G;B;I;A;A;C;C;I;I;C;E;E;E;A;A;G;L;K;C;C;I;H;H;C;E;K;I;F;B;A;D;B;I;A;C;C;L;L;L;L;L;L;L;I;D;D;L;E;G;D;D;A;A;A;E;C;L;L;H;H;H;H;A;C;C;C;G;B;A;A;L;H;H;H;H;E;B;I;I;I;A;A;K;K;I;K;C;D;L;L;G;G;C;D;H;I;D;D;L;I;A;H;H;C;B;B;C;B;L;L;H;K;A;H;A;G;B;E;F;F;F;B;I;I;E;E;I;E;L;A;C;B;B;I;I;I;C;E;E;E;F;F;L;L;I;B;G;B;I;E;E;L;E;H;H;H;H;B;D;D;B;B;B;E;E;E;L;F;F;F;L;L;L;L;L;L;B;L;L;B;K;K;K;K;H;L;L;B;B;B;L;L;H;H;I;I;I;I;J;J;J;G;D;D;F;I;L;L;B;B;L;L;L;L;C;C;B;D;B;A;G;I;I;I;L;L;H;I;L;L;L;A;A;G;F;C;B;G;B;A;F;A;A;F;H;H;H;H;H;B;I;I;G;D;D;A;B;B;B;I;I;I;F;F;C;C;C;H;H;H;H;H;H;H;H;H;A;G;G;G;C;C;B;E;E;L;L;E;E;E;E;E;C;G;I;E;D;D;C;C;B;B;C;C;C;C;G;D;D;D;D;D;D;D;D;D;D;D;I;I;I;I;A;A;I;I;K;K;I;I;M;B;C;I;E;J;K;K;L;L;L;E;E;I;A;B;I;A;A;L;L;I;I;H;H;G;G;G;B;B;B;B;B;B;B;B;B;B;B;B;G;J;C;E;G;G;B;D;D;D;D;B;I;B;E;G;D;A;A;I;I;B;B;B;G;I;I;C;C;E;B;B;B;I;C;I;A;F;C;D;D;D;L;L;L;L;L;E;E;E;B;D;D;E;E;E;E;L;A;A;A;A;C;L;B;B;B;B;C;C;C;B;B;B;K;L;L;L;L;F;H;H;K;C;D;D;D;D;M;L;L;L;L;L;L;L;B;B;B;A;K;B;C;B;L;E;E;E;G;E;E;E;E;A;L;H;H;H;H;G;G;C;D;D;D;D;L;L;K;E;H;J;B;C;I;H;A;L;I;E;A;B;B;I;B;B;B;G;B;E;G;F;F;F;F;E;F;F;E;E;E;E;E;E;E;E;E;K;A;A;A;A;A;A;H;H;A;I;I;I;H;D;B;D;E;E;C;C;L;A;B;E;E;E;E;H;H;E;E;E;E;E;E;F;C;B;B;C;L;L;I;C;C;C;H;C;L;L;L;L;I;I;B;I;G;D;D;L;L;L;E;E;I;I;F;F;F;I;G;B;L;C;H;H;F;F;B;K;K;K;K;K;K;C;E;B;L;L;L;L;L;L;A;E;H;G;B;I;I;I;L;F;F;B;L;A;A;I;C;C;C;C;C;C;C;G;C;A;A;B;E;G;I;A;F;B;L;H;H;B;B;B;E;E;A;D;D;D;E;E;E;B;G;F;F;B;A;A;E;E;B;B;E;F;I;G;B;F;F;H;H;I;J;I;B;A;A;M;G;G;G;C;C;C;C;C;G;C;G;L;E;B;B;L;L;L;E;E;E;E;C;C;L;L;I;E;L;E;H;A;D;E;E;E;C;A;A;A;H;H;B;B;B;E;L;L;I;E;E;E;L;L;E;E;L;A;A;L;L;L;H;B;E;E;L;L;L;B;C;L;E;D;L;A;C;B;E;L;H;D;B;E;I;E;C;E;E;E;E;B;B;L;L;L;E;F;L;B;A;A;M;G;G;G;C;C;C;C;C;G;C;G;C;L;E;D;D;D;D;H;H;C;C;B;B;B;B;B;B;L;L;L;L;L;L;I;I;E;C;C;I;J;I;C;B;B;E;L;B;C;B;B;E;F;G;G;C;C;C;C;C;C;F;A;A;F;I;E;E;E;I;I;I;E;I;M;B;E;I;I;F;B;B;D;H;L;L;H;E;A;A;A;C;C;C;D;D;D;D;D;D;D;B;L;L;E;E;F;H;B;F;F;F;C;C;C;B;C;L;L;I;I;I;G;I;L;B;B;B;B;B;L;L;A;B;L;K;K;K;K;K;K;C;E;B;L;L;L;L;L;L;A;E;H;B;L;A;A;E;H;F;A;B;I;I;I;D;L;L;C;C;L;L;E;E;E;E;E;E;E;H;H;H;D;H;L;L;E;C;I;G;I;F;C;G;C;B;E;G;A;A;F;A;A;A;D;D;I;I;E;L;L;I;G;D;K;L;B;B;E;D;B;L;L;B;H;C;G;D;D;L;L;L;B;C;C;E;I;I;I;I;I;E;E;G;B;I;H;B;L;B;G;G;L;F;I;A;A;I;I;I;I;I;I;H;L;B;G;I;B;B;L;L;L;L;F;F;F;B;I;K;C;L;D;C;D;D;D;B;C;F;F;A;B;B;B;B;B;B;E;I;I;A;A;C;C;B;B;B;B;B;B;B;B;F;I;L;L;G;D;B;I;I;L;I;I;B;B;A;C;B;E;B;B;B;B;B;B;L;B;I;I;I;F;I;J;B;E;L;E;H;H;H;H;H;H;H;H;H;H;H;H;H;H;K;K;E;J;B;B;F;I;B;H;J;A;D;D;I;I;C;B;B;B;I;I;I;I;F;H;L;L;L;C;E;D;B;B;G;G;E;D;B;L;B;B;A;A;A;A;D;H;H;H;G;K;K;B;B;B;G;B;I;C;C;B;B;L;C;B;L;L;G;B;B;F;J;D;D;D;D;B;B;B;L;L;L;H;L;I;F;I;G;L;L;L;L;L;H;H;H;A;C;G;C;C;C;C;F;I;C;A;C;K;C;E;L;L;B;H;A;A;A;F;B;B;B;D;B;D;L;D;E;B;F;F;J;G;I;I;I;F;A;G;C;E;E;K;L;I;I;D;D;L;L;L;B;E;E;E;E;B;B;B;E;D;H;L;L;F;A;I;J;L;L;H;B;L;H;G;L;I;J;D;B;I;I;L;L;L;L;C;L;L;I;I;C;L;L;B;L;L;L;L;L;I;I;I;G;I;A;I;G;D;D;A;A;A;I;I;B;D;I;I;G;E;D;B;I;I;I;F;B;D;F;F;D;D;B;B;B;E;E;E;E;B;B;A;A;A;D;L;A;C;C;F;C;E;E;G;G;C;D;D;D;D;L;L;C;C;C;H;I;I;J;B;B;A;H;L;A;A;A;A;F;L;I;I;L;G;D;E;L;L;F;B;D;B;B;I;I;C;D;M;B;B;E;H;G;J;B;B;I;A;I;G;F;F;B;G;L;B;D;B;B;B;B;B;K;I;F;F;L;L;L;D;E;L;K;C;C;D;D;L;E;G;G;G;C;B;D;E;E;E;B;E;L;A;A;A;C;E;E;E;H;I;E;E;L;L;I;F;I;F;I;A;A;A;I;D;B;B;I;B;B;I;I;E;E;L;E;E;C;E;C;B;L;L;L;G;G;L;E;L;L;B;B;F;L;I;E;B;A;A;A;A;A;I;I;I;I;I;J;G;E;G;C;H;A;E;A;B;B;I;I;B;D;D;L;I;L;B;I;A;L;L;L;L;F;B;H;I;B;K;B;I;H;B;B;A;A;J;B;B;A;A;A;I;L;H;H;H;H;B;C;B;A;A;A;A;A;D;L;G;B;F;B;B;B;H;E;C;C;C;C;B;H;B;L;L;A;I;I;I;A;B;B;B;E;E;J;B;B;B;E;E;E;E;L;E;E;A;A;A;K;K;K;A;A;L;L;E;E;F;F;F;F;D;D;D;H;E;L;L;E;E;G;G;H;I;I;I;I;F;I;I;I;B;B;B;B;L;F;A;A;E;G;C;M;E;C;C;B;B;E;G;H;G;L;L;L;L;L;L;L;F;F;C;B;L;I;L;L;I;A;G;C;E;E;E;C;A;A;A;B;B;L;E;C;I;L;I;C;F;G;L;A;B;G;H;L;B;E;E;L;L;L;L;L;B;B;B;B;I;C;D;L;E;C;C;B;E;E;C;L;D;D;A;E;C;B;B;B;B;A;F;F;F;A;A;A;H;H;H;B;C;C;B;D;C;F;F;A;E;E;E;F;F;L;E;I;C;B;I;I;L;L;F;B;A;B;I;D;B;B;B;A;B;B;B;L;L;A;A;E;C;C;L;L;L;L;L;L;K;A;I;C;C;B;G;B;A;A;G;G;G;L;L;B;B;A;A;A;A;A;I;I;I;I;I;B;B;D;L;L;H;C;B;B;A;D;E;B;I;G;F;F;F;F;B;D;L;B;A;I;I;I;I;I;E;E;C;C;C;C;B;H;H;E;L;E;J;E;F;D;D;D;D;I;C;C;H;B;B;B;G;B;C;C;A;L;A;C;D;B;B;B;B;B;B;I;I;A;L;I;H;I;C;E;E;E;L;B;E;C;B;B;B;F;B;B;B;A;A;A;A;I;I;G;B;B;C;H;B;B;G;A;G;I;D;D;D;E;G;C;C;C;C;C;B;B;B;B;E;E;E;E;E;I;K;K;D;G;E;E;E;I;E;E;D;I;I;I;I;I;E;E;A;C;C;C;F;A;C;B;E;J;I;L;F;F;I;I;B;B;G;G;K;K;K;K;B;B;B;B;E;E;E;E;L;L;L;L;L;L;L;L;L;L;D;D;G;G;B;B;L;L;I;I;C;C;C;C;L;L;C;C;A;A;I;I;I;I;H;E;C;C;B;L;F;F;F;F;F;F;F;G;B;B;B;B;B;B;I;G;C;B;B;B;B;K;K;L;L;L;L;I;I;I;C;I;B;B;B;B;I;I;G;F;F;F;F;F;F;F;F;L;I;I;L;G;L;L;H;H;B;B;L;E;D;D;B;D;D;D;A;F;G;B;B;B;B;J;B;G;C;C;C;C;D;D;D;D;D;D;D;D;D;D;D;D;D;D;D;D;D;D;D;D;D;D;L;L;F;E;F;E;B;B;B;A;B;B;B;B;A;A;A;A;A;A;A;I;I;I;I;I;G;I;I;B;L;I;E;E;H;H;B;B;B;A;A;A;B;B;B;B;B;B;E;E;E;E;E;E;E;E;E;J;G;I;I;I;I;I;E;E;E;E;L;L;D;D;F;F;H;H;D;G;E;B;A;G;C;A;B;B;B;B;L;I;E;E;E;E;H;H;H;H;H;H;H;F;B;B;D;D;D;D;B;B;B;B;B;B;E;E;E;E;E;E;G;G;C;C;G;G;L;L;B;B;B;B;E;G;G;J;J;B;B;A;A;B;B;E;E;I;I;B;B;D;D;C;C;C;C;G;G;C;C;C;D;D;D;D;E;E;E;E;I;I;G;G;C;C;D;D;D;D;I;B;B;G;B;B;D;D;D;D;D;D;C;I;I;I;I;F;F;B;B;E;B;B;B;B;B;B;B;B;C;C;C;I;F;F;C;E;E;F;B;B;B;B;B;B;I;E;E;B;B;A;A;E;A;A;I;D;D;D;D;D;C;F;L;L;L;L;L;L;L;L;C;L;I;B;B;B;E;A;A;A;A;L;A;A;A;A;A;A;A;A;A;D;D;D;D;D;D;E;B;B;B;B;B;B;I;C;C;C;C;L;J;B;B;B;D;D;D;E;E;A;A;A;A;L;L;L;L;I;C;D;D;D;L;L;B;B;B;B;B;B;B;B;B;A;A;A;H;H;A;B;B;B;B;B;B;B;B;B;B;D;D;D;D;H;H;A;A;L;L;H;B;B;E;E;D;D;D;D;D;D;L;L;L;L;L;L;E;E;A;A;E;E;E;E;E;E;C;C;C;C;B;B;F;F;F;F;E;E;E;E;B;B;B;B;E;E;E;E;J;J;D;D;B;B;I;I;I;I;I;F;F;K;K;B;B;L;L;D;D;D;D;B;B;C;C;C;C;B;B;G;G;B;B;F;F;B;B;C;C;C;C;C;C;C;C;C;C;C;C;C;C;B;B;B;B;B;B;H;H;H;H;H;H;F;H;B;B;D;D;D;D;B;B;B;B;B;B;E;E;E;E;E;E;F;F;I;I;B;B;E;E;E;E;G;G;B;B;E;E;E;E;H;H;H;H;F;F;B;B;D;D;A;A;B;B;B;B;L;L;I;I;E;E;E;E;E;E;E;E;E;E;E;E;C;C;L;L;L;L;C;C;E;E;E;E;E;E;I;I;I;I;I;I;B;B;L;L;L;L;L;L;L;E;E;E;E;H;H;I;I;B;B;D;D;D;D;D;D;E;E;E;H;B;J;B;B;B;B;E;B;L;C;B;B;L;L;L;L;H;H;E;B;B;B;L;L;D;D;L;L;L;E;D;L;B;L;L;L;C;C;B;B;B;B;F;B;B;G;F;F;F;B;B;E;E;L;L;H;B;B;B;B;F;F;F;F;B;B;L;L;A;A;F;F;D;D;D;D;E;E;E;E;B;B;E;E;A;A;K;K;K;K;C;C;E;E;C;C;C;C;C;C;C;C;C;C;D;D;D;D;D;D;D;D;D;D;M;M;D;D;D;D;A;A;C;C;B;B;L;L;L;L;L;L;L;L;L;L;L;L;E;E;E;E;B;B;E;E;I;I;A;A;A;A;H;H;I;I;I;I;E;E;B;C;C;C;C;C;C;L;L;L;L;L;L;L;L;L;L;L;L;F;F;E;E;E;E;A;A;I;I;I;I;L;L;D;D;E;E;E;E;E;E;E;E;E;E;F;F;G;G;F;F;G;G;G;G;I;I;A;A;J;J;G;G;D;D;L;L;A;A;C;C;B;B;E;E;E;E;E;E;A;A;C;C;C;C;C;C;F;F;F;F;F;F;F;F;F;F;F;F;H;H;H;H;H;H;H;H;B;B;D;D;D;B;B;B;E;E;E;B;B;B;J;I;L;E;G;G;F;F;F;A;A;A;A;A;B;I;I;B;I;D;B;F;L;L;L;L;L;C;B;B;L;I;I;F;C;A;E;A;A;D;F;H;L;I;J;B;I;C;E;E;E;E;B;B;F;E;C;C;C;H;L;L;C;C;C;C;C;C;C;C;D;I;H;H;C;C;F;K;A;A;A;C;B;B;E;E;E;C;C;C;A;J;B;F;G;I;C;B;A;I;L;D;B;B;C;L;I;I;H;B;L;L;A;I;B;E;E;I;I;C;C;C;C;B;C;L;L;A;A;L;B;B;B;B;A;I;I;I;I;I;I;I;G;B;B;I;F;B;A;C;C;B;F;B;H;L;I;B;L;L;H;B;I;I;L;L;L;E;G;B;L;E;C;B;B;G;B;L;H;L;L;L;L;K;A;G;D;D;A;A;A;B;I;I;I;L;J;L;L;L;L;L;L;E;E;L;C;B;L;I;D;D;D;D;E;E;E;I;C;G;C;A;F;F;L;D;K;K;K;D;D;D;D;D;E;E;K;K;B;I;F;L;L;I;I;A;L;L;H;C;C;C;A;A;B;B;H;D;I;G;A;G;E;L;I;I;I;I;H;B;L;L;K;B;D;B;B;B;E;B;B;E;D;C;K;B;H;G;B;E;I;J;D;B;I;H;I;I;B;B;I;I;E;E;E;E;E;H;D;B;B;H;H;E;H;B;G;G;C;C;I;I;H;H;H;C;C;C;C;B;B;F;B;B;D;F;G;C;F;D;D;H;E;F;E;B;G;F;D;B;B;C;L;B;D;L;H;L;A;A;G;G;C;C;G;G;F;F;B;G;I;I;B;B;A;A;A;A;A;B;K;K;B;L;I;I;I;I;A;B;L;I;B;K;B;A;A;M;G;G;G;C;C;C;C;C;G;C;G;G;E;H;K;B;L;L;H;L;L;B;I;B;B;L;L;H;H;C;B;E;B;B;B;G;C;G;G;L;L;L;L;L;L;L;L;F;D;D;D;D;D;E;E;L;E;E;G;D;B;B;B;B;L;E;F;B;L;L;L;L;L;I;I;I;C;B;B;L;B;I;I;J;B;C;C;E;C;L;H;G;G;B;D;B;L;E;I;B;B;C;D;B;A;L;L;C;G;B;C;B;B;H;D;D;D;I;I;L;L;L;L;L;L;E;E;C;F;A;A;A;C;G;D;H;L;F;B;C;B;C;C;L;I;I;I;I;D;L;L;E;F;D;F;C;C;C;G;C;C;C;G;D;D;D;D;D;D;D;D;D;B;B;B;B;B;B;F;K;K;K;K;K;K;K;K;K;C;B;D;D;D;D;B;D;D;D;D;H;L;L;L;E;E;E;H;H;B;D;D;E;F;A;D;D;D;D;L;H;F;H;F;L;L;L;L;L;I;I;I;I;G;F;G;L;A;E;E;B;A;B;E;L;L;F;F;F;F;I;B;B;B;E;E;E;H;A;H;I;I;B;B;G;D;C;F;B;G;B;L;B;L;D;A;B;D;E;G;C;B;A;L;I;I;D;I;D;B;I;F;I;I;C;H;H;C;B;I;B;C;C;B;B;M;H;H;H;H;G;F;F;D;E;E;B;B;B;B;E;C;L;H;F;A;A;B;G;F;C;F;E;A;B;B;B;B;B;C;E;C;E;I;I;I;G;C;B;B;E;G;K;B;J;L;L;I;I;H;I;E;B;B;K;B;B;B;B;B;I;D;L;B;H;C;F;L;L;L;L;H;L;C;C;C;C;B;F;G;F;B;G;G;A;A;B;B;B;F;I;C;C;C;C;D;A;I;I;G;B;B;B;B;H;H;H;F;H;H;H;H;H;G;I;I;A;D;A;A;A;C;C;C;C;C;B;I;I;I;I;F;H;B;D;D;D;D;B;I;C;B;J;H;I;E;E;E;B;D;D;D;D;D;D;H;H;H;H;H;H;H;C;L;B;B;B;F;B;E;E;I;B;B;L;G;B;L;B;K;L;L;F;F;I;I;I;F;G;B;B;B;E;G;L;B;L;G;K;K;K;I;I;I;H;G;G;D;I;G;A;C;C;D;E;C;H;H;H;A;A;A;A;L;A;L;F;F;F;F;G;B;B;B;B;B;H;H;H;H;H;C;A;A;L;L;L;C;C;C;D;I;I;E;L;E;E;E;E;E;E;E;I;L;B;L;C;B;I;C;B;B;D;E;A;I;L;I;F;F;B;E;E;A;A;A;A;M;L;L;L;L;L;L;A;H;H;H;H;H;G;F;F;F;F;F;A;A;F;B;F;J;A;A;B;L;I;I;I;H;A;B;K;B;H;F;G;B;I;I;G;H;H;L;I;I;I;I;B;B;B;B;I;I;B;E;A;A;A;A;L;I;B;E;J;F;F;F;I;I;L;A;B;L;L;L;G;D;A;L;L;F;B;B;B;I;G;E;G;B;L;L;H;C;I;I;I;G;F;F;C;A;A;A;H;H;H;E;J;G;C;L;I;G;B;L;L;L;H;I;I;I;I;I;C;C;F;C;D;L;L;L;E;F;F;L;B;C;C;F;G;L;L;A;G;G;I;H;G;I;H;F;B;B;B;B;B;B;L;L;L;I;L;B;B;A;G;A;A;A;A;L;A;C;E;E;B;B;I;L;L;L;L;L;L;I;E;G;C;E;G;B;D;C;L;A;H;A;A;A;B;H;C;D;M;B;B;F;F;C;C;L;L;E;E;E;E;E;E;E;H;H;H;B;B;L;K;E;L;G;A;A;E;E;I;H;L;L;L;A;H;B;B;D;A;B;I;I;I;H;B;G;C;D;L;F;E;B;G;F;B;H;L;L;G;B;F;C;B;D;I;I;I;H;I;J;L;H;C;C;B;B;B;B;B;B;L;L;L;L;L;L;I;I;G;E;E;I;I;I;F;L;G;A;A;L;E;B;L;A;A;A;B;E;I;I;H;H;B;I;C;C;H;H;D;D;B;B;L;F;L;A;E;E;F;E;L;D;B;C;C;L;L;I;L;L;K;I;B;C;L;L;E;A;H;B;B;B;B;A;D;D;I;I;A;A;D;I;F;C;C;B;B;B;B;B;F;F;L;L;F;F;H;C;I;I;D;B;B;K;K;K;K;K;B;F;I;I;I;L;K;K;E;E;E;H;H;G;B;B;D;H;D;L;L;B;B;L;I;B;I;I;G;B;L;L;L;I;B;C;C;D;D;H;H;I;E;I;J;B;B;E;B;E;F;B;I;I;L;J;L;K;K;F;I;H;H;D;D;D;D;D;D;D;D;D;E;E;E;E;E;E;E;H;B;A;C;C;I;G;I;D;D;L;E;I;G;D;L;L;B;B;B;G;B;K;A;B;E;I;I;I;E;L;L;L;C;C;C;C;C;G;G;C;B;D;B;B;C;B;B;C;F;F;F;F;L;I;B;F;L;L;L;B;F;F;I;I;A;A;G;G;C;C;G;G;F;F;E;G;B;B;I;I;F;B;C;B;B;L;L;L;L;I;A;C;E;G;B;B;B;B;A;I;L;B;L;B;B;C;G;B;L;H;C;L;E;E;E;E;E;I;I;I;L;H;D;D;D;D;D;E;E;E;E;E;K;I;H;D;B;B;E;B;E;I;I;C;D;D;L;G;I;L;A;I;D;C;F;L;I;I;I;I;L;E;L;E;H;G;C;C;C;I;E;L;L;L;L;H;G;B;E;E;E;B;E;E;E;F;F;G;C;C;E;B;F;F;J;L;A;D;C;C;D;B;L;G;L;L;L;E;L;L;B;B;B;L;I;G;E;F;G;G;C;C;C;B;E;C;C;G;G;I;I;E;A;C;C;I;I;I;L;A;A;H;A;A;L;H;K;K;K;K;A;G;B;L;L;L;L;L;L;L;D;A;C;B;B;E;E;A;B;I;I;L;A;C;C;I;J;L;J;E;H;G;F;L;H;H;L;L;L;L;L;L;L;H;H;H;L;B;L;L;A;F;B;I;I;F;B;F;H;A;D;L;L;I;B;F;B;B;I;B;B;I;A;B;J;F;F;B;I;F;F;I;A;A;A;B;F;E;B;I;B;A;C;C;C;D;D;D;D;D;D;D;B;L;L;E;E;F;H;I;B;I;H;D;D;D;E;D;D;G;I;C;H;A;E;E;C;B;D;B;B;I;I;I;I;A;B;A;D;H;F;E;D;D;D;D;B;I;I;I;I;G;D;D;E;B;F;I;B;B;C;B;I;D;L;L;E;E;F;F;A;A;C;E;E;E;E;E;F;J;A;E;D;I;A;B;C;C;L;I;I;I;C;D;C;C;C;B;B;L;L;E;I;C;B;L;L;L;B;L;L;L;I;E;E;E;E;E;A;G;E;E;E;C;C;L;L;L;D;F;F;G;D;B;B;B;B;M;A;A;A;G;L;F;C;B;L;E;B;C;G;B;L;L;E;F;I;I;E;G;F;B;B;I;J;B;C;C;B;D;D;E;A;A;B;A;G;B;B;B;B;E;G;L;L;L;L;L;L;L;E;F;F;F;A;C;C;F;L;H;K;K;K;K;K;K;K;K;K;K;C;D;D;D;B;B;L;L;L;L;L;L;L;L;H;B;I;I;G;I;F;B;B;G;K;K;B;H;H;E;B;I;A;B;E;G;B;B;K;K;L;L;L;E;E;A;L;C;C;C;C;C;B;I;H;H;B;L;L;L;H;B;C;K;C;C;I;H;H;D;D;B;B;B;L;E;E;F;A;B;D;I;F;C;D;D;D;D;B;B;E;C;B;E;C;C;C;D;D;D;D;D;M;D;D;D;D;F;F;H;C;I;I;I;B;B;B;H;D;D;L;C;B;B;L;L;L;L;B;B;B;F;D;B;C;L;L;L;I;G;A;B;L;A;E;G;H;E;B;E;E;E;I;E;D;B;L;B;B;E;E;E;E;K;H;B;D;A;A;I;E;I;I;I;F;H;C;G;M;K;H;G;L;I;K;A;B;C;C;D;L;H;F;I;I;I;I;B;F;F;D;H;L;D;B;A;B;B;H;K;B;I;L;E;E;L;I;L;E;E;I;B;B;L;D;L;A;B;A;A;A;A;G;B;B;B;D;D;C;B;A;I;D;D;D;E;E;H;A;A;A;A;A;I;C;B;B;B;A;A;H;H;F;F;F;L;A;B;K;F;F;D;B;I;B;I;B;C;B;L;L;H;B;I;I;A;F;G;B;B;B;B;L;L;L;H;H;B;B;F;E;B;C;C;H;C;D;B;L;L;L;L;L;E;E;E;E;A;B;B;C;B;A;B;B;A;A;A;A;B;G;G;F;F;F;E;E;E;E;L;A;J;F;B;B;F;J;F;E;F;B;A;B;L;B;B;G;F;B;B;G;G;G;C;B;D;E;E;E;D;D;D;D;D;K;I;C;L;L;E;E;E;A;I;B;J;E;H;F;I;I;A;L;L;H;E;C;C;B;B;L;L;H;I;I;B;I;C;C;C;D;D;B;B;B;B;B;B;E;E;C;B;L;I;E;B;A;A;D;E;E;E;E;E;F;F;F;F;E;G;G;A;C;I;F;I;K;C;H;G;G;B;D;B;B;L;E;J;E;E;F;C;D;F;F;F;H;L;H;D;D;B;B;A;E;I;B;G;E;H;L;I;L;I;I;I;B;D;L;H;G;G;E;F;F;E;C;I;I;I;I;C;I;E;D;C;C;B;L;L;L;L;F;L;L;G;I;F;M;M;L;I;E;E;C;D;A;B;I;B;B;B;B;A;A;B;I;I;I;B;B;B;I;I;J;I;E;E;L;B;A;B;B;L;E;D;I;G;F;A;A;F;C;B;E;E;G;B;E;G;C;C;C;D;E;L;L;L;L;E;H;E;E;A;I;I;L;I;L;L;I;B;E;E;B;E;E;A;A;B;A;A;A;I;A;E;E;E;H;B;L;L;E;I;I;I;L;E;E;F;F;F;I;J;J;B;B;L;B;E;B;F;C;E;B;E;E;A;A;B;B;I;F;F;D;F;F;F;B;B;B;B;B;G;I;F;C;E;H;B;B;D;D;G;B;L;L;L;H;H;C;J;B;H;I;C;E;E;F;F;A;C;F;F;H;B;H;C;D;B;B;L;L;L;L;E;A;I;J;L;E;C;G;A;I;I;I;E;G;E;L;E;J;D;D;L;L;B;L;A;E;E;B;A;A;B;B;B;B;B;H;I;F;I;I;L;I;A;A;A;A;D;H;H;H;G;I;B;L;E;C;B;G;D;D;L;C;F;F;G;C;C;G;M;M;F;B;C;L;L;L;L;L;L;E;E;B;D;L;D;J;E;F;F;F;L;L;H;L;L;A;A;I;I;E;A;B;H;H;H;F;B;D;D;B;B;B;E;E;E;K;F;H;C;C;B;B;B;B;B;B;L;L;L;L;L;L;I;I;F;F;B;G;B;D;D;I;A;L;I;K;K;L;L;L;E;E;C;B;A;D;H;G;E;E;E;E;E;E;E;B;B;B;B;B;I;J;B;L;E;E;I;B;B;B;L;F;B;H;E;E;B;B;B;B;K;K;K;K;K;K;D;B;I;H;I;G;I;I;C;L;L;L;B;L;L;G;F;F;L;B;B;E;G;L;G;C;C;C;F;L;A;B;B;B;E;E;E;L;L;L;E;I;E;D;H;I;A;I;B;L;L;I;I;L;L;L;H;B;B;I;F;C;C;C;G;C;C;C;G;D;D;D;D;D;D;D;D;D;B;B;B;B;B;B;F;D;E;F;A;F;B;B;B;G;B;L;A;B;I;I;B;E;B;A;B;B;L;L;I;I;I;D;L;B;A;H;E;B;B;F;L;B;E;E;E;H;H;B;I;I;B;B;B;B;B;F;L;L;L;E;E;L;L;E;H;B;L;L;I;B;L;I;L;H;B;B;B;L;L;E;E;D;C;F;H;L;L;L;L;A;E;B;A;A;C;E;F;I;F;C;H;G;G;C;I;A;G;B;B;B;E;G;D;D;A;A;A;B;E;E;E;A;I;I;I;D;F;C;C;C;C;C;C;D;D;D;D;M;M;D;D;I;A;B;F;G;B;G;C;I;B;B;B;B;D;E;I;G;L;L;L;D;D;D;D;A;B;D;I;J;B;B;B;K;C;L;L;L;L;I;H;H;H;H;E;I;B;B;K;A;L;L;B;B;A;I;B;L;J;M;M;H;B;C;H;H;B;B;E;E;B;B;B;A;F;F;B;E;D;H;I;H;H;M;M;B;D;D;E;B;A;B;B;I;I;I;I;F;E;A;B;B;A;A;A;A;A;B;L;L;H;D;A;C;C;F;I;I;A;A;G;C;A;I;H;H;H;H;H;H;E;H;C;F;F;I;I;F;L;H;G;C;C;C;C;E;I;L;H;L;I;B;F;B;B;B;B;I;E;F;B;G;D;D;D;F;B;B;J;K;K;A;A;A;E;E;E;L;B;B;L;I;F;C;L;E;I;A;E;B;D;L;H;L;E;L;L;L;L;E;L;I;C;B;A;B;L;B;B;B;B;B;L;L;F;I;L;C;C;A;D;L;I;B;I;E;E;L;B;C;L;I;I;E;A;A;A;A;A;G;B;B;B;E;F;I;E;C;H;G;A;B;H;C;C;B;B;B;B;B;B;L;L;L;L;L;L;I;I;E;E;B;C;C;B;L;I;I;E;B;C;C;H;H;E;C;B;B;C;B;I;I;G;L;L;A;F;F;F;F;F;C;D;D;L;B;B;B;I;G;B;I;I;B;H;H;H;H;D;H;L;A;G;I;G;B;B;J;I;E;L;E;L;L;I;H;L;G;B;E;H;H;L;E;E;E;E;D;I;I;F;L;H;H;I;H;E;E;E;G;H;K;I;C;C;C;C;C;C;C;C;B;B;F;F;A;A;I;D;L;L;E;E;E;E;B;B;C;H;F;F;E;C;K;B;B;L;E;B;L;D;L;E;E;B;A;I;I;I;L;E;I;D;A;A;I;I;A;L;C;I;I;K;C;D;D;D;D;M;L;L;L;L;L;L;L;E;D;B;B;E;E;E;B;B;D;C;G;M;L;I;I;G;E;E;E;B;G;L;L;L;L;L;H;H;H;B;B;L;E;A;B;G;B;B;I;I;B;J;L;B;E;E;I;H;A;A;C;E;I;I;C;H;I;H;I;I;I;I;I;B;C;A;L;L;B;I;C;C;F;F;A;C;B;G;B;L;E;H;I;H;H;H;H;H;H;H;H;H;H;H;H;H;H;E;G;B;K;E;H;I;D;D;D;A;A;A;A;B;L;L;H;J;E;D;D;M;B;I;D;L;L;I;F;D;G;G;L;D;L;L;C;C;C;C;E;E;E;L;L;D;D;E;E;B;E;D;D;F;F;C;D;L;L;L;L;L;L;B;G;C;A;A;A;I;C;B;E;A;I;I;I;D;L;L;A;A;L;L;L;L;L;L;E;E;I;G;H;H;F;F;L;I;B;I;L;I;L;K;K;L;L;E;L;E;E;A;A;A;A;B;B;B;B;B;L;I;I;I;F;C;E;H;B;I;I;I;I;D;L;L;L;L;D;I;E;C;I;L;L;F;A;B;I;C;F;F;F;F;L;I;D;I;I;I;I;I;B;A;G;G;B;B;F;A;A;A;D;D;D;L;B;I;J;L;H;E;G;C;B;B;I;A;L;L;L;B;F;F;F;H;I;B;F;F;G;I;I;B;H;H;H;F;B;D;D;B;B;B;E;E;E;I;D;D;H;B;I;A;B;B;E;E;C;C;C;C;B;B;B;B;B;B;L;L;C;B;L;I;E;E;E;E;K;K;A;A;A;E;E;E;C;D;L;F;B;D;L;E;E;L;L;I;G;B;B;B;B;L;L;L;H;H;J;B;C;D;L;L;L;K;E;E;E;E;D;C;B;L;L;H;E;E;F;L;L;E;B;E;E;E;E;F;B;G;B;L;L;I;E;E;K;G;G;H;I;B;E;E;E;E;E;F;E;B;B;B;B;B;I;G;A;B;C;C;B;B;B;C;A;I;F;E;E;I;G;B;B;A;A;L;E;K;I;B;B;G;G;I;A;E;E;E;E;E;E;E;E;E;E;E;E;E;L;I;H;H;C;M;H;B;B;J;L;A;A;A;I;B;B;B;B;B;B;B;G;C;E;D;D;D;G;B;A;A;A;E;E;E;E;E;F;F;F;F;E;A;L;L;E;E;B;K;I;E;E;E;E;B;E;E;C;F;B;B;B;B;F;F;F;I;L;L;L;B;B;G;B;I;L;L;L;L;I;I;I;F;I;E;E;E;C;F;L;L;E;E;E;L;I;I;L;L;F;C;H;B;B;L;E;K;K;E;E;C;C;D;D;D;D;D;M;D;D;D;D;F;F;H;C;C;F;B;B;L;I;F;F;B;B;B;I;H;L;B;F;F;H;B;C;F;F;F;B;B;B;B;L;L;L;L;L;L;F;C;E;E;E;E;J;J;J;C;H;F;F;I;D;L;A;A;A;I;I;E;L;L;L;C;K;K;K;I;I;I;H;L;F;F;F;F;F;B;H;F;L;L;L;A;A;H;A;B;B;I;A;B;I;C;A;B;B;L;L;L;L;E;E;H;I;E;E;I;B;C;C;L;L;L;L;L;L;L;L;L;L;I;H;D;B;B;H;H;F;I;G;A;L;L;D;E;B;H;A;A;C;H;L;L;L;B;I;F;F;H;H;G;L;L;H;F;F;I;C;C;D;D;D;D;D;M;D;D;D;D;F;F;H;E;C;C;C;B;F;F;F;F;E;F;F;K;B;G;G;B;B;I;I;I;F;C;C;C;C;B;H;C;L;B;B;B;B;I;L;L;E;D;D;L;E;C;K;K;K;F;I;L;L;L;D;G;H;D;L;L;A;H;A;A;L;L;E;B;I;I;L;F;H;G;A;A;G;B;I;I;L;I;I;F;F;D;A;A;C;C;C;B;D;D;L;L;E;E;L;H;H;H;L;L;L;L;L;L;F;F;H;E;J;I;I;B;B;I;L;C;E;E;E;H;B;B;L;L;F;E;E;B;E;F;F;K;H;F;H;I;H;J;I;J;I;C;G;H;H;A;B;C;B;B;L;A;E;D;D;I;L;B;G;D;B;E;E;C;F;F;H;H;E;E;L;B;I;E;C;L;L;L;H;H;C;I;B;B;H;I;F;A;A;A;I;H;C;C;C;A;E;G;I;B;D;D;D;D;I;I;J;L;E;B;B;F;C;L;H;B;B;L;L;H;L;A;E;I;F;F;I;B;E;B;I;H;H;C;C;B;B;B;B;B;B;L;L;L;L;L;L;I;I;C;D;D;D;D;I;B;B;G;G;G;G;B;C;L;H;A;E;G;F;B;E;L;I;F;L;A;J;H;F;I;I;I;K;A;F;F;H;H;F;F;F;I;I;F;I;L;I;E;G;A;B;B;D;C;C;L;L;C;A;I;C;F;H;H;F;F;I;B;B;B;B;A;A;I;D;B;D;D;D;D;D;B;L;C;L;C;C;I;A;B;B;I;I;F;I;I;D;J;J;F;F;B;L;L;L;L;C;H;H;H;C;C;D;D;L;L;L;C;D;D;D;D;D;D;D;L;L;L;L;L;L;L;A;A;E;C;C;D;D;D;D;D;M;D;D;D;D;F;F;H;C;C;I;B;F;H;H;H;H;H;H;H;H;H;H;H;H;H;H;H;B;H;L;E;C;F;F;C;L;L;L;H;G;G;B;I;B;G;I;F;G;F;G;B;L;E;L;F;F;B;L;L;L;L;H;E;E;E;E;F;I;D;E;E;A;E;E;F;F;B;L;B;G;C;B;B;B;I;C;A;B;B;I;B;B;I;I;F;F;F;H;H;L;L;F;G;F;G;I;I;B;G;I;B;C;L;I;B;I;C;C;L;L;L;L;L;L;L;L;L;L;B;B;I;I;I;M;G;G;G;B;B;B;B;F;F;E;L;L;L;I;E;E;J;B;A;A;I;C;B;B;L;L;L;L;B;I;B;B;D;D;H;H;H;H;F;B;A;H;G;E;E;E;E;D;B;L;C;D;M;B;B;B;H;I;F;F;F;G;A;G;L;K;E;L;F;B;I;H;H;H;H;I;I;I;I;I;F;F;B;G;C;B;I;F;F;B;E;I;F;I;F;E;E;I;I;B;L;H;H;F;F;C;B;B;L;F;B;A;A;M;G;G;G;C;C;C;C;C;G;C;G;E;K;A;E;E;C;F;F;L;C;J;A;B;B;B;I;I;I;H;H;L;E;B;I;L;B;L;A;A;I;B;K;E;H;H;B;B;B;I;C;E;E;I;A;F;C;D;D;E;E;B;D;K;A;I;H;A;C;G;C;C;G;B;I;L;A;A;H;B;I;I;L;L;L;L;L;B;B;D;J;A;A;F;J;B;B;I;B;F;F;B;D;D;D;D;D;D;B;I;D;G;C;C;D;A;A;B;L;E;E;A;A;A;M;F;A;C;D;F;C;L;L;A;B;B;F;G;F;F;F;E;E;A;A;H;F;B;G;L;H;B;A;C;C;C;C;B;H;L;A;A;A;F;B;I;L;L;L;L;L;C;D;D;L;D;E;B;L;B;E;E;E;E;E;E;H;B;B;B;I;A;A;A;B;K;E;B;L;I;F;B;I;I;I;G;C;L;L;I;I;I;E;C;G;I;L;E;C;C;D;D;B;L;K;D;C;C;C;C;E;E;E;A;G;B;B;B;B;I;D;C;E;D;E;I;F;F;B;L;L;L;L;B;A;L;E;B;I;B;I;I;B;B;B;F;L;L;L;E;L;L;E;E;A;D;D;C;E;D;A;E;H;H;K;K;G;G;G;C;G;H;K;L;G;L;L;L;L;L;L;L;H;H;H;H;G;D;D;E;E;E;L;L;A;H;H;E;H;A;C;B;H;L;E;I;H;E;E;C;L;L;L;L;L;L;L;L;L;L;L;F;I;B;I;D;A;A;A;A;A;L;L;L;L;L;B;C;E;E;G;C;G;C;K;K;K;E;F;H;H;H;B;C;E;I;H;C;D;D;D;L;L;L;L;C;C;L;L;E;E;E;E;E;E;E;H;H;H;A;A;I;A;A;F;B;B;G;G;B;I;C;D;F;C;B;C;D;L;I;C;C;L;L;I;D;D;E;I;C;C;C;C;C;C;D;D;D;D;D;M;D;D;B;C;F;E;E;G;B;B;B;L;I;E;E;E;E;C;C;D;D;E;E;E;E;E;G;I;D;I;I;A;I;I;B;B;L;D;B;E;A;E;L;L;L;L;L;C;C;F;F;C;E;L;L;E;B;I;A;A;C;B;B;I;E;B;F;F;C;H;C;B;B;L;L;G;L;B;C;C;D;D;D;D;D;M;D;D;D;D;F;F;H;L;L;L;L;A;A;H;I;J;I;E;E;E;D;D;B;C;L;K;I;E;F;B;B;A;H;J;B;I;I;B;A;L;G;F;C;I;I;B;L;L;H;H;L;C;E;E;E;D;D;D;B;L;L;L;L;L;B;C;C;C;C;C;G;G;C;B;D;B;G;E;I;E;E;C;G;C;I;L;L;L;L;L;L;L;L;F;B;F;I;D;B;I;C;B;C;B;L;L;H;E;J;F;L;E;A;F;D;C;C;B;B;M;H;H;H;H;G;C;C;C;C;D;D;D;D;D;D;D;D;D;D;L;L;F;L;L;C;A;C;C;L;L;L;C;E;E;B;B;F;G;G;I;I;F;F;F;E;I;F;F;J;A;B;C;C;B;B;B;A;C;C;I;B;L;B;L;L;L;L;C;C;C;D;D;H;H;B;I;I;C;L;D;G;B;C;C;L;L;L;L;L;H;H;E;G;C;B;B;C;C;L;F;A;A;I;I;I;B;L;A;A;B;K;A;A;I;F;B;A;B;L;L;L;E;E;B;B;I;E;E;F;F;B;B;I;L;L;D;L;F;B;E;C;F;H;H;C;E;E;E;C;L;L;L;K;E;E;B;L;H;E;D;G;I;I;I;F;F;E;B;L;L;L;B;L;E;E;B;L;L;B;I;E;H;F;D;E;B;L;B;B;C;I;I;I;D;F;C;B;L;L;L;K;K;K;K;K;K;K;K;K;K;C;D;D;D;B;B;L;L;L;L;L;L;L;L;H;A;G;B;B;F;F;H;I;L;L;I;A;C;D;B;B;B;B;B;B;I;I;F;F;B;L;I;I;I;H;I;H;K;C;C;C;D;I;I;I;I;L;L;L;E;C;I;E;I;I;I;E;H;E;E;D;D;D;D;D;E;E;L;E;E;B;L;L;L;L;L;C;A;I;B;E;E;E;A;B;C;G;G;A;A;E;K;C;C;C;D;I;I;I;I;A;C;B;B;L;C;D;E;L;F;E;E;E;A;I;C;L;L;L;L;L;F;G;J;C;B;B;L;L;L;L;E;I;A;B;C;E;E;E;F;F;B;E;B;C;C;D;D;F;G;G;B;A;A;A;A;I;I;G;D;B;I;E;B;B;B;B;B;B;E;B;E;H;G;C;H;G;F;H;K;L;B;I;I;I;I;I;G;F;F;B;G;E;H;C;B;A;C;C;B;F;C;D;L;I;B;L;H;G;C;D;B;L;E;H;D;D;F;I;F;F;I;C;D;D;D;H;E;H;A;A;B;G;D;B;H;E;E;L;L;E;E;E;E;K;K;K;K;K;K;C;E;B;L;L;L;L;L;L;A;E;H;C;D;B;B;B;B;F;F;H;H;C;E;E;C;C;C;D;B;G;D;L;L;C;B;E;A;E;I;D;D;B;A;E;E;L;A;I;I;L;L;B;I;B;A;D;M;G;G;G;C;C;C;C;C;G;C;G;B;L;L;I;D;L;L;G;G;G;L;L;G;I;I;C;F;L;E;I;I;C;E;I;I;I;I;A;A;A;C;I;I;F;H;H;B;B;B;B;L;L;L;B;L;I;I;I;I;I;B;D;I;I;I;L;E;D;E;G;B;F;C;L;D;D;H;H;H;H;G;G;E;D;A;G;F;I;I;C;C;G;C;B;B;C;F;D;J;E;B;B;L;L;L;L;L;L;B;L;B;I;L;B;L;L;I;B;G;B;I;H;C;A;B;A;E;F;A;C;C;L;L;L;A;C;C;C;L;E;I;I;B;B;E;E;E;A;A;I;I;F;A;C;E;C;H;G;B;E;A;C;F;A;C;C;C;C;E;E;E;B;B;B;B;A;A;I;E;D;L;C;L;L;G;G;G;C;D;D;B;H;H;B;E;H;E;E;F;F;E;C;B;L;B;A;C;H;D;E;L;A;I;G;G;C;C;G;G;B;A;C;D;L;B;I;L;B;B;B;B;L;D;L;I;I;A;A;A;A;A;D;D;D;B;L;L;L;L;L;L;I;I;I;L;E;E;E;E;E;E;E;E;I;A;B;E;A;E;L;E;E;B;B;B;B;E;E;E;J;L;F;B;L;L;E;E;G;B;I;G;I;B;L;E;E;J;B;B;B;I;I;K;D;I;D;C;L;L;L;E;E;A;G;I;I;A;B;L;I;F;D;E;L;G;B;A;I;I;C;B;B;L;B;C;B;B;L;L;E;A;H;D;L;L;L;L;L;L;E;E;H;H;B;C;C;J;J;E;H;C;C;C;C;C;C;C;D;B;A;F;I;C;L;L;E;E;E;D;B;L;B;B;B;B;L;L;B;G;G;D;G;B;L;L;G;B;B;E;B;E;L;I;F;C;L;L;L;L;L;L;E;E;B;I;I;I;H;K;L;A;I;G;A;F;B;I;I;E;E;D;D;D;L;L;F;C;B;I;A;A;I;I;C;A;C;H;D;B;B;H;H;B;H;H;H;A;D;A;A;A;D;A;A;D;B;B;E;L;L;L;E;F;E;B;L;E;F;L;L;F;A;B;I;F;B;H;I;C;G;D;L;L;C;C;E;B;B;J;B;D;D;G;F;A;C;L;L;L;L;K;C;D;B;E;F;F;B;F;F;L;F;F;E;B;B;E;A;D;D;E;B;D;D;L;H;A;I;I;B;B;I;J;E;E;E;E;L;E;E;E;E;E;E;B;B;H;E;E;E;I;I;I;C;C;B;B;I;F;I;F;I;C;C;C;L;H;C;C;G;J;L;F;I;L;E;L;F;I;K;K;K;K;K;K;F;C;C;C;B;B;B;B;B;B;B;I;I;I;I;I;F;H;E;I;L;L;E;B;B;B;L;L;H;B;L;B;E;C;F;H;H;D;D;E;A;L;E;E;F;B;B;B;B;C;B;G;D;D;A;L;A;I;I;F;J;I;H;F;A;C;C;C;D;D;D;D;D;D;D;B;L;L;E;E;F;H;F;I;A;L;L;L;L;L;L;L;L;L;L;F;F;B;L;L;L;L;D;B;H;B;E;B;D;J;I;C;C;G;C;L;E;E;C;A;L;I;C;C;L;A;A;A;A;A;L;A;I;I;B;A;B;I;A;A;B;A;A;A;G;B;B;F;F;J;C;C;L;L;D;B;I;C;G;E;E;E;E;J;J;C;E;B;I;I;B;B;B;B;B;I;A;I;B;E;E;I;A;L;H;B;B;B;H;D;B;B;F;I;I;B;L;E;K;L;C;E;E;E;H;I;I;B;I;I;F;F;D;D;G;F;F;F;E;E;A;A;C;E;E;E;J;E;E;J;E;E;C;B;B;L;E;J;J;H;H;G;D;D;D;D;D;L;L;E;A;B;H;H;H;A;A;A;A;D;K;K;G;G;G;C;I;B;L;L;H;H;C;C;G;D;D;L;B;B;B;I;I;G;F;G;C;C;B;I;I;I;I;D;H;I;B;F;I;C;C;B;E;F;A;D;B;A;L;E;B;L;L;L;L;A;E;C;C;E;E;B;B;B;L;H;H;H;H;I;D;H;G;G;B;E;F;G;M;A;A;L;L;L;L;I;H;H;A;B;B;E;A;G;B;B;B;B;D;D;D;D;A;D;D;D;A;A;A;A;G;B;B;B;B;B;B;B;B;B;B;B;L;I;I;L;A;E;L;L;L;L;L;L;A;H;H;H;H;H;B;C;C;F;B;L;C;I;C;B;B;H;F;E;B;I;E;I;I;I;I;G;B;F;F;F;C;L;L;L;L;L;I;A;A;G;G;C;C;G;G;F;F;A;B;B;B;H;E;G;G;C;C;K;K;D;K;B;I;E;G;C;L;L;I;F;F;F;J;E;F;L;L;H;B;G;H;H;H;H;H;H;L;C;C;L;B;B;B;B;B;B;I;I;F;L;L;L;L;L;H;I;E;A;C;L;A;L;G;B;D;B;I;B;B;E;B;B;G;I;B;H;E;I;E;E;E;E;E;B;I;I;G;B;F;L;I;I;F;F;E;G;C;B;H;B;L;A;G;C;C;L;L;L;L;I;I;A;L;L;A;A;A;A;A;B;L;I;H;F;F;F;F;F;F;F;F;B;E;A;G;G;B;G;I;I;I;E;E;E;E;E;A;C;C;C;C;B;H;H;C;C;D;D;H;H;C;L;L;F;I;L;E;D;D;D;L;L;L;A;B;D;B;L;B;A;B;L;L;E;B;A;A;M;G;G;G;C;C;C;C;C;G;C;G;B;D;D;E;E;B;I;B;D;E;H;C;C;C;C;B;B;I;B;B;K;K;K;K;K;K;C;E;B;L;L;L;L;L;L;A;E;H;C;E;E;E;H;B;L;L;H;H;B;B;E;D;E;B;C;L;L;G;I;A;A;E;F;B;B;B;E;I;C;D;L;I;I;I;I;L;A;E;B;D;D;E;E;E;E;E;B;C;C;C;C;G;D;D;D;D;D;D;D;D;D;D;K;B;I;H;B;I;B;I;C;G;G;G;E;H;A;A;A;B;B;I;D;C;C;C;J;B;L;L;E;E;E;E;D;L;E;F;H;K;B;F;B;I;I;L;L;G;C;C;C;C;C;B;B;E;E;E;E;E;A;A;C;D;B;B;B;B;B;B;I;I;B;I;B;D;B;C;A;E;E;C;D;D;L;L;C;A;F;D;E;B;L;B;E;E;E;I;J;H;H;I;A;F;F;C;C;B;B;I;F;C;C;C;C;G;D;D;D;D;D;D;D;D;D;D;G;C;A;B;B;L;L;E;E;E;H;K;L;I;I;C;D;D;C;D;F;I;A;L;B;A;E;I;B;G;D;L;F;B;G;G;E;L;G;B;F;C;C;C;C;C;G;C;G;C;D;B;B;L;I;I;B;E;B;L;L;B;B;B;B;G;C;C;E;E;B;B;B;B;F;F;B;I;B;J;B;C;I;I;F;B;L;H;C;C;C;H;C;D;D;D;D;B;E;E;E;A;G;E;E;J;C;C;D;D;D;A;D;L;L;I;C;C;C;C;L;L;E;E;E;B;J;F;A;A;A;D;L;E;D;K;B;I;B;F;D;D;D;L;E;E;A;F;C;C;C;B;I;E;B;I;C;C;D;I;D;D;D;D;D;E;E;L;E;E;H;H;H;C;D;D;L;B;B;I;I;J;J;L;I;G;G;E;E;I;F;D;B;C;G;C;E;G;G;B;G;L;L;L;L;L;K;B;J;C;B;F;C;I;I;B;B;L;G;B;B;B;B;L;L;L;H;H;E;G;B;J;L;A;E;E;E;E;E;F;B;B;I;E;E;C;L;L;L;I;I;I;I;F;A;A;A;C;D;L;D;D;G;B;K;A;I;F;E;E;E;E;G;A;A;E;I;A;A;G;B;B;G;E;A;J;C;H;H;H;K;C;B;J;F;F;F;B;I;C;F;A;A;G;G;C;C;G;G;F;F;K;I;L;L;E;E;E;E;F;B;H;H;E;E;E;D;L;L;E;B;D;B;L;H;H;H;H;A;B;L;G;E;E;E;L;E;F;F;F;F;G;C;E;F;I;B;F;F;F;A;E;E;H;I;I;I;A;E;B;C;B;E;D;L;K;G;G;H;B;L;L;I;H;L;B;B;L;A;B;B;H;B;B;E;E;D;L;E;L;L;M;B;B;B;G;G;C;C;G;G;B;D;E;H;C;L;L;L;L;D;E;L;I;F;L;M;G;G;G;A;A;B;B;B;A;I;G;G;C;C;B;B;C;C;F;L;L;L;L;L;L;L;F;F;F;F;L;L;B;B;A;B;B;L;L;I;D;H;E;I;I;I;I;B;C;J;B;E;E;L;C;C;D;B;L;L;G;B;L;L;I;I;C'.split(';')))


18988

In [174]:
len(pd.Series('A;B;C;D;D;D;B;E;E;A;C;C;B;F;B;F;D;D;B;F;G;B;H;B;B;I;E;E;F;F;B;F;F;F;H;H;D;D;D;D;D;D;D;D;D;E;E;E;E;E;E;E;H;I;I;J;E;I;K;B;B;L;I;B;A;C;B;B;F;F;C;I;C;B;B;B;F;I;I;B;E;H;C;L;E;E;E;C;C;I;B;I;K;K;D;D;B;L;L;H;H;H;C;A;A;B;B;B;I;F;F;B;A;L;L;L;E;A;G;B;I;I;A;A;A;M;D;E;I;E;I;L;F;B;A;A;B;H;L;L;A;B;B;I;I;I;C;B;L;I;I;J;A;C;C;B;H;D;L;L;L;L;L;L;E;E;H;H;G;E;C;B;A;A;H;B;F;F;F;F;I;I;I;F;F;F;B;B;C;L;I;L;L;L;B;I;I;I;F;H;H;C;C;C;C;B;B;F;I;B;B;E;A;F;B;H;B;E;A;C;C;B;I;I;K;B;E;E;B;F;F;C;J;J;B;L;B;D;L;H;G;I;C;B;B;E;F;C;C;L;L;L;C;C;C;F;F;E;E;B;G;F;F;C;B;L;C;C;B;F;A;B;L;F;F;F;H;F;F;B;B;F;F;A;C;C;C;D;D;D;D;D;D;D;B;L;L;E;E;F;H;E;B;B;A;I;K;K;K;K;K;K;C;E;B;L;L;L;L;L;L;A;E;H;B;B;A;A;I;J;E;G;J;C;E;D;B;A;I;E;B;L;G;I;C;C;C;C;B;B;M;H;H;H;H;B;B;B;I;I;B;F;I;A;B;L;I;I;L;E;I;E;E;L;G;G;A;G;D;C;F;F;D;A;I;F;B;B;L;L;L;E;C;B;B;B;H;H;H;B;B;B;C;C;D;D;D;D;D;M;D;D;D;D;F;F;H;F;E;B;B;F;K;K;K;K;K;K;C;E;B;L;L;L;L;L;L;A;E;H;G;I;A;A;A;B;E;D;C;I;C;E;E;E;E;E;B;E;E;E;E;A;B;B;B;I;I;F;F;F;I;D;D;H;I;J;I;I;I;F;A;A;B;H;B;C;D;B;B;B;B;B;G;C;C;C;B;E;E;E;J;L;B;I;I;I;J;C;E;K;K;K;B;E;G;A;I;B;H;C;G;B;H;G;F;H;L;L;L;L;L;L;L;F;F;C;C;D;L;I;I;B;E;E;L;L;E;E;E;E;E;B;L;I;K;F;C;I;I;I;I;I;I;I;I;I;I;H;E;L;L;C;C;B;C;L;G;D;L;L;I;B;G;C;J;H;H;H;B;G;G;F;F;F;A;A;G;G;B;B;L;E;C;C;C;C;B;L;D;B;J;J;B;C;B;L;L;L;B;B;J;B;C;C;C;C;I;I;I;A;C;D;B;B;B;B;B;B;I;I;F;G;D;L;B;B;L;L;E;A;E;L;B;I;I;H;H;E;E;E;E;E;E;F;L;H;A;L;E;E;E;E;H;C;E;G;A;C;I;C;B;I;F;F;A;B;B;F;B;A;E;G;C;D;E;I;K;I;I;D;F;L;F;L;B;I;L;L;L;L;L;L;L;L;L;L;L;L;L;L;L;L;L;L;K;C;B;B;L;E;E;E;E;E;L;A;A;B;G;L;I;I;I;I;A;A;A;A;A;D;F;K;D;L;C;L;E;E;H;H;H;H;A;B;M;L;L;H;H;G;I;I;C;L;L;L;E;E;D;D;C;L;B;L;E;J;B;B;B;B;B;L;F;E;F;H;I;I;F;E;E;E;E;E;C;E;A;A;C;I;F;J;J;A;C;C;C;D;D;D;D;D;D;D;B;L;L;E;E;F;H;L;L;L;L;L;B;L;I;F;D;F;I;I;C;C;D;D;F;F;H;B;J;G;B;C;C;B;B;D;I;C;F;E;I;B;H;H;H;H;L;L;L;B;D;I;E;H;D;C;K;C;D;L;L;L;E;G;F;I;C;D;B;B;H;H;H;I;L;L;A;A;L;D;L;C;G;L;E;E;I;F;D;F;I;A;L;H;A;E;I;A;C;G;J;L;G;B;E;G;B;I;F;B;F;I;I;B;B;B;B;B;E;E;E;E;E;A;I;D;B;B;B;B;I;I;I;I;J;C;B;L;B;B;L;D;D;D;F;B;H;B;D;I;L;B;B;E;B;B;B;B;I;G;C;B;B;B;H;B;B;L;I;I;I;I;D;I;C;J;J;A;B;G;C;I;I;F;L;G;F;B;B;L;B;E;I;J;B;C;B;E;B;E;C;B;L;E;B;K;C;B;L;B;B;F;A;A;I;I;B;C;F;C;B;L;A;C;B;G;I;H;E;E;D;E;G;C;E;L;C;E;E;E;I;D;A;B;I;I;I;I;F;B;A;D;D;I;I;F;D;L;G;C;L;H;B;E;A;G;B;E;I;E;E;A;B;C;C;C;F;C;C;E;E;E;F;I;H;H;B;I;L;E;B;C;L;G;B;I;F;B;B;B;D;B;L;L;L;B;I;I;K;B;B;I;C;G;A;D;B;I;L;L;L;L;L;L;A;H;H;H;H;H;K;K;K;K;E;L;H;J;K;K;D;D;F;B;B;L;H;F;F;I;I;I;G;F;F;F;F;F;E;E;F;B;B;B;C;E;L;B;B;B;A;M;M;K;L;E;A;A;A;C;C;E;I;I;L;D;B;E;F;F;F;G;G;B;G;L;B;B;G;I;L;C;H;F;L;L;E;E;E;L;L;L;L;L;L;L;E;E;B;F;B;G;L;B;A;F;F;B;B;I;B;D;B;F;F;I;D;B;I;A;I;J;D;F;C;D;L;L;L;L;L;I;I;I;I;H;B;F;F;B;L;L;L;L;G;E;A;B;A;E;B;B;H;H;B;E;E;C;G;E;F;E;A;C;C;C;B;E;D;B;E;C;C;C;C;D;K;K;L;L;L;E;E;E;B;A;I;H;H;H;C;B;I;C;B;B;A;C;L;A;B;B;L;L;E;E;E;H;B;L;E;C;D;E;E;L;I;D;L;L;I;B;F;H;A;D;B;K;A;F;B;B;L;A;L;L;B;E;D;B;I;M;I;I;I;I;F;M;C;E;E;E;E;H;H;H;A;A;B;D;I;I;L;L;I;I;H;D;D;D;E;M;L;E;G;D;D;D;I;E;J;L;K;K;I;A;L;E;F;F;H;L;A;B;C;C;B;B;M;H;H;H;H;K;K;K;K;K;K;F;C;C;C;B;B;B;B;B;B;B;I;I;I;I;I;F;H;B;L;G;L;L;L;L;L;L;B;I;I;I;F;I;B;B;L;L;L;C;F;F;F;E;G;G;K;C;C;L;L;M;M;B;I;I;F;D;L;C;B;I;A;C;C;I;H;E;J;I;B;F;G;B;B;D;L;L;L;I;F;F;B;F;B;D;D;B;B;B;E;I;B;B;B;G;G;D;L;L;L;H;I;C;G;C;B;B;E;L;G;B;C;C;B;L;L;E;E;F;F;B;E;E;B;B;D;D;L;C;C;C;B;A;A;A;A;B;B;A;C;B;B;L;L;L;H;H;H;H;I;G;L;L;L;E;E;B;B;J;E;F;I;I;I;I;E;E;E;E;E;L;I;M;L;E;D;L;L;E;B;G;I;H;H;H;H;G;L;I;G;B;L;L;G;I;I;C;C;C;C;C;C;D;D;D;D;D;M;D;D;G;A;B;G;K;K;K;K;G;G;L;L;L;L;L;L;L;F;L;L;H;L;H;H;C;C;C;C;B;B;F;B;B;B;A;G;C;K;L;I;I;C;C;C;C;B;I;F;L;C;L;L;L;L;L;L;L;F;F;C;B;B;C;D;D;D;D;H;B;E;E;C;C;E;B;D;B;L;I;L;L;E;E;E;E;G;L;B;B;B;B;E;E;G;G;C;D;L;L;L;A;A;H;L;B;B;B;L;E;H;H;L;I;I;I;I;I;L;L;A;L;L;E;E;I;L;F;F;E;E;E;I;I;I;C;C;G;I;I;C;M;A;B;B;C;D;D;B;B;B;B;I;E;E;E;E;G;G;D;B;I;E;F;F;F;F;B;B;F;F;D;E;E;L;I;H;B;I;C;D;D;D;B;I;G;I;F;F;F;F;I;L;E;F;F;J;B;B;B;F;F;G;G;G;A;A;C;C;G;I;I;I;I;B;D;B;B;B;I;H;B;B;J;C;B;B;B;F;D;D;F;G;H;H;I;J;D;D;I;F;G;L;A;A;A;D;I;I;G;B;C;C;C;C;B;H;H;I;I;L;E;B;A;C;L;L;L;L;I;I;C;I;H;E;J;E;E;L;L;B;E;E;E;D;L;H;G;D;B;B;B;B;E;E;E;H;L;L;F;B;G;F;F;L;L;E;I;I;J;G;B;I;E;E;E;E;F;J;B;I;A;A;I;I;B;B;I;A;C;L;B;I;L;A;H;C;C;B;B;B;B;B;B;L;L;L;L;L;L;I;I;G;B;B;I;F;I;B;L;D;E;C;B;B;B;D;B;A;A;E;E;I;B;F;I;L;G;E;E;E;G;B;B;B;B;H;H;H;F;H;H;H;H;F;B;L;L;L;L;L;L;A;H;H;H;H;H;B;C;E;H;B;L;L;I;B;E;G;D;D;A;A;A;H;E;D;B;A;A;L;I;J;H;I;L;B;L;L;L;I;F;F;G;K;C;C;B;E;E;E;E;E;E;H;I;C;B;B;E;C;B;B;L;A;I;B;B;B;L;E;E;E;H;E;D;L;L;L;L;L;L;L;L;L;B;I;H;H;H;H;H;I;B;B;B;B;A;A;I;A;B;B;I;I;C;E;F;D;B;E;G;L;L;L;L;L;C;H;L;L;H;H;G;E;A;G;B;B;F;E;D;I;C;D;G;B;I;I;C;B;L;C;B;E;B;A;A;I;I;K;A;H;L;D;C;F;F;C;F;F;H;B;A;A;B;I;A;E;L;A;A;I;F;F;D;L;B;E;E;E;B;B;D;L;L;L;L;I;B;E;E;E;B;H;H;H;C;C;A;L;L;L;H;H;H;E;F;E;B;E;E;E;E;D;L;L;G;L;L;H;B;B;C;F;I;I;H;H;A;B;D;C;B;E;E;F;B;L;L;L;E;C;B;D;D;E;E;E;E;G;C;L;L;I;I;I;D;E;J;L;I;G;B;B;E;B;B;L;F;D;F;B;F;F;B;E;E;E;B;I;G;B;D;B;B;I;B;I;E;L;L;L;L;L;L;I;E;L;B;I;C;B;L;E;L;L;I;I;I;C;C;I;I;I;F;C;C;L;L;E;H;C;C;B;B;B;B;B;B;L;L;L;L;L;L;I;I;E;D;M;M;L;E;B;B;B;B;B;E;E;B;A;E;D;H;H;H;H;H;H;H;H;H;H;H;H;H;H;G;C;C;C;C;D;D;D;D;D;D;D;D;D;D;L;L;F;B;J;D;G;F;F;D;L;B;B;B;C;B;E;E;F;B;F;F;J;B;A;A;A;D;D;D;L;A;A;K;L;L;C;F;G;B;L;F;J;A;A;F;C;B;G;L;L;E;E;E;F;D;I;K;K;A;B;B;B;I;I;C;E;E;E;E;L;D;A;C;C;D;E;C;D;E;E;D;D;B;D;C;C;L;L;L;L;L;L;F;E;D;D;B;L;F;C;B;B;E;F;D;B;B;L;A;E;E;C;C;C;C;E;E;E;C;E;B;B;G;B;F;B;H;F;F;I;L;F;E;C;C;C;G;I;G;C;L;B;G;C;I;L;C;B;B;B;A;A;H;H;B;B;L;K;C;C;I;H;H;C;I;D;A;B;C;L;L;L;L;K;D;E;E;E;E;E;E;E;E;E;E;E;H;I;I;I;F;A;A;F;B;B;I;L;L;F;F;G;G;G;C;D;D;B;H;F;B;F;I;B;C;C;B;B;H;K;L;J;L;L;E;E;E;E;E;F;L;C;L;L;L;L;L;L;L;F;B;C;E;E;G;D;B;F;F;B;K;K;B;H;H;K;F;C;I;I;I;I;I;I;I;I;I;I;K;G;G;H;A;A;A;E;H;H;B;G;H;E;L;E;E;E;E;E;G;F;F;I;I;I;F;B;B;B;B;F;J;F;F;A;A;A;B;B;D;C;I;F;G;C;C;A;A;G;G;C;C;G;G;F;F;A;G;I;I;C;B;G;B;G;L;B;J;A;I;I;C;C;B;I;L;L;L;E;E;E;E;B;F;L;E;B;E;B;L;E;B;B;I;K;D;D;A;G;G;B;B;B;G;B;L;L;A;I;C;G;B;B;I;F;H;C;L;L;L;F;F;I;B;L;C;C;L;L;L;I;I;I;D;D;D;L;L;L;L;F;B;B;L;L;E;C;B;B;K;K;C;D;D;D;D;M;L;L;L;L;L;L;L;D;B;E;I;E;E;J;B;L;L;L;L;F;F;F;F;F;H;D;D;D;D;D;E;E;L;E;E;E;E;G;D;B;I;B;B;F;L;A;C;B;I;C;L;L;L;L;L;L;E;E;F;F;F;B;I;C;C;C;E;E;C;D;C;F;G;G;F;B;K;G;C;A;F;B;B;B;L;L;F;F;F;B;L;H;B;E;E;C;B;B;I;D;K;B;E;B;C;B;B;H;F;E;E;B;A;D;C;C;C;G;G;B;B;I;I;B;I;J;J;F;L;E;C;E;L;H;G;F;F;G;D;I;D;L;L;H;E;E;E;B;G;I;M;C;C;G;I;I;E;B;I;A;B;B;B;L;E;D;B;C;C;C;C;D;E;B;B;C;B;D;B;B;D;F;G;B;I;B;E;E;F;D;D;D;B;A;B;B;A;A;A;L;F;I;K;K;C;B;L;L;G;G;F;F;F;B;I;F;A;E;B;A;G;E;E;I;L;D;A;A;A;A;A;A;A;K;K;D;E;B;A;B;D;I;I;I;I;L;E;L;L;L;L;A;B;E;F;B;B;L;L;E;B;E;E;E;E;B;M;G;L;B;B;G;I;I;E;L;L;A;A;G;G;I;E;E;E;B;L;B;L;B;F;F;F;G;C;B;F;H;I;K;K;K;I;I;L;I;B;B;G;F;B;I;A;I;I;I;I;I;B;E;K;K;E;D;B;B;B;F;F;E;E;C;L;E;E;E;E;E;A;E;E;E;I;D;B;B;C;L;C;C;B;B;L;I;E;I;A;B;B;C;D;L;L;L;A;J;G;B;B;B;L;L;I;B;B;H;I;F;F;K;K;D;D;L;L;L;H;I;I;J;E;A;G;B;B;F;C;B;L;L;L;F;F;B;B;G;D;D;C;B;L;L;L;L;L;L;E;E;A;F;L;I;C;E;E;E;E;B;B;B;B;E;E;H;C;D;J;L;L;B;B;L;L;H;J;B;B;B;F;F;D;D;B;A;C;C;I;H;D;L;L;C;L;L;B;H;E;F;F;E;H;H;E;K;K;A;B;L;L;G;A;E;F;I;I;I;C;C;C;B;B;D;K;E;C;B;I;I;A;C;C;C;F;A;B;B;F;A;L;K;E;I;G;I;I;I;I;C;E;G;B;B;I;F;A;G;G;G;I;E;E;E;E;F;F;C;B;L;C;B;J;L;I;I;L;I;B;I;F;A;A;A;A;A;G;D;B;F;F;F;F;F;C;C;A;I;C;D;D;K;A;L;A;E;A;A;D;L;B;E;E;D;L;H;B;L;L;A;J;B;I;E;E;F;F;B;L;L;L;L;G;C;I;F;I;L;B;L;L;L;I;C;I;B;B;L;H;C;C;D;E;D;C;C;B;G;E;H;H;A;L;K;L;L;L;L;L;A;A;A;A;D;I;G;L;A;B;D;B;A;E;E;C;B;G;E;E;J;K;H;E;G;G;C;B;D;E;E;E;K;B;I;D;B;A;L;F;D;G;C;L;F;F;E;I;E;B;E;I;I;H;E;H;B;B;B;B;B;B;I;I;F;I;E;B;D;E;I;L;F;L;H;L;L;L;L;H;B;G;J;B;E;I;E;C;B;D;B;B;B;A;B;I;B;D;D;B;E;E;E;E;D;D;C;C;D;B;C;L;L;B;B;B;B;D;L;L;L;L;G;A;A;L;G;I;H;I;H;I;I;E;B;I;G;H;E;E;A;B;B;B;I;I;I;E;E;F;A;H;I;A;A;A;A;A;A;C;B;D;B;I;F;A;A;D;D;C;L;L;L;L;D;C;F;F;B;B;B;B;B;B;B;K;B;D;D;B;I;I;I;I;H;H;H;H;B;F;F;F;B;L;L;L;L;L;C;B;L;I;J;D;B;L;B;L;L;L;L;I;H;H;B;F;G;B;C;E;E;E;B;E;E;I;L;B;F;D;F;B;B;B;L;L;L;E;H;C;C;B;B;B;B;B;B;L;L;L;L;L;L;I;I;G;D;C;F;I;I;L;I;B;I;F;F;F;F;G;C;C;B;H;C;C;F;B;B;L;I;B;F;B;B;I;F;I;G;C;C;C;B;E;L;D;B;B;L;L;L;H;E;E;A;A;A;A;A;G;G;E;E;E;B;F;C;B;B;B;E;I;F;B;L;B;D;A;A;A;C;B;B;B;I;H;F;I;C;C;L;L;L;L;L;H;B;A;L;B;I;I;L;C;A;A;C;C;C;G;G;B;B;I;D;F;C;B;I;I;I;I;B;D;D;H;H;L;L;I;I;L;K;K;K;A;A;L;L;E;E;F;F;F;F;F;B;A;A;E;L;I;I;F;I;I;I;I;C;G;L;L;L;L;L;L;B;J;B;B;B;I;H;E;E;I;L;H;A;A;A;A;G;B;B;B;B;C;B;A;G;B;C;B;B;B;A;A;A;M;B;K;G;B;L;F;I;E;L;L;F;L;I;I;E;E;E;E;E;E;B;B;I;F;F;L;L;A;D;E;E;F;B;L;L;I;B;B;B;L;F;L;I;D;B;B;B;B;I;F;C;L;L;E;L;L;J;A;A;A;A;C;M;G;F;F;K;I;H;E;E;D;D;E;B;I;B;B;D;L;A;D;H;B;G;B;B;L;G;B;B;E;I;G;D;B;E;E;E;I;B;L;L;A;A;I;B;I;I;G;C;C;C;C;C;B;B;E;E;E;E;E;I;A;A;E;L;L;L;L;F;F;I;E;E;B;A;H;L;C;I;F;F;B;C;C;K;L;B;B;L;F;G;B;E;B;G;F;F;A;C;C;C;C;C;C;C;C;C;F;I;I;L;E;L;L;F;F;F;F;E;G;B;I;G;B;B;I;F;I;I;E;B;D;A;D;D;I;I;E;H;E;E;E;A;A;G;I;A;I;B;L;L;L;F;F;L;E;G;I;I;I;F;F;G;G;E;L;L;E;G;G;L;L;A;I;I;F;B;E;E;E;J;I;J;E;E;C;D;D;E;F;G;F;E;E;E;E;E;J;C;E;E;E;E;E;B;I;L;L;L;L;I;D;B;B;B;I;I;I;L;B;G;D;D;A;A;A;B;L;L;A;A;A;A;E;C;F;F;B;B;B;L;I;B;I;C;D;C;L;L;L;L;H;B;B;E;D;D;L;E;A;A;C;E;F;L;L;E;C;F;F;A;C;C;C;C;C;C;C;C;C;I;I;I;I;A;I;I;B;B;A;L;E;E;E;B;I;I;L;L;H;H;D;B;L;L;L;F;C;E;H;H;H;L;I;I;I;I;D;D;I;B;K;K;E;L;I;F;D;F;F;F;F;B;B;B;B;B;I;E;H;D;F;E;I;G;A;B;B;F;F;H;C;L;L;I;H;I;A;J;D;B;C;F;H;G;L;B;A;B;B;B;I;I;I;H;H;D;I;E;I;G;I;A;L;L;F;A;G;B;L;A;A;E;E;E;B;B;L;B;B;K;K;C;B;L;L;B;I;I;E;L;I;F;F;L;B;J;J;B;D;B;L;L;L;L;E;B;A;A;A;A;A;E;I;B;B;B;B;L;L;G;F;B;B;B;A;B;E;I;C;E;C;K;G;B;A;I;B;B;L;D;B;E;B;B;F;E;L;A;I;J;L;I;I;J;I;H;B;E;B;L;G;A;B;B;B;A;D;C;B;F;B;I;B;L;B;B;C;C;D;D;D;D;D;M;D;D;D;D;F;F;H;G;K;B;L;C;D;D;G;D;E;E;E;E;C;C;C;C;C;B;B;C;F;B;I;I;D;I;D;C;C;B;E;A;G;L;F;L;F;I;F;G;D;D;E;E;E;I;K;K;K;K;K;K;K;K;K;K;C;D;D;D;B;B;L;L;L;L;L;L;L;L;H;B;A;L;H;C;C;G;D;D;D;B;C;K;L;C;B;L;L;L;L;L;L;L;L;F;B;A;G;B;B;E;E;A;A;E;E;B;B;B;A;A;A;A;F;G;F;F;F;I;A;I;A;B;C;C;I;M;L;I;B;E;E;A;G;G;F;I;I;L;L;L;L;L;E;E;E;E;E;E;G;B;E;E;E;C;C;L;L;L;L;L;L;L;F;F;L;E;H;I;F;F;B;B;C;L;L;L;L;L;L;F;C;A;G;F;H;H;A;A;A;A;F;A;F;F;F;B;D;H;C;C;H;C;L;L;L;E;I;I;I;A;B;H;L;H;A;B;J;D;F;F;M;B;F;G;E;C;C;C;L;D;D;D;L;L;L;L;H;B;F;I;C;B;D;B;I;I;G;I;B;B;B;B;E;E;A;B;A;B;D;L;I;I;I;L;C;E;I;I;I;L;L;F;D;E;E;K;K;A;H;I;I;J;L;L;C;D;L;L;I;H;L;L;L;L;L;I;G;F;C;L;L;L;L;L;L;F;E;I;E;E;D;B;B;I;J;B;I;A;E;I;D;D;H;G;A;L;L;L;L;L;H;B;B;I;H;E;I;I;H;C;C;B;B;B;B;B;B;L;L;L;L;L;L;I;I;C;F;B;I;I;A;A;B;B;I;I;I;I;I;F;H;H;H;K;B;E;E;I;C;L;E;F;B;B;B;B;I;C;B;L;L;L;I;L;F;F;F;F;F;H;F;L;A;A;G;B;L;I;I;E;L;L;A;D;A;A;C;B;E;C;B;B;L;A;A;A;C;C;I;L;B;K;C;C;D;D;L;E;I;I;H;B;I;G;G;C;C;G;G;K;C;D;D;D;D;M;L;L;L;L;L;L;L;F;F;C;C;C;I;I;F;H;H;H;D;L;L;L;I;C;C;B;L;L;L;L;L;L;L;F;F;K;I;I;I;D;D;C;C;I;A;D;F;H;H;H;H;H;B;I;F;F;J;C;B;G;G;D;L;L;L;L;L;L;L;E;E;I;C;B;E;E;I;F;B;B;B;F;I;I;I;K;B;I;E;I;B;B;B;G;I;I;I;I;D;I;F;D;B;E;G;B;L;L;F;I;I;J;E;E;I;D;L;L;L;C;B;B;B;A;A;H;H;B;B;E;E;E;D;D;D;G;B;L;L;L;L;E;E;A;A;C;I;H;L;L;E;A;L;L;A;A;C;C;C;C;C;I;B;C;D;E;C;B;B;E;E;E;E;E;E;C;A;A;A;M;L;B;E;B;A;C;B;B;I;I;I;J;G;L;L;E;H;C;I;I;A;C;B;K;K;K;D;D;E;E;L;A;E;G;B;C;C;C;C;C;F;C;C;C;G;I;C;B;G;I;B;D;G;B;C;A;A;D;D;D;D;A;A;A;G;B;B;B;B;B;B;B;B;B;B;B;L;K;C;D;L;L;L;E;A;C;C;C;L;C;C;G;G;I;I;F;F;I;C;C;C;I;I;F;H;H;H;D;E;H;H;C;C;C;C;B;B;F;G;L;D;G;B;B;L;L;L;L;E;E;H;D;D;D;E;E;J;C;D;F;C;I;D;D;D;D;C;L;A;A;B;B;L;L;L;L;L;E;E;A;A;B;B;C;G;H;H;H;H;H;H;H;H;H;H;H;H;H;H;I;I;D;G;G;I;E;C;E;E;I;F;F;B;L;L;L;L;C;A;B;I;G;B;C;J;J;B;A;C;C;B;D;D;L;B;B;B;L;I;B;B;B;A;H;H;I;I;B;D;B;B;C;F;F;L;D;D;D;E;E;E;H;C;D;C;G;D;B;B;B;B;I;I;I;G;C;C;C;B;E;B;A;I;I;L;F;I;B;B;E;L;I;B;D;D;B;E;L;I;E;K;K;K;K;I;I;B;B;L;L;H;D;G;A;A;B;E;A;L;E;H;I;L;C;L;B;C;C;H;H;L;L;E;B;F;B;E;L;I;E;J;J;G;A;B;D;D;D;E;E;E;G;G;E;H;B;L;L;H;H;L;D;F;F;B;F;A;D;D;E;E;I;I;I;I;I;G;I;K;F;C;I;I;I;I;I;I;I;I;I;I;C;B;I;I;L;B;B;I;J;A;A;I;J;B;L;C;B;B;L;B;I;C;C;C;C;I;H;H;H;H;L;G;I;I;I;G;H;I;E;C;C;F;F;C;D;M;G;K;K;K;K;G;C;L;L;L;L;L;L;L;F;L;L;L;L;L;L;A;H;H;H;H;H;A;F;G;A;A;A;A;F;C;B;E;E;F;C;B;K;L;H;H;C;C;C;C;B;B;F;K;C;I;D;L;C;L;L;B;B;C;F;C;B;E;E;J;C;E;H;H;J;B;E;H;I;I;H;L;L;L;L;D;L;B;B;B;E;E;I;K;G;B;L;L;L;L;I;I;I;I;B;C;B;B;D;K;A;E;D;D;E;E;E;E;C;B;B;L;F;G;G;B;B;B;L;L;L;L;E;E;H;C;D;D;D;D;D;D;B;L;L;L;L;L;L;L;L;L;I;D;D;I;I;E;I;B;B;L;L;L;L;A;E;A;I;I;E;G;A;G;B;B;E;K;K;E;I;I;B;L;I;L;L;L;L;I;I;B;I;K;B;H;C;H;B;H;H;G;D;D;D;D;H;L;L;L;E;H;C;B;I;D;B;B;D;A;D;B;D;D;L;L;F;B;I;L;A;B;A;C;D;B;A;C;C;C;F;C;B;G;B;A;G;B;A;H;C;D;B;B;G;B;I;L;E;I;H;H;F;I;I;C;C;F;F;C;A;E;C;C;B;D;D;H;H;B;I;C;L;D;B;B;E;A;A;G;I;L;M;M;G;I;I;I;E;C;C;B;L;L;E;E;L;E;E;E;B;F;G;C;B;G;G;E;F;B;E;L;B;D;H;J;D;L;I;A;A;G;G;C;C;G;G;F;F;G;D;D;D;D;D;E;E;L;E;E;B;I;A;D;L;L;L;E;E;B;L;D;B;B;F;E;E;L;L;G;H;A;A;A;A;A;C;C;C;C;C;B;I;I;I;I;F;H;L;I;A;C;C;I;E;H;E;L;E;E;E;E;I;F;F;F;F;B;I;I;F;C;L;K;E;L;E;E;E;E;H;K;H;B;B;A;G;L;L;I;C;B;G;L;L;E;E;H;L;D;I;K;H;E;E;H;G;A;L;L;E;E;F;I;D;E;E;E;E;E;A;A;C;L;J;G;I;L;L;H;G;I;E;F;C;B;B;E;E;E;E;E;E;E;E;E;J;I;B;B;J;E;B;I;B;B;C;D;A;E;H;H;A;I;L;G;I;E;E;E;E;H;B;B;A;D;D;I;I;H;I;I;B;C;C;B;E;E;E;A;G;G;B;H;H;B;G;E;H;H;E;L;E;C;B;L;L;E;D;L;H;A;A;B;B;E;E;E;B;E;F;B;I;E;E;E;C;C;G;K;C;I;C;D;D;L;A;A;B;B;K;L;E;E;B;A;E;L;E;I;B;C;M;M;L;I;G;B;C;L;L;L;L;L;L;L;L;L;L;D;L;A;D;G;C;A;I;K;K;K;K;C;L;L;L;L;E;L;L;D;B;B;I;I;A;L;H;L;B;D;D;D;B;E;G;G;A;A;A;A;D;A;B;I;I;I;F;G;E;F;G;G;D;E;I;F;E;I;I;L;I;L;L;L;J;D;B;B;B;B;B;B;B;I;J;J;E;B;B;C;I;K;K;K;I;I;I;B;C;B;B;I;F;L;G;G;G;B;L;L;B;L;I;G;D;L;E;L;E;I;E;F;F;B;L;L;L;L;B;C;A;A;C;D;L;L;C;C;E;B;L;L;L;L;B;K;I;L;B;B;B;B;L;L;F;I;D;B;E;E;E;C;C;A;A;I;I;L;L;L;C;L;L;L;H;H;I;D;C;C;C;C;C;C;C;E;A;L;L;L;I;H;A;A;A;D;A;D;F;B;E;E;E;C;B;L;I;F;F;F;F;E;D;D;B;E;E;E;F;B;L;L;L;C;B;F;G;F;I;E;D;B;L;L;L;G;M;B;B;L;E;E;F;F;B;D;L;E;F;F;B;L;L;L;L;E;E;E;E;E;A;B;B;L;F;I;I;I;B;F;F;F;F;D;D;L;E;A;B;I;H;J;L;L;D;E;E;E;E;E;E;F;F;E;E;A;A;A;F;L;F;F;F;F;F;H;F;F;C;L;F;G;E;E;L;L;L;L;L;I;I;I;I;L;I;F;F;F;F;I;K;K;B;K;C;C;I;H;H;B;I;B;B;G;D;D;A;B;B;B;B;B;B;I;L;L;L;L;F;A;G;G;G;B;E;E;A;A;C;I;F;E;K;D;H;A;L;D;B;A;A;I;A;E;I;B;B;C;H;E;B;A;A;M;G;G;G;C;C;C;C;C;G;C;G;B;B;C;G;D;D;A;M;B;B;F;E;G;I;I;I;A;A;I;I;L;E;D;L;L;I;H;H;E;E;I;G;B;B;B;I;L;D;D;D;C;A;I;E;E;J;C;B;E;C;B;B;L;L;K;K;F;A;A;B;D;D;I;I;I;I;B;G;D;D;D;G;I;I;I;I;I;A;G;B;B;B;B;G;C;C;C;C;D;D;D;D;D;D;D;D;D;D;L;L;F;D;D;D;L;L;L;C;D;D;D;D;D;D;D;L;L;L;L;L;L;L;A;A;A;A;A;A;A;G;A;A;C;C;B;B;B;B;B;B;B;B;F;B;G;L;I;I;F;F;L;H;B;E;I;I;B;B;I;A;I;L;L;E;F;C;L;C;C;C;B;K;H;E;G;C;D;F;B;B;B;E;E;A;B;B;E;B;D;L;L;L;L;L;K;B;B;H;H;H;C;B;E;D;L;K;K;D;D;L;L;L;H;E;E;E;E;F;L;L;L;F;B;B;I;B;I;A;A;A;M;G;G;G;B;G;A;I;F;F;A;L;A;A;A;A;A;A;I;I;G;G;B;B;I;I;I;F;C;L;C;B;B;D;B;D;L;B;E;C;F;B;C;H;F;J;G;H;F;G;G;B;B;I;I;I;F;A;A;I;B;B;B;B;F;A;A;A;M;E;E;B;B;D;C;C;C;C;E;E;E;D;L;I;G;J;E;I;H;H;E;E;A;A;A;B;G;F;F;B;C;B;A;A;I;I;G;B;B;I;L;L;E;B;A;E;I;D;L;L;G;D;D;C;B;A;H;I;E;E;F;F;F;H;K;K;G;G;G;C;L;E;L;L;A;L;C;C;D;E;L;L;L;L;E;F;A;A;B;I;C;C;C;C;L;L;L;L;L;L;E;E;H;D;I;F;F;E;E;E;E;E;E;E;E;E;B;B;L;I;D;E;E;I;E;B;B;L;E;E;E;L;G;B;B;D;L;L;L;L;I;I;A;B;C;C;B;B;B;A;B;A;A;I;B;J;L;E;C;B;I;I;I;L;C;B;B;I;I;I;L;J;D;L;L;E;B;B;F;F;F;F;F;D;D;B;H;A;A;F;L;H;H;H;E;B;L;L;H;A;A;I;C;B;B;D;D;K;E;H;J;A;B;L;B;B;I;I;I;C;G;B;B;L;L;L;G;F;B;L;C;B;L;L;L;B;I;F;F;A;I;D;H;H;C;B;B;I;D;E;E;A;I;I;E;L;B;E;E;E;B;B;C;E;E;E;E;B;L;G;A;K;K;K;K;I;E;E;E;B;G;L;H;H;L;L;L;L;L;L;L;H;H;E;A;A;A;E;L;L;E;E;I;L;C;E;E;E;E;J;J;E;F;M;L;L;L;F;F;J;D;F;F;F;K;G;G;H;C;E;A;B;I;I;C;E;E;E;E;B;B;H;I;I;I;L;E;E;C;I;B;B;L;L;L;E;B;A;D;D;I;I;A;K;E;B;F;E;E;I;A;D;C;I;I;C;C;L;E;B;F;G;D;D;B;I;I;I;J;A;D;B;B;B;B;H;B;I;D;B;D;L;I;I;I;I;K;K;K;K;I;H;H;I;F;G;I;B;B;B;B;H;H;H;I;B;E;A;A;C;B;B;E;E;L;I;D;D;D;B;L;E;E;E;E;E;B;E;E;E;E;B;E;D;B;B;B;F;B;M;M;L;F;F;B;E;C;B;D;B;L;C;E;E;E;D;B;I;I;I;I;I;I;H;B;A;C;E;E;L;F;B;L;K;K;B;I;I;I;A;E;E;C;L;B;C;B;B;E;B;L;L;L;H;C;I;B;B;L;B;K;I;E;E;E;E;C;G;L;E;E;D;D;B;B;L;K;K;K;K;K;K;K;K;K;K;C;D;D;D;B;B;L;L;L;L;L;L;L;L;H;B;E;E;E;E;E;E;I;I;J;L;L;H;H;H;H;H;H;G;G;D;L;A;A;B;D;A;B;C;C;C;C;C;C;C;D;L;E;L;F;F;F;F;B;A;E;I;K;K;K;K;E;E;H;H;H;E;E;E;F;C;E;C;C;L;L;L;L;L;H;H;E;E;E;C;I;C;C;L;L;L;L;L;H;H;G;C;A;A;B;L;L;L;H;H;I;B;F;H;B;E;I;A;L;E;D;A;D;B;I;F;F;G;C;C;C;C;D;D;D;D;D;D;D;D;D;D;L;L;F;C;B;I;B;L;A;L;L;E;L;C;G;G;C;C;C;C;C;C;L;H;H;L;D;D;E;I;F;I;E;E;B;B;G;B;B;B;F;F;C;H;F;G;G;C;C;G;G;C;D;M;B;B;L;L;L;A;G;G;D;H;H;L;C;C;C;C;C;G;C;B;K;D;L;F;B;F;I;I;B;F;B;E;E;E;E;I;D;D;C;B;B;L;L;L;L;I;E;E;E;E;E;C;B;I;B;E;C;L;B;B;B;C;B;B;B;D;L;C;C;C;F;B;B;I;C;B;B;I;I;I;F;I;I;E;F;F;B;B;E;E;E;E;E;H;E;A;C;C;A;C;C;C;L;B;A;B;B;B;B;C;C;D;G;I;K;B;E;E;G;G;B;I;A;A;C;C;I;I;C;E;E;E;A;A;G;L;K;C;C;I;H;H;C;E;K;I;F;B;A;D;B;I;A;C;C;L;L;L;L;L;L;L;I;D;D;L;E;G;D;D;A;A;A;E;C;L;L;H;H;H;H;A;C;C;C;G;B;A;A;L;H;H;H;H;E;B;I;I;I;A;A;K;K;I;K;C;D;L;L;G;G;C;D;H;I;D;D;L;I;A;H;H;C;B;B;C;B;L;L;H;K;A;H;A;G;B;E;F;F;F;B;I;I;E;E;I;E;L;A;C;B;B;I;I;I;C;E;E;E;F;F;L;L;I;B;G;B;I;E;E;L;E;H;H;H;F;B;D;D;B;B;B;E;E;E;L;F;F;F;L;L;L;L;L;L;B;L;L;B;K;K;K;K;H;L;L;B;B;B;L;L;H;H;I;I;I;I;J;J;J;G;D;D;F;I;L;L;B;B;L;L;L;L;C;C;B;D;B;A;G;I;I;I;L;L;H;I;L;L;L;A;A;G;F;C;B;G;B;A;F;A;A;F;H;H;H;H;H;B;I;I;G;A;A;A;B;B;B;I;I;I;F;F;C;C;C;H;H;H;H;H;H;H;H;H;A;G;G;G;C;C;B;E;E;L;L;E;E;E;E;E;C;G;I;E;D;D;C;C;B;B;C;C;C;C;G;D;D;D;D;D;D;D;D;D;D;D;I;I;I;I;A;A;I;I;K;K;I;I;A;B;C;I;E;J;K;K;L;L;L;E;E;I;A;B;I;A;A;L;L;I;I;H;H;G;G;G;B;B;B;B;B;B;B;B;B;B;B;B;G;J;C;E;G;G;B;D;D;D;D;B;I;B;E;G;B;A;A;I;I;B;B;B;G;I;I;C;C;E;B;B;B;I;C;I;A;F;C;D;D;D;L;L;L;L;L;E;E;E;B;D;D;E;E;E;E;L;A;A;A;A;C;L;B;B;B;B;C;C;C;B;K;L;L;L;L;F;H;H;K;C;D;D;D;D;M;L;L;L;L;L;L;L;B;B;B;A;K;B;C;B;L;E;E;E;G;E;E;E;E;A;L;H;H;H;H;G;G;C;D;D;D;D;L;L;K;E;H;J;B;C;I;H;A;L;I;E;A;B;B;I;B;B;B;G;B;E;G;F;F;F;F;E;F;F;E;E;E;E;E;E;E;E;E;K;A;A;A;A;A;A;H;H;A;I;I;I;H;D;B;D;E;E;C;C;L;A;B;E;E;E;E;H;H;E;E;E;E;E;E;F;C;B;B;C;L;L;I;C;C;C;H;C;L;L;L;L;I;I;B;I;G;D;D;L;L;L;E;E;I;I;F;F;F;I;G;B;L;C;H;H;F;F;B;K;K;K;K;K;K;C;E;B;L;L;L;L;L;L;A;E;H;G;B;I;I;I;L;F;F;B;L;A;A;I;C;C;C;C;C;C;C;G;C;A;A;B;E;G;I;A;F;B;L;H;H;B;B;B;E;E;A;D;D;D;E;E;E;B;G;F;F;B;A;A;E;E;B;B;E;F;I;G;B;F;F;H;H;I;J;I;B;A;A;M;G;G;G;C;C;C;C;C;G;C;G;L;E;B;B;L;L;L;E;E;E;E;C;C;L;L;I;E;L;E;H;A;D;E;E;E;C;A;A;A;H;H;B;B;B;E;L;L;I;E;E;E;L;L;E;E;L;A;A;L;L;L;H;B;E;E;L;L;L;B;C;L;E;D;L;A;C;B;E;L;H;D;B;E;I;E;C;E;E;E;E;B;B;L;L;L;E;F;L;B;A;A;M;G;G;G;C;C;C;C;C;G;C;G;C;L;E;D;D;D;D;H;H;C;C;B;B;B;B;B;B;L;L;L;L;L;L;I;I;E;C;C;I;J;I;C;B;B;E;L;B;C;B;B;E;F;G;G;C;C;C;C;C;C;F;A;A;F;I;E;E;E;I;I;I;E;I;M;B;E;I;I;F;B;B;D;H;L;L;H;E;A;C;C;C;D;D;D;D;D;D;D;B;L;L;E;E;F;H;B;F;F;F;C;C;C;B;C;L;L;I;I;I;G;I;L;B;B;B;B;B;L;L;A;B;L;K;K;K;K;K;K;C;E;B;L;L;L;L;L;L;A;E;H;B;L;A;A;E;H;F;A;B;I;I;I;D;L;L;C;C;L;L;E;E;E;E;E;E;E;H;H;H;D;H;L;L;E;C;I;G;I;F;C;G;C;B;E;G;A;A;F;A;D;D;I;I;E;L;L;I;G;D;K;L;B;B;E;D;B;L;L;B;H;C;G;D;D;L;L;L;B;C;C;E;I;I;I;I;I;E;E;G;B;I;H;B;L;H;G;G;L;F;I;A;A;I;I;I;I;I;I;B;L;B;G;I;B;B;L;L;L;L;F;F;F;B;I;K;C;L;D;C;D;D;D;B;C;F;F;A;B;B;B;B;B;B;E;I;I;A;A;C;C;B;B;B;B;B;B;B;B;F;I;L;L;G;D;B;I;I;L;I;I;B;B;D;C;B;E;B;B;B;B;B;B;L;B;I;I;I;F;I;J;B;E;L;E;H;H;H;H;H;H;H;H;H;H;H;H;H;H;K;K;E;J;B;B;F;I;B;H;J;A;D;D;I;I;C;B;B;B;I;I;I;I;F;H;L;L;L;C;E;D;B;B;G;G;E;D;B;L;B;B;A;A;A;A;D;H;H;H;G;K;K;B;B;B;G;B;I;C;C;B;B;L;C;B;L;L;G;B;B;F;J;D;D;D;D;B;B;B;L;L;L;H;L;I;F;I;G;L;L;L;L;L;H;H;H;A;C;G;C;C;C;C;F;I;C;A;C;K;C;E;L;L;B;H;A;A;A;F;B;B;B;D;B;D;L;D;E;B;F;F;J;G;I;I;I;F;A;G;C;E;E;K;L;I;I;D;D;L;L;L;B;E;E;E;E;B;B;B;E;D;H;L;L;F;A;I;J;L;L;H;B;L;H;G;L;I;J;D;B;I;I;L;L;L;L;C;L;L;I;I;C;L;L;B;L;L;L;L;L;I;I;I;G;I;A;I;G;D;D;A;A;A;I;I;B;D;I;I;G;E;D;B;I;I;I;F;B;D;F;F;D;D;B;B;B;E;E;E;E;B;B;A;D;L;A;C;C;F;C;E;E;G;G;C;D;D;D;D;L;L;C;C;C;H;I;I;J;B;B;A;H;L;A;A;A;A;F;L;I;I;L;G;D;E;L;L;F;B;D;B;B;I;I;C;D;M;B;B;E;H;G;J;B;B;I;A;I;G;F;F;B;G;L;B;D;B;B;B;B;B;K;I;F;F;L;L;L;D;E;L;K;C;C;D;D;L;E;G;G;G;C;B;D;E;E;E;B;E;L;A;A;A;C;E;E;E;H;I;E;E;L;L;I;F;I;F;I;A;A;D;I;D;B;B;I;B;B;I;I;E;E;L;E;E;C;E;C;B;L;L;L;G;G;L;E;L;L;B;B;F;L;I;E;B;A;A;A;A;A;I;I;I;I;I;J;G;E;G;C;H;A;E;A;B;B;I;I;B;D;B;L;I;L;B;I;A;L;L;L;L;F;B;H;I;B;K;B;I;H;B;B;A;A;J;B;B;A;A;A;I;L;H;H;H;H;B;C;B;A;A;A;A;A;D;L;G;B;F;B;B;B;H;E;C;C;C;C;B;H;B;L;L;A;I;I;I;A;B;E;E;J;B;E;E;E;E;L;E;E;A;K;K;K;A;A;L;L;E;E;F;F;F;F;D;H;E;L;L;E;E;G;G;H;I;I;I;I;F;I;I;I;B;B;B;B;L;F;A;A;E;G;C;M;E;C;C;B;B;E;G;H;G;L;L;L;L;L;L;L;F;F;C;B;L;I;L;L;I;A;G;C;E;E;E;C;A;B;B;L;E;C;I;L;I;C;F;G;L;A;B;G;H;L;B;E;E;L;L;L;L;L;B;B;I;C;D;L;E;C;C;B;E;E;C;L;D;D;A;E;C;B;B;B;B;A;F;A;A;A;H;H;H;B;C;C;B;D;C;F;F;A;E;E;E;F;F;L;E;I;C;B;I;I;L;L;F;B;A;B;I;D;B;B;B;A;B;B;B;L;L;A;A;E;C;C;L;L;L;L;L;L;K;A;I;C;C;B;G;B;A;A;G;G;G;L;L;B;B;A;A;A;A;A;I;I;I;I;I;B;B;D;L;L;H;C;B;B;A;D;E;B;I;G;F;F;B;D;L;B;A;I;I;I;I;I;E;E;C;C;C;C;B;H;H;E;L;E;J;E;F;D;D;I;C;C;H;B;G;B;C;C;A;L;A;C;D;B;B;B;B;B;B;I;I;A;L;I;H;I;C;E;E;E;L;B;E;C;B;B;H;F;B;B;B;A;A;A;A;I;I;G;B;B;C;H;B;B;G;A;G;I;D;D;D;E;G;C;C;C;C;C;B;B;E;E;E;E;E;I;K;K;D;G;E;E;E;I;E;E;D;I;I;I;I;I;E;E;A;C;C;C;F;D;C;B;E;J;I;L;F;F;I;I;B;G;K;K;B;B;E;E;L;L;L;L;L;D;G;B;L;I;C;C;L;C;A;I;I;I;H;E;C;C;B;L;F;F;F;G;B;B;I;G;C;B;B;K;K;L;L;L;L;I;I;I;C;I;B;B;I;I;G;F;F;F;F;F;F;F;F;L;I;I;L;G;L;L;H;H;B;B;L;E;B;D;A;F;G;B;B;J;B;G;C;C;C;C;D;D;D;D;D;D;D;D;D;D;L;L;F;E;F;E;B;A;B;B;A;A;A;A;A;I;I;I;I;I;G;I;I;B;L;I;E;E;H;H;B;A;B;B;B;B;E;E;E;E;E;E;E;E;E;J;G;I;I;I;E;E;L;D;F;H;H;D;G;E;B;A;G;C;A;B;B;B;B;L;I;E;E;H;H;H;F;B;D;D;B;B;B;E;E;E;G;C;G;L;B;B;E;G;J;B;A;B;E;I;B;D;C;C;G;C;C;D;D;E;E;E;E;I;G;C;D;D;I;B;G;B;D;D;D;C;I;I;F;B;E;B;B;B;B;C;C;C;I;F;C;E;E;F;B;B;B;I;E;B;A;E;A;I;D;D;C;F;L;L;L;L;L;L;L;L;C;L;I;B;E;A;A;L;A;A;A;D;D;E;B;B;I;C;C;L;J;B;D;E;E;A;A;L;L;L;L;I;C;D;L;L;B;B;B;A;H;H;A;B;B;B;B;B;D;D;H;A;L;H;B;E;D;D;D;L;L;L;E;A;E;E;E;C;C;B;F;F;E;E;B;B;E;E;J;D;B;I;I;I;F;K;B;L;D;D;B;C;C;B;G;B;F;B;C;C;C;C;C;C;C;B;B;B;H;H;H;H;B;D;D;B;B;B;E;E;E;F;I;B;E;E;G;B;E;E;H;H;F;B;D;A;B;B;L;I;E;E;E;E;E;E;C;L;L;C;E;E;E;I;I;I;B;L;L;L;L;E;E;H;I;B;D;D;D;E;E;E;H;B;J;B;B;B;B;E;B;L;C;B;B;L;L;L;L;H;H;E;B;B;B;L;L;D;D;L;L;L;E;D;L;B;L;L;L;C;C;B;B;B;B;F;B;B;G;F;F;F;B;E;L;H;B;B;F;F;B;L;A;F;D;D;E;E;B;E;A;K;K;C;E;C;C;C;C;C;C;D;D;D;D;D;M;D;D;A;C;B;L;L;L;L;L;L;E;E;B;E;I;A;A;H;I;I;E;B;C;C;C;C;L;L;L;L;L;L;F;E;E;A;I;I;L;D;E;E;E;E;E;F;G;F;G;G;I;A;J;G;D;L;A;C;B;E;E;E;A;C;C;C;F;F;F;F;F;F;H;H;H;H;B;D;D;B;B;B;E;E;E;B;B;B;J;I;L;E;G;G;F;F;F;A;A;A;A;A;A;I;I;B;I;D;B;F;L;L;L;L;L;C;B;B;L;I;I;F;C;A;E;A;A;D;F;H;L;I;J;B;I;C;E;E;E;E;B;B;F;E;C;C;C;H;L;L;C;C;C;C;C;C;C;C;D;I;H;H;C;C;F;K;A;A;A;C;B;B;E;E;E;C;C;C;A;J;B;F;G;I;C;B;A;I;L;D;B;B;C;L;I;I;H;B;L;L;A;I;B;E;E;I;I;C;C;C;C;B;C;L;L;A;A;L;B;B;B;B;A;I;I;I;I;I;I;I;G;B;B;I;F;B;A;C;C;B;F;B;H;L;I;B;L;L;H;B;I;I;L;L;L;E;G;B;L;E;C;B;B;G;B;L;H;L;L;L;L;K;A;G;D;D;A;A;A;B;I;I;I;L;J;L;L;L;L;L;L;E;E;L;C;B;L;I;D;D;D;D;E;E;E;I;C;G;C;A;F;F;L;D;K;K;K;D;D;D;D;D;E;E;K;K;B;I;F;L;L;I;I;A;L;L;H;C;C;C;B;B;A;B;H;D;I;G;A;G;E;L;I;I;I;I;H;B;L;L;K;B;D;B;B;B;E;B;B;E;D;C;K;B;H;G;B;E;I;J;D;B;I;H;I;I;B;B;I;I;E;E;E;E;E;H;D;B;B;H;H;E;H;B;G;C;I;H;H;C;C;C;C;B;B;F;B;B;D;F;G;C;F;D;D;H;E;F;E;B;G;F;D;B;B;C;L;B;D;L;H;L;A;A;G;G;C;C;G;G;F;F;B;G;I;I;B;B;A;A;A;A;A;B;K;K;B;L;I;I;I;I;A;B;L;I;B;K;B;A;D;M;G;G;G;C;C;C;C;C;G;C;G;G;E;H;K;B;L;L;H;L;L;B;I;B;B;L;L;H;H;C;B;E;B;B;B;G;C;G;G;L;L;L;L;L;L;L;L;F;D;D;D;D;D;E;E;L;E;E;G;D;B;B;B;B;L;E;F;B;L;L;L;L;L;I;I;I;C;B;B;L;B;I;I;J;B;C;C;E;C;L;H;G;G;B;D;B;L;E;I;B;B;C;D;B;A;L;L;C;G;B;C;B;B;H;D;D;D;I;I;L;L;L;L;L;L;E;E;C;F;D;A;A;C;G;D;H;L;F;B;C;B;C;C;L;I;I;I;I;D;L;L;E;F;D;F;C;C;C;G;C;C;C;G;D;D;D;D;D;D;D;D;D;B;B;B;B;B;B;F;K;K;K;K;K;K;K;K;K;C;B;D;D;D;D;B;D;D;D;D;H;L;L;L;E;E;E;H;H;B;D;D;E;F;A;D;D;D;D;L;H;F;H;F;L;L;L;L;L;I;I;I;I;G;F;G;L;A;E;E;B;A;B;E;L;L;F;F;F;F;I;B;B;B;E;E;E;H;A;H;I;I;B;B;G;D;C;F;B;G;B;L;B;L;D;B;B;D;E;G;C;B;A;L;I;I;D;I;D;B;I;F;I;I;C;H;H;C;B;I;B;C;C;B;B;M;H;H;H;H;G;F;F;D;E;E;B;B;B;B;E;C;L;H;F;A;A;B;G;F;C;F;E;A;B;B;B;B;B;C;E;C;E;I;I;I;G;C;B;B;E;G;K;B;J;L;L;I;I;H;I;E;B;B;K;B;B;B;B;B;I;D;L;B;H;C;F;L;L;L;L;H;L;C;C;C;C;B;F;G;F;B;G;G;A;A;B;B;B;F;I;C;C;C;C;D;A;I;I;G;B;B;B;B;H;H;H;F;H;H;H;H;F;G;I;I;A;A;A;A;A;C;C;C;C;C;B;I;I;I;I;F;H;B;D;D;D;D;B;I;C;B;J;H;I;E;E;E;B;D;D;D;D;D;D;H;H;H;H;H;H;H;C;L;B;B;B;F;B;E;E;I;B;B;L;G;B;L;B;K;L;L;F;F;I;I;I;F;G;B;B;B;E;G;L;B;L;G;K;K;K;I;I;I;H;G;G;D;I;G;A;C;C;D;E;C;H;H;H;A;A;A;A;L;A;L;F;F;F;F;G;B;B;B;B;B;H;H;H;H;H;C;A;A;L;L;L;C;C;C;D;I;I;E;L;E;E;E;E;E;E;E;I;L;B;L;C;B;I;C;B;B;D;E;A;I;L;I;F;F;B;E;E;A;A;A;A;M;L;L;L;L;L;L;A;H;H;H;H;H;G;F;F;F;F;F;A;A;F;B;F;J;A;A;B;L;I;I;I;H;A;B;K;B;H;F;G;B;I;I;G;F;H;L;I;I;I;I;B;B;B;B;I;I;B;E;D;A;A;A;L;I;B;E;J;F;F;F;I;I;L;A;B;L;L;L;G;D;A;L;L;F;B;B;B;I;G;E;G;B;L;L;H;C;I;I;I;G;F;F;C;A;A;A;H;H;H;E;J;G;C;L;I;G;B;L;L;L;H;I;I;I;I;I;C;C;F;C;D;L;L;L;E;F;F;L;B;C;C;F;G;L;L;A;G;G;I;H;G;I;H;F;B;B;B;B;B;B;L;L;L;I;L;B;B;A;G;A;A;A;A;L;A;C;E;E;B;B;I;L;L;L;L;L;L;I;E;G;C;E;G;B;D;C;L;A;H;A;A;A;B;H;C;D;M;B;B;F;F;C;C;L;L;E;E;E;E;E;E;E;H;H;H;B;B;L;K;E;L;G;A;A;E;E;I;H;L;L;L;A;H;B;B;D;A;B;I;I;I;H;B;G;C;D;L;F;E;B;G;F;B;H;L;L;G;B;F;C;B;D;I;I;I;H;I;J;L;H;C;C;B;B;B;B;B;B;L;L;L;L;L;L;I;I;G;E;E;I;I;I;F;L;G;A;A;L;E;B;L;A;A;A;B;E;I;I;H;H;B;I;C;C;H;H;D;D;B;B;L;F;L;A;E;E;F;E;L;D;B;C;C;L;L;I;L;L;K;I;B;C;L;L;E;A;H;B;B;B;B;A;D;D;I;I;A;A;D;I;F;C;C;B;B;B;B;B;F;F;L;L;F;F;H;C;I;I;D;B;B;K;K;K;K;K;B;F;I;I;I;L;K;K;E;E;E;H;H;G;B;B;D;H;D;L;L;B;B;L;I;B;I;I;G;B;L;L;L;I;B;C;C;D;D;H;H;I;E;I;J;B;B;E;B;E;F;B;I;I;L;J;L;K;K;F;I;H;H;D;D;D;D;D;D;D;D;D;E;E;E;E;E;E;E;H;D;A;C;C;I;G;I;D;D;L;E;I;G;D;L;L;B;B;B;G;B;K;A;B;E;I;I;I;E;L;L;L;C;C;C;C;C;G;G;C;B;D;B;B;C;B;B;C;F;F;F;F;L;I;B;F;L;L;L;B;F;F;I;I;A;A;G;G;C;C;G;G;F;F;E;G;B;B;I;I;F;B;C;B;B;L;L;L;L;I;A;C;E;G;B;B;B;B;A;I;L;B;L;B;B;C;G;B;L;H;C;L;E;E;E;E;E;I;I;I;L;H;D;D;D;D;D;E;E;E;E;E;K;I;H;D;B;B;E;B;E;I;I;C;D;D;L;G;I;L;A;I;D;C;F;L;I;I;I;I;L;E;L;E;H;G;C;C;C;I;E;L;L;L;L;H;G;B;E;E;E;B;E;E;E;F;F;G;C;C;E;B;F;F;J;L;A;D;C;C;D;B;L;G;L;L;L;E;L;L;B;B;B;L;I;G;E;F;G;G;C;C;C;B;E;C;C;G;G;I;I;E;A;C;C;I;I;I;L;A;A;H;A;A;L;H;K;K;K;K;A;G;B;L;L;L;L;L;L;L;A;A;C;B;B;E;E;A;B;I;I;L;A;C;C;I;J;L;J;E;H;G;F;L;H;H;L;L;L;L;L;L;L;H;H;B;L;B;L;L;A;F;B;I;I;F;B;F;H;D;D;L;L;I;B;F;B;B;I;B;B;I;A;B;J;F;F;B;I;F;F;I;A;A;A;B;F;E;B;I;B;A;C;C;C;D;D;D;D;D;D;D;B;L;L;E;E;F;H;I;B;I;H;D;D;D;E;D;D;G;I;C;H;A;E;E;C;B;D;B;B;I;I;I;I;A;B;A;D;D;F;E;B;D;D;D;B;I;I;I;I;G;D;D;E;B;F;I;B;B;C;B;I;D;L;L;E;E;F;F;A;A;C;E;E;E;E;E;F;J;A;E;D;I;A;B;C;C;L;I;I;I;C;D;C;C;C;B;B;L;L;E;I;C;B;L;L;L;B;L;L;L;I;E;E;E;E;E;A;G;E;E;E;C;C;L;L;L;D;F;F;G;D;B;B;B;B;M;A;A;A;G;L;F;C;B;L;E;B;C;G;B;L;L;E;F;I;I;E;G;F;B;B;I;J;B;C;C;B;D;D;E;A;A;B;A;G;B;B;B;B;E;G;L;L;L;L;L;L;L;E;F;F;F;A;C;C;F;L;H;K;K;K;K;K;K;K;K;K;K;C;D;D;D;B;B;L;L;L;L;L;L;L;L;H;B;I;I;G;I;F;B;B;G;K;K;B;H;H;E;B;I;A;B;E;G;B;B;K;K;L;L;L;E;E;A;L;C;C;C;C;C;B;I;H;H;B;L;L;L;H;B;C;K;C;C;I;H;H;D;D;B;B;B;L;E;E;F;A;B;A;I;F;C;D;D;D;D;B;B;E;C;B;E;C;C;C;D;D;D;D;D;M;D;D;D;D;F;F;H;C;I;I;I;B;B;B;H;D;D;L;C;B;B;L;L;L;L;B;B;B;F;D;B;C;L;L;L;I;G;A;B;L;A;I;G;H;E;B;E;E;E;I;E;D;B;L;B;B;E;E;E;E;K;H;B;D;A;A;I;E;I;I;I;F;H;C;G;M;K;H;G;L;I;K;A;B;C;C;D;L;H;F;I;I;I;I;B;F;F;D;H;L;D;B;A;B;B;H;K;B;I;L;E;E;L;I;L;E;E;I;B;B;L;D;L;A;B;A;A;A;A;G;B;B;B;D;D;C;B;A;I;D;D;D;E;E;H;A;A;A;A;A;I;C;B;B;B;A;A;H;H;F;F;F;L;A;B;K;F;F;D;B;I;B;I;B;C;B;L;L;H;B;I;I;A;F;G;B;B;B;B;L;L;L;H;H;B;B;F;E;B;C;C;H;C;D;B;L;L;L;L;L;E;E;E;E;A;B;B;C;B;B;B;B;A;A;A;A;B;G;G;F;F;F;E;E;E;E;L;A;J;F;B;B;F;J;F;E;F;B;A;B;L;B;B;G;F;B;B;G;G;G;C;B;B;E;E;E;D;D;D;D;D;K;I;C;L;L;E;E;E;A;I;B;J;E;H;F;I;I;D;L;L;H;E;C;C;B;B;L;L;H;I;I;B;I;C;C;C;D;D;B;B;B;B;B;B;E;E;C;B;L;I;E;B;A;A;A;E;E;E;E;E;F;F;F;F;E;G;G;A;C;I;F;I;K;C;H;G;G;B;D;B;B;L;E;J;E;E;F;C;D;F;F;F;H;L;H;D;D;B;B;A;E;I;B;G;E;H;L;I;L;I;I;I;B;D;L;H;G;G;E;F;F;E;C;I;I;I;I;C;I;E;D;C;C;B;L;L;L;L;F;L;L;G;I;F;M;M;L;I;E;E;C;D;A;B;I;B;B;B;B;A;A;B;I;I;I;B;B;B;I;I;J;I;E;E;L;B;A;B;B;L;E;D;I;G;F;A;A;F;C;B;E;E;G;B;E;G;C;C;C;D;E;L;L;L;L;E;H;E;E;A;I;I;L;I;L;L;I;B;E;E;B;E;E;A;A;B;A;A;A;I;A;E;E;E;H;B;L;L;E;I;I;I;L;E;E;F;F;F;I;J;J;B;B;L;B;E;B;F;C;E;B;E;E;A;A;B;B;I;F;F;D;F;H;F;B;B;B;B;B;G;I;F;C;E;H;B;B;D;D;G;B;L;L;L;H;H;C;J;B;H;I;C;E;E;F;F;A;C;F;F;H;B;H;C;D;B;B;L;L;L;L;E;A;I;J;L;E;C;G;A;I;I;I;E;G;E;L;E;J;D;D;L;L;B;L;A;E;E;B;A;A;B;B;B;B;B;H;I;F;I;I;L;I;A;A;A;A;D;H;H;H;G;I;B;L;E;C;B;G;D;D;L;C;F;F;G;C;C;G;M;M;F;B;C;L;L;L;L;L;L;E;E;B;D;L;D;J;E;F;F;F;L;L;H;L;L;A;A;I;I;E;A;B;H;H;H;H;B;D;D;B;B;B;E;E;E;K;F;H;C;C;B;B;B;B;B;B;L;L;L;L;L;L;I;I;F;F;B;G;B;D;D;I;A;L;I;K;K;L;L;L;E;E;C;B;A;D;H;G;E;E;E;E;E;E;E;B;B;B;B;B;I;J;B;L;E;E;I;B;B;B;L;F;B;H;E;E;B;B;B;D;K;K;K;K;K;K;D;B;I;H;I;G;I;I;C;L;L;L;B;L;L;G;F;F;L;B;B;E;G;L;G;C;C;C;F;L;A;B;B;B;E;E;E;L;L;L;E;I;E;D;H;I;A;I;B;L;L;I;I;L;L;L;H;B;B;I;F;C;C;C;G;C;C;C;G;D;D;D;D;D;D;D;D;D;B;B;B;B;B;B;F;D;E;F;A;F;B;B;B;G;B;L;A;B;I;I;B;E;B;A;B;B;L;L;I;I;I;D;L;B;A;H;E;B;B;F;L;B;E;E;E;H;H;B;I;I;B;B;B;B;B;F;L;L;L;E;E;L;L;E;H;B;L;L;I;B;L;I;L;H;B;B;B;L;L;E;E;D;C;F;H;L;L;L;L;A;E;B;A;A;C;E;F;I;F;C;H;G;G;C;I;A;G;B;B;B;E;G;D;D;A;A;A;B;E;E;E;A;I;I;I;D;F;C;C;C;C;C;C;D;D;D;D;M;M;D;D;I;A;B;F;G;B;G;C;I;B;B;B;B;D;E;I;G;L;L;L;D;D;D;D;A;B;D;I;J;B;B;B;K;C;L;L;L;L;I;H;H;H;H;E;I;B;B;K;A;L;L;B;B;A;I;B;L;J;M;M;B;B;C;H;H;B;B;E;E;B;B;B;A;F;F;B;E;D;H;I;H;K;M;M;B;D;D;E;B;B;B;B;I;I;I;I;F;E;A;B;B;A;A;A;A;A;B;L;L;H;D;A;C;C;F;I;I;A;A;G;C;A;I;H;H;H;H;H;H;E;H;C;H;F;I;I;F;L;H;G;C;C;C;C;E;I;L;H;L;I;B;F;B;B;B;B;I;E;F;B;G;D;D;D;F;B;B;J;K;K;A;A;A;E;E;E;L;B;B;L;I;F;C;L;E;I;A;E;B;D;L;H;L;E;L;L;L;L;E;L;I;C;B;A;B;L;B;B;B;B;B;L;L;F;I;L;C;C;A;D;L;I;B;I;E;E;L;B;C;L;I;I;E;A;A;A;A;A;G;B;B;B;E;F;I;E;C;H;G;A;B;H;C;C;B;B;B;B;B;B;L;L;L;L;L;L;I;I;E;E;B;C;C;B;L;I;I;E;B;C;C;H;H;E;C;B;B;C;B;I;I;G;L;L;A;F;F;F;F;F;C;D;D;L;B;B;B;I;G;B;I;I;B;H;H;H;H;D;H;L;A;G;I;G;B;B;J;I;E;L;E;L;L;I;H;L;G;B;E;H;H;L;E;E;E;E;D;I;I;F;L;H;H;I;H;E;E;E;G;H;K;I;C;C;C;C;C;C;C;C;B;B;F;F;A;A;I;D;L;L;E;E;E;E;B;B;C;H;F;F;E;C;K;B;B;L;E;B;L;D;L;E;E;B;A;I;I;I;L;E;I;D;A;A;I;I;A;L;C;I;I;K;C;D;D;D;D;M;L;L;L;L;L;L;L;E;D;B;B;E;E;E;A;A;D;C;G;M;L;I;I;G;E;E;E;B;G;L;L;L;L;L;H;H;H;B;B;L;E;A;B;G;B;B;I;I;B;J;L;B;E;E;I;H;A;A;C;E;I;I;C;H;I;H;I;I;I;I;I;B;C;A;L;L;B;I;C;C;F;F;A;C;B;G;B;L;E;H;I;H;H;H;H;H;H;H;H;H;H;H;H;H;H;E;G;B;K;E;H;I;D;D;D;A;A;A;A;D;L;L;H;J;E;D;D;M;B;I;D;L;L;I;F;D;G;G;L;D;L;L;C;C;C;C;E;E;E;L;L;D;D;E;E;B;E;D;D;F;F;C;D;L;L;L;L;L;L;B;G;C;A;A;A;I;C;B;E;A;I;I;I;D;L;L;A;A;L;L;L;L;L;L;E;E;I;G;H;H;F;F;L;I;B;I;L;I;L;K;K;L;L;E;L;E;E;A;A;A;A;B;B;B;B;B;L;I;I;I;F;C;E;H;B;I;I;I;I;D;L;L;L;L;D;I;E;C;I;L;L;F;A;B;I;C;F;F;F;F;L;I;D;I;I;I;I;I;B;A;G;G;B;B;F;A;A;A;D;D;D;L;B;I;J;L;H;E;G;C;B;B;I;A;L;L;L;B;F;F;F;H;I;B;F;F;G;I;I;B;H;H;H;F;D;D;D;B;B;B;E;E;E;I;D;D;H;B;I;A;B;B;E;E;C;C;C;C;B;B;B;B;B;B;L;L;C;B;L;I;E;E;E;E;K;K;A;A;A;E;E;E;C;D;L;F;B;D;L;E;E;L;L;I;G;B;B;B;B;L;L;L;H;H;J;B;C;D;L;L;L;K;E;E;E;E;D;C;B;L;L;H;E;E;F;L;L;E;B;E;E;E;E;F;B;G;B;L;L;I;E;E;K;G;G;H;I;B;E;E;E;E;E;F;E;B;B;B;B;B;I;G;A;B;C;C;B;B;B;C;A;I;F;E;E;I;G;B;B;A;A;L;E;K;I;B;B;G;G;I;A;E;E;E;E;E;E;E;E;E;E;E;E;E;L;I;H;H;C;M;H;B;B;J;L;A;A;A;I;B;B;B;B;B;B;B;G;C;E;D;D;D;G;B;A;A;D;E;E;E;E;E;F;F;F;F;E;A;L;L;E;E;B;K;I;E;E;E;E;B;E;E;C;F;B;B;B;B;F;F;F;I;L;L;L;B;B;G;B;I;L;L;L;L;I;I;I;F;I;E;E;E;C;F;L;L;E;E;E;L;I;I;L;L;F;C;H;B;B;L;E;K;K;E;E;C;C;D;D;D;D;D;M;D;D;D;D;F;F;H;C;C;F;B;B;L;I;F;F;B;B;B;I;H;L;B;F;F;H;B;C;F;F;F;B;B;B;B;L;L;L;L;L;L;F;C;E;E;E;E;J;J;J;C;H;F;F;I;D;L;A;A;A;I;I;E;L;L;L;C;K;K;K;I;I;I;H;L;F;F;F;F;F;B;H;F;L;L;L;A;A;H;A;B;B;I;A;B;I;C;A;B;B;L;L;L;L;E;E;H;I;E;E;I;B;C;C;L;L;L;L;L;L;L;L;L;L;I;H;D;B;B;H;H;F;I;G;A;L;L;D;E;B;H;A;A;C;H;L;L;L;B;I;F;F;H;H;G;L;L;H;F;F;I;C;C;D;D;D;D;D;M;D;D;D;D;F;F;H;E;C;C;C;B;F;F;F;F;E;F;F;K;B;G;G;B;B;I;I;I;F;C;C;C;C;B;H;C;L;B;B;B;B;I;L;L;E;D;D;L;E;C;K;K;K;F;I;L;L;L;D;G;H;D;L;L;A;H;A;A;L;L;E;B;I;I;L;F;H;G;A;A;G;B;I;I;L;I;I;F;F;D;A;A;C;C;C;B;D;D;L;L;E;E;L;H;H;H;L;L;L;L;L;L;F;F;H;E;J;I;I;B;B;I;L;C;E;E;E;H;B;B;L;L;F;E;E;B;E;F;F;K;H;F;H;I;H;J;I;J;I;C;G;H;H;A;B;C;B;B;L;A;E;D;D;I;L;B;G;D;B;E;E;C;F;F;H;H;E;E;L;B;I;E;C;L;L;L;H;H;C;I;B;B;B;I;F;A;A;A;I;H;C;C;C;A;E;G;I;B;D;D;D;D;I;I;J;L;E;B;B;F;C;L;H;B;B;L;L;H;L;A;E;I;F;F;I;B;E;B;I;H;H;C;C;B;B;B;B;B;B;L;L;L;L;L;L;I;I;C;D;D;D;D;I;B;B;G;G;G;G;B;C;L;H;A;E;G;F;B;E;L;I;F;L;A;J;H;F;I;I;I;K;A;F;F;H;H;F;F;F;I;I;F;I;L;I;E;G;A;B;B;B;C;C;L;L;C;A;I;C;F;H;H;F;F;I;B;B;B;B;A;A;I;D;B;D;D;D;D;D;B;L;C;L;C;C;I;A;B;B;I;I;F;I;I;D;J;J;F;F;B;L;L;L;L;C;H;H;H;C;C;D;D;L;L;L;C;D;D;D;D;D;D;D;L;L;L;L;L;L;L;A;D;E;C;C;D;D;D;D;D;M;D;D;D;D;F;F;H;C;C;I;B;F;H;H;H;H;H;H;H;H;H;H;H;H;H;H;H;B;H;L;E;C;F;F;C;L;L;L;H;G;G;B;I;B;G;I;F;G;F;G;B;L;E;L;F;F;B;L;L;L;L;H;E;E;E;E;F;I;D;E;E;A;E;E;F;F;B;L;B;G;C;B;B;B;I;C;A;B;B;I;B;B;I;I;F;F;F;H;H;L;L;F;G;F;G;I;I;B;G;I;B;C;L;I;B;I;C;C;L;L;L;L;L;L;L;L;L;L;B;B;I;I;I;M;G;G;G;B;B;B;B;F;F;E;L;L;L;I;E;E;J;B;A;A;I;C;B;B;L;L;L;L;B;I;B;B;D;D;H;H;H;H;F;B;A;H;G;E;E;E;E;D;B;L;C;D;M;B;B;B;H;I;F;F;F;G;A;G;L;K;E;L;F;B;I;H;H;H;H;I;I;I;I;I;F;F;B;G;C;B;I;F;F;B;E;I;F;I;F;E;E;I;I;B;L;H;H;F;F;C;B;B;L;F;B;D;A;M;G;G;G;C;C;C;C;C;G;C;G;E;K;A;E;E;C;F;F;L;C;J;A;B;B;B;I;I;I;H;H;L;E;B;I;L;B;L;A;A;I;B;K;E;H;H;B;B;B;I;C;E;E;I;A;H;C;D;D;E;E;B;D;K;A;I;H;A;C;G;C;C;G;B;I;L;A;A;H;B;I;I;L;L;L;L;L;B;B;D;J;A;A;F;J;B;B;I;B;F;F;B;D;D;D;D;D;D;B;I;D;G;C;C;D;A;A;B;L;E;E;A;A;A;M;F;A;C;D;F;C;L;L;A;B;B;H;G;F;F;F;E;E;A;A;H;F;B;G;L;H;B;A;C;C;C;C;B;H;L;A;A;A;F;B;I;L;L;L;L;L;C;D;D;L;D;E;B;L;B;E;E;E;E;E;E;H;B;B;B;I;A;A;A;B;K;E;B;L;I;F;B;I;I;I;G;C;L;L;I;I;I;E;C;G;I;L;E;C;C;D;D;B;L;K;D;C;C;C;C;E;E;E;A;G;B;B;B;B;I;D;C;E;D;E;I;F;F;B;L;L;L;L;B;A;L;E;B;I;B;I;I;B;B;B;F;L;L;L;E;L;L;E;E;A;D;A;C;E;D;A;E;H;H;K;K;G;G;G;C;G;H;K;L;G;L;L;L;L;L;L;L;H;H;H;H;G;D;D;E;E;E;L;L;A;H;H;E;H;A;C;B;H;L;E;I;H;E;E;C;L;L;L;L;L;L;L;L;L;L;L;F;I;B;I;D;A;A;A;A;A;L;L;L;L;L;B;C;E;E;G;C;G;C;K;K;K;E;F;H;H;H;B;C;E;I;H;C;D;D;D;L;L;L;L;C;C;L;L;E;E;E;E;E;E;E;H;H;H;A;A;I;A;D;F;B;B;G;G;B;I;C;D;F;C;B;C;D;L;I;C;C;L;L;I;D;D;E;I;C;C;C;C;C;C;D;D;D;D;D;M;D;D;B;C;H;E;E;G;B;B;B;L;I;E;E;E;E;C;C;D;D;E;E;E;E;E;G;I;D;I;I;A;I;I;B;B;L;D;B;E;A;E;L;L;L;L;L;C;C;F;F;C;E;L;L;E;B;I;A;A;C;B;B;I;E;B;F;F;C;H;C;B;B;L;L;G;L;B;C;C;D;D;D;D;D;M;D;D;D;D;F;F;H;L;L;L;L;A;A;H;I;J;I;E;E;E;D;D;B;C;L;K;I;E;F;B;B;A;H;J;B;I;I;B;A;L;G;F;C;I;I;B;L;L;H;H;L;C;E;E;E;D;D;D;B;L;L;L;L;L;B;C;C;C;C;C;G;G;C;B;D;B;G;E;I;E;E;C;G;C;I;L;L;L;L;L;L;L;L;F;B;F;I;D;B;I;C;B;C;B;L;L;H;E;J;F;L;E;A;H;D;C;C;B;B;M;H;H;H;H;G;C;C;C;C;D;D;D;D;D;D;D;D;D;D;L;L;F;L;L;C;A;C;C;L;L;L;C;E;E;B;B;F;G;G;I;I;F;F;F;E;I;F;F;J;A;B;C;C;B;B;B;A;C;C;I;B;L;B;L;L;L;L;C;C;C;D;D;H;H;B;I;I;C;L;D;G;B;C;C;L;L;L;L;L;H;H;E;G;C;B;B;C;C;L;F;A;A;I;I;I;B;L;A;A;B;K;A;A;I;F;B;A;B;L;L;L;E;E;B;B;I;E;E;F;F;B;B;I;L;L;D;L;F;B;E;C;F;H;H;C;E;E;E;C;L;L;L;K;E;E;B;L;H;E;D;G;I;I;I;F;F;E;B;L;L;L;B;L;E;E;B;L;L;B;I;E;H;F;D;E;B;L;B;B;C;I;I;I;D;F;C;B;L;L;L;K;K;K;K;K;K;K;K;K;K;C;D;D;D;B;B;L;L;L;L;L;L;L;L;H;A;G;B;B;F;F;H;I;L;L;I;A;C;D;B;B;B;B;B;B;I;I;F;F;B;L;I;I;I;H;I;H;K;C;C;C;D;I;I;I;I;L;L;L;E;C;I;E;I;I;I;E;H;E;E;D;D;D;D;D;E;E;L;E;E;B;L;L;L;L;L;C;A;I;B;E;E;E;A;B;C;G;G;A;A;E;K;C;C;C;D;I;I;I;I;B;C;B;B;L;C;D;E;L;F;E;E;E;A;I;C;L;L;L;L;L;F;G;J;C;B;B;L;L;L;L;E;I;A;B;C;E;E;E;F;F;B;E;B;C;C;D;D;F;G;G;B;A;A;A;A;I;I;G;D;B;I;E;B;B;B;B;B;B;E;B;E;H;G;C;H;G;F;H;K;L;B;I;I;I;I;I;G;F;F;B;G;E;H;C;B;A;C;C;B;F;C;D;L;I;B;L;H;G;C;D;B;L;E;H;D;D;F;I;F;F;I;C;D;D;D;H;E;H;A;A;B;G;D;B;H;E;E;L;L;E;E;E;E;K;K;K;K;K;K;C;E;B;L;L;L;L;L;L;A;E;H;C;D;B;B;B;B;F;F;H;H;C;E;E;C;C;C;D;B;G;D;L;L;C;B;E;A;E;I;D;D;B;A;E;E;L;A;I;I;L;L;B;I;B;A;A;M;G;G;G;C;C;C;C;C;G;C;G;B;L;L;I;D;L;L;G;G;G;L;L;G;I;I;C;F;L;E;I;I;C;E;I;I;I;I;A;A;A;C;I;I;F;H;H;B;B;B;B;L;L;L;B;L;I;I;I;I;I;B;D;I;I;I;L;E;D;E;G;B;F;C;L;D;D;H;H;H;H;G;G;E;D;A;G;F;I;I;C;C;G;C;B;B;C;F;D;J;E;B;B;L;L;L;L;L;L;B;L;B;I;L;B;L;L;I;B;G;B;I;H;C;A;B;A;E;F;D;C;C;L;L;L;A;C;C;C;L;E;I;I;B;B;E;E;E;A;A;I;I;F;A;C;E;C;H;G;B;E;A;C;F;A;C;C;C;C;E;E;E;B;B;B;B;A;A;I;E;D;L;C;L;L;G;G;G;C;D;D;B;H;H;B;E;H;E;E;F;F;E;C;B;L;B;A;C;H;D;E;L;A;I;G;G;C;C;G;G;B;A;C;D;L;B;I;L;B;B;B;B;L;D;L;I;I;A;A;A;A;A;D;D;D;B;L;L;L;L;L;L;I;I;I;L;E;E;E;E;E;E;E;E;I;A;B;E;A;E;L;E;E;B;B;B;B;E;E;E;J;L;F;B;L;L;E;E;G;B;I;G;I;B;L;E;E;J;B;B;B;I;I;K;D;I;D;C;L;L;L;E;E;A;G;I;I;A;B;L;I;F;D;E;L;G;B;A;I;I;C;B;B;L;B;C;B;B;L;L;E;A;H;D;L;L;L;L;L;L;E;E;H;H;B;C;C;J;J;E;H;C;C;C;C;C;C;C;D;B;A;F;I;C;L;L;E;E;E;D;B;L;B;B;B;B;L;L;B;G;G;D;G;B;L;L;G;B;B;E;B;E;L;I;F;C;L;L;L;L;L;L;E;E;B;I;I;I;H;K;L;A;I;G;A;F;B;I;I;E;E;D;D;D;L;L;F;C;B;I;A;A;I;I;C;A;C;H;D;B;B;H;H;B;H;H;H;A;A;A;A;A;A;A;A;D;B;B;E;L;L;L;E;F;E;B;L;E;F;L;L;F;A;B;I;F;B;H;I;C;G;D;L;L;C;C;E;B;B;J;B;D;D;G;F;A;C;L;L;L;L;K;C;D;B;E;F;F;B;F;F;L;F;F;E;B;B;E;D;D;D;E;B;D;D;L;H;A;I;I;B;B;I;J;E;E;E;E;L;E;E;E;E;E;E;B;B;H;E;E;E;I;I;I;C;C;B;B;I;F;I;I;I;C;C;C;L;H;C;C;G;J;L;F;I;L;E;L;F;I;K;K;K;K;K;K;F;C;C;C;B;B;B;B;B;B;B;I;I;I;I;I;F;H;E;I;L;L;E;B;B;B;L;L;H;B;L;B;E;C;F;H;H;D;D;E;A;L;E;E;F;B;B;B;B;C;B;G;D;D;A;L;A;I;I;F;J;I;H;F;D;C;C;C;D;D;D;D;D;D;D;B;L;L;E;E;F;H;F;I;A;L;L;L;L;L;L;L;L;L;L;F;F;B;L;L;L;L;D;B;H;B;E;B;D;J;I;C;C;G;C;L;E;E;C;A;L;I;C;C;L;A;A;A;A;A;L;A;I;I;B;D;B;I;A;A;B;A;A;A;G;B;B;F;F;J;C;C;L;L;D;B;I;C;G;E;E;E;E;J;J;C;E;B;I;I;B;B;B;B;B;I;A;I;B;E;E;I;A;L;H;B;B;B;H;D;B;B;F;I;I;B;L;E;K;L;C;E;E;E;H;I;I;B;I;I;F;F;D;D;G;F;F;F;E;E;A;D;C;E;E;E;J;E;E;J;E;E;C;B;B;L;E;J;J;H;H;G;D;D;D;D;D;L;L;E;A;B;H;H;H;A;A;A;A;D;K;K;G;G;G;C;I;B;L;L;H;H;C;C;G;D;D;L;B;B;B;I;I;G;F;G;C;C;B;I;I;I;I;D;H;I;B;F;I;C;C;B;E;F;B;D;B;D;L;E;B;L;L;L;L;A;E;C;C;E;E;B;B;B;L;H;H;H;H;I;D;H;G;G;B;E;F;G;M;A;A;L;L;L;L;I;H;H;A;B;B;E;A;G;B;B;B;B;D;D;D;A;A;A;D;D;A;A;A;A;G;B;B;B;B;B;B;B;B;B;B;B;L;I;I;L;A;E;L;L;L;L;L;L;A;H;H;H;H;H;B;C;C;F;B;L;C;I;C;B;B;B;F;E;B;I;E;I;I;I;I;G;B;F;F;F;C;L;L;L;L;L;I;A;A;G;G;C;C;G;G;F;F;A;B;B;B;H;E;G;G;C;C;K;K;A;K;B;I;E;G;C;L;L;I;F;F;F;J;E;F;L;L;H;B;G;D;H;H;H;H;H;L;C;C;L;B;B;B;B;B;B;I;I;F;L;L;L;L;L;H;I;E;D;C;L;A;L;G;B;D;B;I;B;B;E;B;B;G;I;D;H;E;I;E;E;E;E;E;B;I;I;G;B;F;L;I;I;F;F;E;G;C;B;H;B;L;A;G;C;C;L;L;L;L;I;I;A;L;L;A;A;A;A;A;B;L;I;H;F;F;F;F;F;F;F;F;B;E;A;G;G;B;G;I;I;I;E;E;E;E;E;A;C;C;C;C;B;H;H;C;C;D;D;H;H;C;L;L;F;I;L;E;D;D;D;L;L;L;A;B;D;B;L;B;A;H;L;L;E;B;A;A;M;G;G;G;C;C;C;C;C;G;C;G;B;D;D;E;E;B;I;B;D;E;H;C;C;C;C;B;B;I;B;B;K;K;K;K;K;K;C;E;B;L;L;L;L;L;L;A;E;H;C;E;E;E;H;B;L;L;H;H;B;B;E;D;E;B;C;L;L;G;I;A;A;E;F;B;B;B;E;I;C;D;L;I;I;I;I;L;A;E;B;D;D;E;E;E;E;E;B;C;C;C;C;G;D;D;D;D;D;D;D;D;D;D;K;B;I;H;B;I;B;I;C;G;G;G;E;H;A;A;A;A;B;I;D;C;C;C;J;B;L;L;E;E;E;E;D;L;E;F;H;K;B;F;B;I;I;L;L;G;C;C;C;C;C;B;B;E;E;E;E;E;A;A;C;D;B;B;B;B;B;B;I;I;B;I;B;D;B;C;A;E;E;C;D;D;L;L;C;D;F;D;E;B;L;B;E;E;E;I;J;H;H;I;A;F;F;C;C;B;B;I;F;C;C;C;C;G;D;D;D;D;D;D;D;D;D;D;G;C;A;B;B;L;L;E;E;E;H;K;L;I;I;C;D;D;C;D;F;I;A;L;B;A;E;I;B;G;D;L;F;B;G;G;E;L;G;B;F;C;C;C;C;C;G;C;G;C;D;B;B;L;I;I;B;E;B;L;L;B;B;B;B;G;C;C;E;E;B;B;B;B;F;F;B;I;B;J;B;C;I;I;F;B;L;H;C;C;C;H;C;D;D;D;D;B;E;E;E;A;G;E;E;J;C;C;D;D;D;A;D;L;L;I;C;C;C;C;L;L;E;E;E;B;J;F;A;A;A;D;L;E;D;K;B;I;B;F;D;D;D;L;E;E;A;F;C;C;C;B;I;E;B;I;C;C;D;I;D;D;D;D;D;E;E;L;E;E;H;H;H;C;D;D;L;B;B;I;I;J;J;L;I;G;G;E;E;I;F;A;B;C;G;C;E;G;G;B;G;L;L;L;L;L;K;B;J;C;B;F;C;I;I;B;B;L;G;B;B;B;B;L;L;L;H;H;E;G;B;J;L;A;E;E;E;E;E;F;B;B;I;E;E;C;L;L;L;I;I;I;I;F;A;A;A;C;D;L;D;D;G;B;K;A;I;F;E;E;E;E;G;A;A;E;I;D;A;G;B;B;G;E;A;J;C;H;H;H;K;C;B;J;F;F;F;B;I;C;F;A;A;G;G;C;C;G;G;F;F;K;I;L;L;E;E;E;E;F;B;H;H;E;E;E;D;L;L;E;B;D;B;L;H;H;H;H;A;B;L;G;E;E;E;L;E;F;F;F;F;G;C;E;F;I;B;F;F;F;A;E;E;H;I;I;I;A;E;B;C;B;E;D;L;K;G;G;H;B;L;L;I;H;L;B;B;L;A;B;B;H;B;B;E;E;D;L;E;L;L;M;B;B;B;G;G;C;C;G;G;B;A;E;H;C;L;L;L;L;D;E;L;I;F;L;M;G;G;G;A;A;B;B;B;A;I;G;G;C;C;B;B;C;C;F;L;L;L;L;L;L;L;F;F;F;F;L;L;B;B;A;B;B;L;L;I;D;H;E;I;I;I;I;B;C;J;B;E;E;L;C;C;D;B;L;L;G;B;L;L;I;I;C'.split(';')))

18549

In [175]:
mapper_df_x=pd.read_json('mapper_df_pid.json')

In [177]:
# mapper_df_x

In [184]:
mapper_df

,index,stay_id,org_name,storetime,AST_PATTERN
0,0,30000153,ENTEROBACTER CLOACAE,2174-10-07 10:31:00,"['ENTEROBACTER CLOACAE', 'TRIMETHOPRIM/SULFA', 'S']['ENTEROBACTER CLOACAE', 'GENTAMICIN', 'S']['ENTEROBACTER CLOACAE', 'TOBRAMYCIN', 'S']['ENTEROBACTER CLOACAE', 'CEFTAZIDIME', 'S']['ENTEROBACTER CLOACAE', 'CEFTRIAXONE', 'S']['ENTEROBACTER CLOACAE', 'CIPROFLOXACIN', 'S']['ENTEROBACTER CLOACAE', 'PIPERACILLIN', 'S']['ENTEROBACTER CLOACAE', 'CEFEPIME', 'S']['ENTEROBACTER CLOACAE', 'MEROPENEM', 'S']"
1,1,30000153,KLEBSIELLA PNEUMONIAE,2174-10-07 10:31:00,"['KLEBSIELLA PNEUMONIAE', 'CEFAZOLIN', 'S']['KLEBSIELLA PNEUMONIAE', 'TRIMETHOPRIM/SULFA', 'S']['KLEBSIELLA PNEUMONIAE', 'GENTAMICIN', 'S']['KLEBSIELLA PNEUMONIAE', 'TOBRAMYCIN', 'S']['KLEBSIELLA PNEUMONIAE', 'CEFTAZIDIME', 'S']['KLEBSIELLA PNEUMONIAE', 'CEFTRIAXONE', 'S']['KLEBSIELLA PNEUMONIAE', 'CIPROFLOXACIN', 'S']['KLEBSIELLA PNEUMONIAE', 'AMPICILLIN/SULBACTAM', 'S']['KLEBSIELLA PNEUMONIAE', 'CEFUROXIME', 'S']['KLEBSIELLA PNEUMONIAE', 'PIPERACILLIN/TAZO', 'S']['KLEBSIELLA PNEUMONIAE', 'CEFEPIME', 'S']['KLEBSIELLA PNEUMONIAE', 'MEROPENEM', 'S']"
2,2,30000484,ENTEROCOCCUS SP.,2136-01-21 11:25:00,"['ENTEROCOCCUS SP.', 'PENICILLIN G', 'R']['ENTEROCOCCUS SP.', 'AMPICILLIN', 'R']['ENTEROCOCCUS SP.', 'VANCOMYCIN', 'R']['ENTEROCOCCUS SP.', 'LINEZOLID', 'S']"
3,3,30003598,KLEBSIELLA PNEUMONIAE,2189-03-25 10:53:00,"['KLEBSIELLA PNEUMONIAE', 'CEFAZOLIN', 'R']['KLEBSIELLA PNEUMONIAE', 'TRIMETHOPRIM/SULFA', 'R']['KLEBSIELLA PNEUMONIAE', 'NITROFURANTOIN', 'R']['KLEBSIELLA PNEUMONIAE', 'GENTAMICIN', 'S']['KLEBSIELLA PNEUMONIAE', 'TOBRAMYCIN', 'S']['KLEBSIELLA PNEUMONIAE', 'CEFTAZIDIME', 'R']['KLEBSIELLA PNEUMONIAE', 'CEFTRIAXONE', 'R']['KLEBSIELLA PNEUMONIAE', 'CIPROFLOXACIN', 'R']['KLEBSIELLA PNEUMONIAE', 'AMPICILLIN/SULBACTAM', 'R']['KLEBSIELLA PNEUMONIAE', 'CEFUROXIME', 'R']['KLEBSIELLA PNEUMONIAE', 'PIPERACILLIN/TAZO', 'R']['KLEBSIELLA PNEUMONIAE', 'CEFEPIME', 'R']['KLEBSIELLA PNEUMONIAE', 'MEROPENEM', 'S']"
4,4,30003598,KLEBSIELLA PNEUMONIAE,2189-03-27 10:35:00,"['KLEBSIELLA PNEUMONIAE', 'CEFAZOLIN', 'R']['KLEBSIELLA PNEUMONIAE', 'TRIMETHOPRIM/SULFA', 'R']['KLEBSIELLA PNEUMONIAE', 'NITROFURANTOIN', 'R']['KLEBSIELLA PNEUMONIAE', 'GENTAMICIN', 'S']['KLEBSIELLA PNEUMONIAE', 'TOBRAMYCIN', 'S']['KLEBSIELLA PNEUMONIAE', 'CEFTAZIDIME', 'R']['KLEBSIELLA PNEUMONIAE', 'CEFTRIAXONE', 'R']['KLEBSIELLA PNEUMONIAE', 'CIPROFLOXACIN', 'R']['KLEBSIELLA PNEUMONIAE', 'AMPICILLIN/SULBACTAM', 'R']['KLEBSIELLA PNEUMONIAE', 'CEFUROXIME', 'R']['KLEBSIELLA PNEUMONIAE', 'PIPERACILLIN/TAZO', 'R']['KLEBSIELLA PNEUMONIAE', 'CEFEPIME', 'R']['KLEBSIELLA PNEUMONIAE', 'MEROPENEM', 'S']"
...,...,...,...,...,...
18544,18544,39999168,PSEUDOMONAS AERUGINOSA,2190-03-13 10:23:00,"['PSEUDOMONAS AERUGINOSA', 'GENTAMICIN', 'S']['PSEUDOMONAS AERUGINOSA', 'TOBRAMYCIN', 'S']['PSEUDOMONAS AERUGINOSA', 'CEFTAZIDIME', 'S']['PSEUDOMONAS AERUGINOSA', 'CIPROFLOXACIN', 'R']['PSEUDOMONAS AERUGINOSA', 'PIPERACILLIN/TAZO', 'S']['PSEUDOMONAS AERUGINOSA', 'CEFEPIME', 'S']['PSEUDOMONAS AERUGINOSA', 'MEROPENEM', 'S']"
18545,18545,39999168,PSEUDOMONAS AERUGINOSA,2190-04-08 12:45:00,"['PSEUDOMONAS AERUGINOSA', 'GENTAMICIN', 'S']['PSEUDOMONAS AERUGINOSA', 'TOBRAMYCIN', 'S']['PSEUDOMONAS AERUGINOSA', 'CEFTAZIDIME', 'S']['PSEUDOMONAS AERUGINOSA', 'CIPROFLOXACIN', 'R']['PSEUDOMONAS AERUGINOSA', 'PIPERACILLIN/TAZO', 'S']['PSEUDOMONAS AERUGINOSA', 'CEFEPIME', 'S']['PSEUDOMONAS AERUGINOSA', 'MEROPENEM', 'S']"
18546,18546,39999168,STAPH AUREUS COAG +,2190-03-13 10:23:00,"['STAPH AUREUS COAG +', 'ERYTHROMYCIN', 'R']['STAPH AUREUS COAG +', 'CLINDAMYCIN', 'R']['STAPH AUREUS COAG +', 'TRIMETHOPRIM/SULFA', 'S']['STAPH AUREUS COAG +', 'TETRACYCLINE', 'S']['STAPH AUREUS COAG +', 'GENTAMICIN', 'S']['STAPH AUREUS COAG +', 'OXACILLIN', 'S']['STAPH AUREUS COAG +', 'LEVOFLOXACIN', 'S']"
18547,18547,39999168,STAPH AUREUS COAG +,2190-04-09 10:45:00,"['STAPH AUREUS COAG +', 'ERYTHROMYCIN', 'R']['STAPH AUREUS COAG +', 'CLINDAMYCIN', 'R']['STAPH AUREUS COAG +', 'TRIMET

In [185]:
mapper_df['mapped_letter']=(pd.Series('A;B;C;D;D;D;B;E;E;A;C;C;B;F;B;F;D;D;B;F;G;B;H;B;B;I;E;E;F;F;B;F;F;F;H;H;D;D;D;D;D;D;D;D;D;E;E;E;E;E;E;E;H;I;I;J;E;I;K;B;B;L;I;B;A;C;B;B;F;F;C;I;C;B;B;B;F;I;I;B;E;H;C;L;E;E;E;C;C;I;B;I;K;K;D;D;B;L;L;H;H;H;C;A;A;B;B;B;I;F;F;B;A;L;L;L;E;A;G;B;I;I;A;A;A;M;D;E;I;E;I;L;F;B;A;A;B;H;L;L;A;B;B;I;I;I;C;B;L;I;I;J;A;C;C;B;H;D;L;L;L;L;L;L;E;E;H;H;G;E;C;B;A;A;H;B;F;F;F;F;I;I;I;F;F;F;B;B;C;L;I;L;L;L;B;I;I;I;F;H;H;C;C;C;C;B;B;F;I;B;B;E;A;F;B;H;B;E;A;C;C;B;I;I;K;B;E;E;B;F;F;C;J;J;B;L;B;D;L;H;G;I;C;B;B;E;F;C;C;L;L;L;C;C;C;F;F;E;E;B;G;F;F;C;B;L;C;C;B;F;A;B;L;F;F;F;H;F;F;B;B;F;F;A;C;C;C;D;D;D;D;D;D;D;B;L;L;E;E;F;H;E;B;B;A;I;K;K;K;K;K;K;C;E;B;L;L;L;L;L;L;A;E;H;B;B;A;A;I;J;E;G;J;C;E;D;B;A;I;E;B;L;G;I;C;C;C;C;B;B;M;H;H;H;H;B;B;B;I;I;B;F;I;A;B;L;I;I;L;E;I;E;E;L;G;G;A;G;D;C;F;F;D;A;I;F;B;B;L;L;L;E;C;B;B;B;H;H;H;B;B;B;C;C;D;D;D;D;D;M;D;D;D;D;F;F;H;F;E;B;B;F;K;K;K;K;K;K;C;E;B;L;L;L;L;L;L;A;E;H;G;I;A;A;A;B;E;D;C;I;C;E;E;E;E;E;B;E;E;E;E;A;B;B;B;I;I;F;F;F;I;D;D;H;I;J;I;I;I;F;A;A;B;H;B;C;D;B;B;B;B;B;G;C;C;C;B;E;E;E;J;L;B;I;I;I;J;C;E;K;K;K;B;E;G;A;I;B;H;C;G;B;H;G;F;H;L;L;L;L;L;L;L;F;F;C;C;D;L;I;I;B;E;E;L;L;E;E;E;E;E;B;L;I;K;F;C;I;I;I;I;I;I;I;I;I;I;H;E;L;L;C;C;B;C;L;G;D;L;L;I;B;G;C;J;H;H;H;B;G;G;F;F;F;A;A;G;G;B;B;L;E;C;C;C;C;B;L;D;B;J;J;B;C;B;L;L;L;B;B;J;B;C;C;C;C;I;I;I;A;C;D;B;B;B;B;B;B;I;I;F;G;D;L;B;B;L;L;E;A;E;L;B;I;I;H;H;E;E;E;E;E;E;F;L;H;A;L;E;E;E;E;H;C;E;G;A;C;I;C;B;I;F;F;A;B;B;F;B;A;E;G;C;D;E;I;K;I;I;D;F;L;F;L;B;I;L;L;L;L;L;L;L;L;L;L;L;L;L;L;L;L;L;L;K;C;B;B;L;E;E;E;E;E;L;A;A;B;G;L;I;I;I;I;A;A;A;A;A;D;F;K;D;L;C;L;E;E;H;H;H;H;A;B;M;L;L;H;H;G;I;I;C;L;L;L;E;E;D;D;C;L;B;L;E;J;B;B;B;B;B;L;F;E;F;H;I;I;F;E;E;E;E;E;C;E;A;A;C;I;F;J;J;A;C;C;C;D;D;D;D;D;D;D;B;L;L;E;E;F;H;L;L;L;L;L;B;L;I;F;D;F;I;I;C;C;D;D;F;F;H;B;J;G;B;C;C;B;B;D;I;C;F;E;I;B;H;H;H;H;L;L;L;B;D;I;E;H;D;C;K;C;D;L;L;L;E;G;F;I;C;D;B;B;H;H;H;I;L;L;A;A;L;D;L;C;G;L;E;E;I;F;D;F;I;A;L;H;A;E;I;A;C;G;J;L;G;B;E;G;B;I;F;B;F;I;I;B;B;B;B;B;E;E;E;E;E;A;I;D;B;B;B;B;I;I;I;I;J;C;B;L;B;B;L;D;D;D;F;B;H;B;D;I;L;B;B;E;B;B;B;B;I;G;C;B;B;B;H;B;B;L;I;I;I;I;D;I;C;J;J;A;B;G;C;I;I;F;L;G;F;B;B;L;B;E;I;J;B;C;B;E;B;E;C;B;L;E;B;K;C;B;L;B;B;F;A;A;I;I;B;C;F;C;B;L;A;C;B;G;I;H;E;E;D;E;G;C;E;L;C;E;E;E;I;D;A;B;I;I;I;I;F;B;A;D;D;I;I;F;D;L;G;C;L;H;B;E;A;G;B;E;I;E;E;A;B;C;C;C;F;C;C;E;E;E;F;I;H;H;B;I;L;E;B;C;L;G;B;I;F;B;B;B;D;B;L;L;L;B;I;I;K;B;B;I;C;G;A;D;B;I;L;L;L;L;L;L;A;H;H;H;H;H;K;K;K;K;E;L;H;J;K;K;D;D;F;B;B;L;H;F;F;I;I;I;G;F;F;F;F;F;E;E;F;B;B;B;C;E;L;B;B;B;A;M;M;K;L;E;A;A;A;C;C;E;I;I;L;D;B;E;F;F;F;G;G;B;G;L;B;B;G;I;L;C;H;F;L;L;E;E;E;L;L;L;L;L;L;L;E;E;B;F;B;G;L;B;A;F;F;B;B;I;B;D;B;F;F;I;D;B;I;A;I;J;D;F;C;D;L;L;L;L;L;I;I;I;I;H;B;F;F;B;L;L;L;L;G;E;A;B;A;E;B;B;H;H;B;E;E;C;G;E;F;E;A;C;C;C;B;E;D;B;E;C;C;C;C;D;K;K;L;L;L;E;E;E;B;A;I;H;H;H;C;B;I;C;B;B;A;C;L;A;B;B;L;L;E;E;E;H;B;L;E;C;D;E;E;L;I;D;L;L;I;B;F;H;A;D;B;K;A;F;B;B;L;A;L;L;B;E;D;B;I;M;I;I;I;I;F;M;C;E;E;E;E;H;H;H;A;A;B;D;I;I;L;L;I;I;H;D;D;D;E;M;L;E;G;D;D;D;I;E;J;L;K;K;I;A;L;E;F;F;H;L;A;B;C;C;B;B;M;H;H;H;H;K;K;K;K;K;K;F;C;C;C;B;B;B;B;B;B;B;I;I;I;I;I;F;H;B;L;G;L;L;L;L;L;L;B;I;I;I;F;I;B;B;L;L;L;C;F;F;F;E;G;G;K;C;C;L;L;M;M;B;I;I;F;D;L;C;B;I;A;C;C;I;H;E;J;I;B;F;G;B;B;D;L;L;L;I;F;F;B;F;B;D;D;B;B;B;E;I;B;B;B;G;G;D;L;L;L;H;I;C;G;C;B;B;E;L;G;B;C;C;B;L;L;E;E;F;F;B;E;E;B;B;D;D;L;C;C;C;B;A;A;A;A;B;B;A;C;B;B;L;L;L;H;H;H;H;I;G;L;L;L;E;E;B;B;J;E;F;I;I;I;I;E;E;E;E;E;L;I;M;L;E;D;L;L;E;B;G;I;H;H;H;H;G;L;I;G;B;L;L;G;I;I;C;C;C;C;C;C;D;D;D;D;D;M;D;D;G;A;B;G;K;K;K;K;G;G;L;L;L;L;L;L;L;F;L;L;H;L;H;H;C;C;C;C;B;B;F;B;B;B;A;G;C;K;L;I;I;C;C;C;C;B;I;F;L;C;L;L;L;L;L;L;L;F;F;C;B;B;C;D;D;D;D;H;B;E;E;C;C;E;B;D;B;L;I;L;L;E;E;E;E;G;L;B;B;B;B;E;E;G;G;C;D;L;L;L;A;A;H;L;B;B;B;L;E;H;H;L;I;I;I;I;I;L;L;A;L;L;E;E;I;L;F;F;E;E;E;I;I;I;C;C;G;I;I;C;M;A;B;B;C;D;D;B;B;B;B;I;E;E;E;E;G;G;D;B;I;E;F;F;F;F;B;B;F;F;D;E;E;L;I;H;B;I;C;D;D;D;B;I;G;I;F;F;F;F;I;L;E;F;F;J;B;B;B;F;F;G;G;G;A;A;C;C;G;I;I;I;I;B;D;B;B;B;I;H;B;B;J;C;B;B;B;F;D;D;F;G;H;H;I;J;D;D;I;F;G;L;A;A;A;D;I;I;G;B;C;C;C;C;B;H;H;I;I;L;E;B;A;C;L;L;L;L;I;I;C;I;H;E;J;E;E;L;L;B;E;E;E;D;L;H;G;D;B;B;B;B;E;E;E;H;L;L;F;B;G;F;F;L;L;E;I;I;J;G;B;I;E;E;E;E;F;J;B;I;A;A;I;I;B;B;I;A;C;L;B;I;L;A;H;C;C;B;B;B;B;B;B;L;L;L;L;L;L;I;I;G;B;B;I;F;I;B;L;D;E;C;B;B;B;D;B;A;A;E;E;I;B;F;I;L;G;E;E;E;G;B;B;B;B;H;H;H;F;H;H;H;H;F;B;L;L;L;L;L;L;A;H;H;H;H;H;B;C;E;H;B;L;L;I;B;E;G;D;D;A;A;A;H;E;D;B;A;A;L;I;J;H;I;L;B;L;L;L;I;F;F;G;K;C;C;B;E;E;E;E;E;E;H;I;C;B;B;E;C;B;B;L;A;I;B;B;B;L;E;E;E;H;E;D;L;L;L;L;L;L;L;L;L;B;I;H;H;H;H;H;I;B;B;B;B;A;A;I;A;B;B;I;I;C;E;F;D;B;E;G;L;L;L;L;L;C;H;L;L;H;H;G;E;A;G;B;B;F;E;D;I;C;D;G;B;I;I;C;B;L;C;B;E;B;A;A;I;I;K;A;H;L;D;C;F;F;C;F;F;H;B;A;A;B;I;A;E;L;A;A;I;F;F;D;L;B;E;E;E;B;B;D;L;L;L;L;I;B;E;E;E;B;H;H;H;C;C;A;L;L;L;H;H;H;E;F;E;B;E;E;E;E;D;L;L;G;L;L;H;B;B;C;F;I;I;H;H;A;B;D;C;B;E;E;F;B;L;L;L;E;C;B;D;D;E;E;E;E;G;C;L;L;I;I;I;D;E;J;L;I;G;B;B;E;B;B;L;F;D;F;B;F;F;B;E;E;E;B;I;G;B;D;B;B;I;B;I;E;L;L;L;L;L;L;I;E;L;B;I;C;B;L;E;L;L;I;I;I;C;C;I;I;I;F;C;C;L;L;E;H;C;C;B;B;B;B;B;B;L;L;L;L;L;L;I;I;E;D;M;M;L;E;B;B;B;B;B;E;E;B;A;E;D;H;H;H;H;H;H;H;H;H;H;H;H;H;H;G;C;C;C;C;D;D;D;D;D;D;D;D;D;D;L;L;F;B;J;D;G;F;F;D;L;B;B;B;C;B;E;E;F;B;F;F;J;B;A;A;A;D;D;D;L;A;A;K;L;L;C;F;G;B;L;F;J;A;A;F;C;B;G;L;L;E;E;E;F;D;I;K;K;A;B;B;B;I;I;C;E;E;E;E;L;D;A;C;C;D;E;C;D;E;E;D;D;B;D;C;C;L;L;L;L;L;L;F;E;D;D;B;L;F;C;B;B;E;F;D;B;B;L;A;E;E;C;C;C;C;E;E;E;C;E;B;B;G;B;F;B;H;F;F;I;L;F;E;C;C;C;G;I;G;C;L;B;G;C;I;L;C;B;B;B;A;A;H;H;B;B;L;K;C;C;I;H;H;C;I;D;A;B;C;L;L;L;L;K;D;E;E;E;E;E;E;E;E;E;E;E;H;I;I;I;F;A;A;F;B;B;I;L;L;F;F;G;G;G;C;D;D;B;H;F;B;F;I;B;C;C;B;B;H;K;L;J;L;L;E;E;E;E;E;F;L;C;L;L;L;L;L;L;L;F;B;C;E;E;G;D;B;F;F;B;K;K;B;H;H;K;F;C;I;I;I;I;I;I;I;I;I;I;K;G;G;H;A;A;A;E;H;H;B;G;H;E;L;E;E;E;E;E;G;F;F;I;I;I;F;B;B;B;B;F;J;F;F;A;A;A;B;B;D;C;I;F;G;C;C;A;A;G;G;C;C;G;G;F;F;A;G;I;I;C;B;G;B;G;L;B;J;A;I;I;C;C;B;I;L;L;L;E;E;E;E;B;F;L;E;B;E;B;L;E;B;B;I;K;D;D;A;G;G;B;B;B;G;B;L;L;A;I;C;G;B;B;I;F;H;C;L;L;L;F;F;I;B;L;C;C;L;L;L;I;I;I;D;D;D;L;L;L;L;F;B;B;L;L;E;C;B;B;K;K;C;D;D;D;D;M;L;L;L;L;L;L;L;D;B;E;I;E;E;J;B;L;L;L;L;F;F;F;F;F;H;D;D;D;D;D;E;E;L;E;E;E;E;G;D;B;I;B;B;F;L;A;C;B;I;C;L;L;L;L;L;L;E;E;F;F;F;B;I;C;C;C;E;E;C;D;C;F;G;G;F;B;K;G;C;A;F;B;B;B;L;L;F;F;F;B;L;H;B;E;E;C;B;B;I;D;K;B;E;B;C;B;B;H;F;E;E;B;A;D;C;C;C;G;G;B;B;I;I;B;I;J;J;F;L;E;C;E;L;H;G;F;F;G;D;I;D;L;L;H;E;E;E;B;G;I;M;C;C;G;I;I;E;B;I;A;B;B;B;L;E;D;B;C;C;C;C;D;E;B;B;C;B;D;B;B;D;F;G;B;I;B;E;E;F;D;D;D;B;A;B;B;A;A;A;L;F;I;K;K;C;B;L;L;G;G;F;F;F;B;I;F;A;E;B;A;G;E;E;I;L;D;A;A;A;A;A;A;A;K;K;D;E;B;A;B;D;I;I;I;I;L;E;L;L;L;L;A;B;E;F;B;B;L;L;E;B;E;E;E;E;B;M;G;L;B;B;G;I;I;E;L;L;A;A;G;G;I;E;E;E;B;L;B;L;B;F;F;F;G;C;B;F;H;I;K;K;K;I;I;L;I;B;B;G;F;B;I;A;I;I;I;I;I;B;E;K;K;E;D;B;B;B;F;F;E;E;C;L;E;E;E;E;E;A;E;E;E;I;D;B;B;C;L;C;C;B;B;L;I;E;I;A;B;B;C;D;L;L;L;A;J;G;B;B;B;L;L;I;B;B;H;I;F;F;K;K;D;D;L;L;L;H;I;I;J;E;A;G;B;B;F;C;B;L;L;L;F;F;B;B;G;D;D;C;B;L;L;L;L;L;L;E;E;A;F;L;I;C;E;E;E;E;B;B;B;B;E;E;H;C;D;J;L;L;B;B;L;L;H;J;B;B;B;F;F;D;D;B;A;C;C;I;H;D;L;L;C;L;L;B;H;E;F;F;E;H;H;E;K;K;A;B;L;L;G;A;E;F;I;I;I;C;C;C;B;B;D;K;E;C;B;I;I;A;C;C;C;F;A;B;B;F;A;L;K;E;I;G;I;I;I;I;C;E;G;B;B;I;F;A;G;G;G;I;E;E;E;E;F;F;C;B;L;C;B;J;L;I;I;L;I;B;I;F;A;A;A;A;A;G;D;B;F;F;F;F;F;C;C;A;I;C;D;D;K;A;L;A;E;A;A;D;L;B;E;E;D;L;H;B;L;L;A;J;B;I;E;E;F;F;B;L;L;L;L;G;C;I;F;I;L;B;L;L;L;I;C;I;B;B;L;H;C;C;D;E;D;C;C;B;G;E;H;H;A;L;K;L;L;L;L;L;A;A;A;A;D;I;G;L;A;B;D;B;A;E;E;C;B;G;E;E;J;K;H;E;G;G;C;B;D;E;E;E;K;B;I;D;B;A;L;F;D;G;C;L;F;F;E;I;E;B;E;I;I;H;E;H;B;B;B;B;B;B;I;I;F;I;E;B;D;E;I;L;F;L;H;L;L;L;L;H;B;G;J;B;E;I;E;C;B;D;B;B;B;A;B;I;B;D;D;B;E;E;E;E;D;D;C;C;D;B;C;L;L;B;B;B;B;D;L;L;L;L;G;A;A;L;G;I;H;I;H;I;I;E;B;I;G;H;E;E;A;B;B;B;I;I;I;E;E;F;A;H;I;A;A;A;A;A;A;C;B;D;B;I;F;A;A;D;D;C;L;L;L;L;D;C;F;F;B;B;B;B;B;B;B;K;B;D;D;B;I;I;I;I;H;H;H;H;B;F;F;F;B;L;L;L;L;L;C;B;L;I;J;D;B;L;B;L;L;L;L;I;H;H;B;F;G;B;C;E;E;E;B;E;E;I;L;B;F;D;F;B;B;B;L;L;L;E;H;C;C;B;B;B;B;B;B;L;L;L;L;L;L;I;I;G;D;C;F;I;I;L;I;B;I;F;F;F;F;G;C;C;B;H;C;C;F;B;B;L;I;B;F;B;B;I;F;I;G;C;C;C;B;E;L;D;B;B;L;L;L;H;E;E;A;A;A;A;A;G;G;E;E;E;B;F;C;B;B;B;E;I;F;B;L;B;D;A;A;A;C;B;B;B;I;H;F;I;C;C;L;L;L;L;L;H;B;A;L;B;I;I;L;C;A;A;C;C;C;G;G;B;B;I;D;F;C;B;I;I;I;I;B;D;D;H;H;L;L;I;I;L;K;K;K;A;A;L;L;E;E;F;F;F;F;F;B;A;A;E;L;I;I;F;I;I;I;I;C;G;L;L;L;L;L;L;B;J;B;B;B;I;H;E;E;I;L;H;A;A;A;A;G;B;B;B;B;C;B;A;G;B;C;B;B;B;A;A;A;M;B;K;G;B;L;F;I;E;L;L;F;L;I;I;E;E;E;E;E;E;B;B;I;F;F;L;L;A;D;E;E;F;B;L;L;I;B;B;B;L;F;L;I;D;B;B;B;B;I;F;C;L;L;E;L;L;J;A;A;A;A;C;M;G;F;F;K;I;H;E;E;D;D;E;B;I;B;B;D;L;A;D;H;B;G;B;B;L;G;B;B;E;I;G;D;B;E;E;E;I;B;L;L;A;A;I;B;I;I;G;C;C;C;C;C;B;B;E;E;E;E;E;I;A;A;E;L;L;L;L;F;F;I;E;E;B;A;H;L;C;I;F;F;B;C;C;K;L;B;B;L;F;G;B;E;B;G;F;F;A;C;C;C;C;C;C;C;C;C;F;I;I;L;E;L;L;F;F;F;F;E;G;B;I;G;B;B;I;F;I;I;E;B;D;A;D;D;I;I;E;H;E;E;E;A;A;G;I;A;I;B;L;L;L;F;F;L;E;G;I;I;I;F;F;G;G;E;L;L;E;G;G;L;L;A;I;I;F;B;E;E;E;J;I;J;E;E;C;D;D;E;F;G;F;E;E;E;E;E;J;C;E;E;E;E;E;B;I;L;L;L;L;I;D;B;B;B;I;I;I;L;B;G;D;D;A;A;A;B;L;L;A;A;A;A;E;C;F;F;B;B;B;L;I;B;I;C;D;C;L;L;L;L;H;B;B;E;D;D;L;E;A;A;C;E;F;L;L;E;C;F;F;A;C;C;C;C;C;C;C;C;C;I;I;I;I;A;I;I;B;B;A;L;E;E;E;B;I;I;L;L;H;H;D;B;L;L;L;F;C;E;H;H;H;L;I;I;I;I;D;D;I;B;K;K;E;L;I;F;D;F;F;F;F;B;B;B;B;B;I;E;H;D;F;E;I;G;A;B;B;F;F;H;C;L;L;I;H;I;A;J;D;B;C;F;H;G;L;B;A;B;B;B;I;I;I;H;H;D;I;E;I;G;I;A;L;L;F;A;G;B;L;A;A;E;E;E;B;B;L;B;B;K;K;C;B;L;L;B;I;I;E;L;I;F;F;L;B;J;J;B;D;B;L;L;L;L;E;B;A;A;A;A;A;E;I;B;B;B;B;L;L;G;F;B;B;B;A;B;E;I;C;E;C;K;G;B;A;I;B;B;L;D;B;E;B;B;F;E;L;A;I;J;L;I;I;J;I;H;B;E;B;L;G;A;B;B;B;A;D;C;B;F;B;I;B;L;B;B;C;C;D;D;D;D;D;M;D;D;D;D;F;F;H;G;K;B;L;C;D;D;G;D;E;E;E;E;C;C;C;C;C;B;B;C;F;B;I;I;D;I;D;C;C;B;E;A;G;L;F;L;F;I;F;G;D;D;E;E;E;I;K;K;K;K;K;K;K;K;K;K;C;D;D;D;B;B;L;L;L;L;L;L;L;L;H;B;A;L;H;C;C;G;D;D;D;B;C;K;L;C;B;L;L;L;L;L;L;L;L;F;B;A;G;B;B;E;E;A;A;E;E;B;B;B;A;A;A;A;F;G;F;F;F;I;A;I;A;B;C;C;I;M;L;I;B;E;E;A;G;G;F;I;I;L;L;L;L;L;E;E;E;E;E;E;G;B;E;E;E;C;C;L;L;L;L;L;L;L;F;F;L;E;H;I;F;F;B;B;C;L;L;L;L;L;L;F;C;A;G;F;H;H;A;A;A;A;F;A;F;F;F;B;D;H;C;C;H;C;L;L;L;E;I;I;I;A;B;H;L;H;A;B;J;D;F;F;M;B;F;G;E;C;C;C;L;D;D;D;L;L;L;L;H;B;F;I;C;B;D;B;I;I;G;I;B;B;B;B;E;E;A;B;A;B;D;L;I;I;I;L;C;E;I;I;I;L;L;F;D;E;E;K;K;A;H;I;I;J;L;L;C;D;L;L;I;H;L;L;L;L;L;I;G;F;C;L;L;L;L;L;L;F;E;I;E;E;D;B;B;I;J;B;I;A;E;I;D;D;H;G;A;L;L;L;L;L;H;B;B;I;H;E;I;I;H;C;C;B;B;B;B;B;B;L;L;L;L;L;L;I;I;C;F;B;I;I;A;A;B;B;I;I;I;I;I;F;H;H;H;K;B;E;E;I;C;L;E;F;B;B;B;B;I;C;B;L;L;L;I;L;F;F;F;F;F;H;F;L;A;A;G;B;L;I;I;E;L;L;A;D;A;A;C;B;E;C;B;B;L;A;A;A;C;C;I;L;B;K;C;C;D;D;L;E;I;I;H;B;I;G;G;C;C;G;G;K;C;D;D;D;D;M;L;L;L;L;L;L;L;F;F;C;C;C;I;I;F;H;H;H;D;L;L;L;I;C;C;B;L;L;L;L;L;L;L;F;F;K;I;I;I;D;D;C;C;I;A;D;F;H;H;H;H;H;B;I;F;F;J;C;B;G;G;D;L;L;L;L;L;L;L;E;E;I;C;B;E;E;I;F;B;B;B;F;I;I;I;K;B;I;E;I;B;B;B;G;I;I;I;I;D;I;F;D;B;E;G;B;L;L;F;I;I;J;E;E;I;D;L;L;L;C;B;B;B;A;A;H;H;B;B;E;E;E;D;D;D;G;B;L;L;L;L;E;E;A;A;C;I;H;L;L;E;A;L;L;A;A;C;C;C;C;C;I;B;C;D;E;C;B;B;E;E;E;E;E;E;C;A;A;A;M;L;B;E;B;A;C;B;B;I;I;I;J;G;L;L;E;H;C;I;I;A;C;B;K;K;K;D;D;E;E;L;A;E;G;B;C;C;C;C;C;F;C;C;C;G;I;C;B;G;I;B;D;G;B;C;A;A;D;D;D;D;A;A;A;G;B;B;B;B;B;B;B;B;B;B;B;L;K;C;D;L;L;L;E;A;C;C;C;L;C;C;G;G;I;I;F;F;I;C;C;C;I;I;F;H;H;H;D;E;H;H;C;C;C;C;B;B;F;G;L;D;G;B;B;L;L;L;L;E;E;H;D;D;D;E;E;J;C;D;F;C;I;D;D;D;D;C;L;A;A;B;B;L;L;L;L;L;E;E;A;A;B;B;C;G;H;H;H;H;H;H;H;H;H;H;H;H;H;H;I;I;D;G;G;I;E;C;E;E;I;F;F;B;L;L;L;L;C;A;B;I;G;B;C;J;J;B;A;C;C;B;D;D;L;B;B;B;L;I;B;B;B;A;H;H;I;I;B;D;B;B;C;F;F;L;D;D;D;E;E;E;H;C;D;C;G;D;B;B;B;B;I;I;I;G;C;C;C;B;E;B;A;I;I;L;F;I;B;B;E;L;I;B;D;D;B;E;L;I;E;K;K;K;K;I;I;B;B;L;L;H;D;G;A;A;B;E;A;L;E;H;I;L;C;L;B;C;C;H;H;L;L;E;B;F;B;E;L;I;E;J;J;G;A;B;D;D;D;E;E;E;G;G;E;H;B;L;L;H;H;L;D;F;F;B;F;A;D;D;E;E;I;I;I;I;I;G;I;K;F;C;I;I;I;I;I;I;I;I;I;I;C;B;I;I;L;B;B;I;J;A;A;I;J;B;L;C;B;B;L;B;I;C;C;C;C;I;H;H;H;H;L;G;I;I;I;G;H;I;E;C;C;F;F;C;D;M;G;K;K;K;K;G;C;L;L;L;L;L;L;L;F;L;L;L;L;L;L;A;H;H;H;H;H;A;F;G;A;A;A;A;F;C;B;E;E;F;C;B;K;L;H;H;C;C;C;C;B;B;F;K;C;I;D;L;C;L;L;B;B;C;F;C;B;E;E;J;C;E;H;H;J;B;E;H;I;I;H;L;L;L;L;D;L;B;B;B;E;E;I;K;G;B;L;L;L;L;I;I;I;I;B;C;B;B;D;K;A;E;D;D;E;E;E;E;C;B;B;L;F;G;G;B;B;B;L;L;L;L;E;E;H;C;D;D;D;D;D;D;B;L;L;L;L;L;L;L;L;L;I;D;D;I;I;E;I;B;B;L;L;L;L;A;E;A;I;I;E;G;A;G;B;B;E;K;K;E;I;I;B;L;I;L;L;L;L;I;I;B;I;K;B;H;C;H;B;H;H;G;D;D;D;D;H;L;L;L;E;H;C;B;I;D;B;B;D;A;D;B;D;D;L;L;F;B;I;L;A;B;A;C;D;B;A;C;C;C;F;C;B;G;B;A;G;B;A;H;C;D;B;B;G;B;I;L;E;I;H;H;F;I;I;C;C;F;F;C;A;E;C;C;B;D;D;H;H;B;I;C;L;D;B;B;E;A;A;G;I;L;M;M;G;I;I;I;E;C;C;B;L;L;E;E;L;E;E;E;B;F;G;C;B;G;G;E;F;B;E;L;B;D;H;J;D;L;I;A;A;G;G;C;C;G;G;F;F;G;D;D;D;D;D;E;E;L;E;E;B;I;A;D;L;L;L;E;E;B;L;D;B;B;F;E;E;L;L;G;H;A;A;A;A;A;C;C;C;C;C;B;I;I;I;I;F;H;L;I;A;C;C;I;E;H;E;L;E;E;E;E;I;F;F;F;F;B;I;I;F;C;L;K;E;L;E;E;E;E;H;K;H;B;B;A;G;L;L;I;C;B;G;L;L;E;E;H;L;D;I;K;H;E;E;H;G;A;L;L;E;E;F;I;D;E;E;E;E;E;A;A;C;L;J;G;I;L;L;H;G;I;E;F;C;B;B;E;E;E;E;E;E;E;E;E;J;I;B;B;J;E;B;I;B;B;C;D;A;E;H;H;A;I;L;G;I;E;E;E;E;H;B;B;A;D;D;I;I;H;I;I;B;C;C;B;E;E;E;A;G;G;B;H;H;B;G;E;H;H;E;L;E;C;B;L;L;E;D;L;H;A;A;B;B;E;E;E;B;E;F;B;I;E;E;E;C;C;G;K;C;I;C;D;D;L;A;A;B;B;K;L;E;E;B;A;E;L;E;I;B;C;M;M;L;I;G;B;C;L;L;L;L;L;L;L;L;L;L;D;L;A;D;G;C;A;I;K;K;K;K;C;L;L;L;L;E;L;L;D;B;B;I;I;A;L;H;L;B;D;D;D;B;E;G;G;A;A;A;A;D;A;B;I;I;I;F;G;E;F;G;G;D;E;I;F;E;I;I;L;I;L;L;L;J;D;B;B;B;B;B;B;B;I;J;J;E;B;B;C;I;K;K;K;I;I;I;B;C;B;B;I;F;L;G;G;G;B;L;L;B;L;I;G;D;L;E;L;E;I;E;F;F;B;L;L;L;L;B;C;A;A;C;D;L;L;C;C;E;B;L;L;L;L;B;K;I;L;B;B;B;B;L;L;F;I;D;B;E;E;E;C;C;A;A;I;I;L;L;L;C;L;L;L;H;H;I;D;C;C;C;C;C;C;C;E;A;L;L;L;I;H;A;A;A;D;A;D;F;B;E;E;E;C;B;L;I;F;F;F;F;E;D;D;B;E;E;E;F;B;L;L;L;C;B;F;G;F;I;E;D;B;L;L;L;G;M;B;B;L;E;E;F;F;B;D;L;E;F;F;B;L;L;L;L;E;E;E;E;E;A;B;B;L;F;I;I;I;B;F;F;F;F;D;D;L;E;A;B;I;H;J;L;L;D;E;E;E;E;E;E;F;F;E;E;A;A;A;F;L;F;F;F;F;F;H;F;F;C;L;F;G;E;E;L;L;L;L;L;I;I;I;I;L;I;F;F;F;F;I;K;K;B;K;C;C;I;H;H;B;I;B;B;G;D;D;A;B;B;B;B;B;B;I;L;L;L;L;F;A;G;G;G;B;E;E;A;A;C;I;F;E;K;D;H;A;L;D;B;A;A;I;A;E;I;B;B;C;H;E;B;A;A;M;G;G;G;C;C;C;C;C;G;C;G;B;B;C;G;D;D;A;M;B;B;F;E;G;I;I;I;A;A;I;I;L;E;D;L;L;I;H;H;E;E;I;G;B;B;B;I;L;D;D;D;C;A;I;E;E;J;C;B;E;C;B;B;L;L;K;K;F;A;A;B;D;D;I;I;I;I;B;G;D;D;D;G;I;I;I;I;I;A;G;B;B;B;B;G;C;C;C;C;D;D;D;D;D;D;D;D;D;D;L;L;F;D;D;D;L;L;L;C;D;D;D;D;D;D;D;L;L;L;L;L;L;L;A;A;A;A;A;A;A;G;A;A;C;C;B;B;B;B;B;B;B;B;F;B;G;L;I;I;F;F;L;H;B;E;I;I;B;B;I;A;I;L;L;E;F;C;L;C;C;C;B;K;H;E;G;C;D;F;B;B;B;E;E;A;B;B;E;B;D;L;L;L;L;L;K;B;B;H;H;H;C;B;E;D;L;K;K;D;D;L;L;L;H;E;E;E;E;F;L;L;L;F;B;B;I;B;I;A;A;A;M;G;G;G;B;G;A;I;F;F;A;L;A;A;A;A;A;A;I;I;G;G;B;B;I;I;I;F;C;L;C;B;B;D;B;D;L;B;E;C;F;B;C;H;F;J;G;H;F;G;G;B;B;I;I;I;F;A;A;I;B;B;B;B;F;A;A;A;M;E;E;B;B;D;C;C;C;C;E;E;E;D;L;I;G;J;E;I;H;H;E;E;A;A;A;B;G;F;F;B;C;B;A;A;I;I;G;B;B;I;L;L;E;B;A;E;I;D;L;L;G;D;D;C;B;A;H;I;E;E;F;F;F;H;K;K;G;G;G;C;L;E;L;L;A;L;C;C;D;E;L;L;L;L;E;F;A;A;B;I;C;C;C;C;L;L;L;L;L;L;E;E;H;D;I;F;F;E;E;E;E;E;E;E;E;E;B;B;L;I;D;E;E;I;E;B;B;L;E;E;E;L;G;B;B;D;L;L;L;L;I;I;A;B;C;C;B;B;B;A;B;A;A;I;B;J;L;E;C;B;I;I;I;L;C;B;B;I;I;I;L;J;D;L;L;E;B;B;F;F;F;F;F;D;D;B;H;A;A;F;L;H;H;H;E;B;L;L;H;A;A;I;C;B;B;D;D;K;E;H;J;A;B;L;B;B;I;I;I;C;G;B;B;L;L;L;G;F;B;L;C;B;L;L;L;B;I;F;F;A;I;D;H;H;C;B;B;I;D;E;E;A;I;I;E;L;B;E;E;E;B;B;C;E;E;E;E;B;L;G;A;K;K;K;K;I;E;E;E;B;G;L;H;H;L;L;L;L;L;L;L;H;H;E;A;A;A;E;L;L;E;E;I;L;C;E;E;E;E;J;J;E;F;M;L;L;L;F;F;J;D;F;F;F;K;G;G;H;C;E;A;B;I;I;C;E;E;E;E;B;B;H;I;I;I;L;E;E;C;I;B;B;L;L;L;E;B;A;D;D;I;I;A;K;E;B;F;E;E;I;A;D;C;I;I;C;C;L;E;B;F;G;D;D;B;I;I;I;J;A;D;B;B;B;B;H;B;I;D;B;D;L;I;I;I;I;K;K;K;K;I;H;H;I;F;G;I;B;B;B;B;H;H;H;I;B;E;A;A;C;B;B;E;E;L;I;D;D;D;B;L;E;E;E;E;E;B;E;E;E;E;B;E;D;B;B;B;F;B;M;M;L;F;F;B;E;C;B;D;B;L;C;E;E;E;D;B;I;I;I;I;I;I;H;B;A;C;E;E;L;F;B;L;K;K;B;I;I;I;A;E;E;C;L;B;C;B;B;E;B;L;L;L;H;C;I;B;B;L;B;K;I;E;E;E;E;C;G;L;E;E;D;D;B;B;L;K;K;K;K;K;K;K;K;K;K;C;D;D;D;B;B;L;L;L;L;L;L;L;L;H;B;E;E;E;E;E;E;I;I;J;L;L;H;H;H;H;H;H;G;G;D;L;A;A;B;D;A;B;C;C;C;C;C;C;C;D;L;E;L;F;F;F;F;B;A;E;I;K;K;K;K;E;E;H;H;H;E;E;E;F;C;E;C;C;L;L;L;L;L;H;H;E;E;E;C;I;C;C;L;L;L;L;L;H;H;G;C;A;A;B;L;L;L;H;H;I;B;F;H;B;E;I;A;L;E;D;A;D;B;I;F;F;G;C;C;C;C;D;D;D;D;D;D;D;D;D;D;L;L;F;C;B;I;B;L;A;L;L;E;L;C;G;G;C;C;C;C;C;C;L;H;H;L;D;D;E;I;F;I;E;E;B;B;G;B;B;B;F;F;C;H;F;G;G;C;C;G;G;C;D;M;B;B;L;L;L;A;G;G;D;H;H;L;C;C;C;C;C;G;C;B;K;D;L;F;B;F;I;I;B;F;B;E;E;E;E;I;D;D;C;B;B;L;L;L;L;I;E;E;E;E;E;C;B;I;B;E;C;L;B;B;B;C;B;B;B;D;L;C;C;C;F;B;B;I;C;B;B;I;I;I;F;I;I;E;F;F;B;B;E;E;E;E;E;H;E;A;C;C;A;C;C;C;L;B;A;B;B;B;B;C;C;D;G;I;K;B;E;E;G;G;B;I;A;A;C;C;I;I;C;E;E;E;A;A;G;L;K;C;C;I;H;H;C;E;K;I;F;B;A;D;B;I;A;C;C;L;L;L;L;L;L;L;I;D;D;L;E;G;D;D;A;A;A;E;C;L;L;H;H;H;H;A;C;C;C;G;B;A;A;L;H;H;H;H;E;B;I;I;I;A;A;K;K;I;K;C;D;L;L;G;G;C;D;H;I;D;D;L;I;A;H;H;C;B;B;C;B;L;L;H;K;A;H;A;G;B;E;F;F;F;B;I;I;E;E;I;E;L;A;C;B;B;I;I;I;C;E;E;E;F;F;L;L;I;B;G;B;I;E;E;L;E;H;H;H;F;B;D;D;B;B;B;E;E;E;L;F;F;F;L;L;L;L;L;L;B;L;L;B;K;K;K;K;H;L;L;B;B;B;L;L;H;H;I;I;I;I;J;J;J;G;D;D;F;I;L;L;B;B;L;L;L;L;C;C;B;D;B;A;G;I;I;I;L;L;H;I;L;L;L;A;A;G;F;C;B;G;B;A;F;A;A;F;H;H;H;H;H;B;I;I;G;A;A;A;B;B;B;I;I;I;F;F;C;C;C;H;H;H;H;H;H;H;H;H;A;G;G;G;C;C;B;E;E;L;L;E;E;E;E;E;C;G;I;E;D;D;C;C;B;B;C;C;C;C;G;D;D;D;D;D;D;D;D;D;D;D;I;I;I;I;A;A;I;I;K;K;I;I;A;B;C;I;E;J;K;K;L;L;L;E;E;I;A;B;I;A;A;L;L;I;I;H;H;G;G;G;B;B;B;B;B;B;B;B;B;B;B;B;G;J;C;E;G;G;B;D;D;D;D;B;I;B;E;G;B;A;A;I;I;B;B;B;G;I;I;C;C;E;B;B;B;I;C;I;A;F;C;D;D;D;L;L;L;L;L;E;E;E;B;D;D;E;E;E;E;L;A;A;A;A;C;L;B;B;B;B;C;C;C;B;K;L;L;L;L;F;H;H;K;C;D;D;D;D;M;L;L;L;L;L;L;L;B;B;B;A;K;B;C;B;L;E;E;E;G;E;E;E;E;A;L;H;H;H;H;G;G;C;D;D;D;D;L;L;K;E;H;J;B;C;I;H;A;L;I;E;A;B;B;I;B;B;B;G;B;E;G;F;F;F;F;E;F;F;E;E;E;E;E;E;E;E;E;K;A;A;A;A;A;A;H;H;A;I;I;I;H;D;B;D;E;E;C;C;L;A;B;E;E;E;E;H;H;E;E;E;E;E;E;F;C;B;B;C;L;L;I;C;C;C;H;C;L;L;L;L;I;I;B;I;G;D;D;L;L;L;E;E;I;I;F;F;F;I;G;B;L;C;H;H;F;F;B;K;K;K;K;K;K;C;E;B;L;L;L;L;L;L;A;E;H;G;B;I;I;I;L;F;F;B;L;A;A;I;C;C;C;C;C;C;C;G;C;A;A;B;E;G;I;A;F;B;L;H;H;B;B;B;E;E;A;D;D;D;E;E;E;B;G;F;F;B;A;A;E;E;B;B;E;F;I;G;B;F;F;H;H;I;J;I;B;A;A;M;G;G;G;C;C;C;C;C;G;C;G;L;E;B;B;L;L;L;E;E;E;E;C;C;L;L;I;E;L;E;H;A;D;E;E;E;C;A;A;A;H;H;B;B;B;E;L;L;I;E;E;E;L;L;E;E;L;A;A;L;L;L;H;B;E;E;L;L;L;B;C;L;E;D;L;A;C;B;E;L;H;D;B;E;I;E;C;E;E;E;E;B;B;L;L;L;E;F;L;B;A;A;M;G;G;G;C;C;C;C;C;G;C;G;C;L;E;D;D;D;D;H;H;C;C;B;B;B;B;B;B;L;L;L;L;L;L;I;I;E;C;C;I;J;I;C;B;B;E;L;B;C;B;B;E;F;G;G;C;C;C;C;C;C;F;A;A;F;I;E;E;E;I;I;I;E;I;M;B;E;I;I;F;B;B;D;H;L;L;H;E;A;C;C;C;D;D;D;D;D;D;D;B;L;L;E;E;F;H;B;F;F;F;C;C;C;B;C;L;L;I;I;I;G;I;L;B;B;B;B;B;L;L;A;B;L;K;K;K;K;K;K;C;E;B;L;L;L;L;L;L;A;E;H;B;L;A;A;E;H;F;A;B;I;I;I;D;L;L;C;C;L;L;E;E;E;E;E;E;E;H;H;H;D;H;L;L;E;C;I;G;I;F;C;G;C;B;E;G;A;A;F;A;D;D;I;I;E;L;L;I;G;D;K;L;B;B;E;D;B;L;L;B;H;C;G;D;D;L;L;L;B;C;C;E;I;I;I;I;I;E;E;G;B;I;H;B;L;H;G;G;L;F;I;A;A;I;I;I;I;I;I;B;L;B;G;I;B;B;L;L;L;L;F;F;F;B;I;K;C;L;D;C;D;D;D;B;C;F;F;A;B;B;B;B;B;B;E;I;I;A;A;C;C;B;B;B;B;B;B;B;B;F;I;L;L;G;D;B;I;I;L;I;I;B;B;D;C;B;E;B;B;B;B;B;B;L;B;I;I;I;F;I;J;B;E;L;E;H;H;H;H;H;H;H;H;H;H;H;H;H;H;K;K;E;J;B;B;F;I;B;H;J;A;D;D;I;I;C;B;B;B;I;I;I;I;F;H;L;L;L;C;E;D;B;B;G;G;E;D;B;L;B;B;A;A;A;A;D;H;H;H;G;K;K;B;B;B;G;B;I;C;C;B;B;L;C;B;L;L;G;B;B;F;J;D;D;D;D;B;B;B;L;L;L;H;L;I;F;I;G;L;L;L;L;L;H;H;H;A;C;G;C;C;C;C;F;I;C;A;C;K;C;E;L;L;B;H;A;A;A;F;B;B;B;D;B;D;L;D;E;B;F;F;J;G;I;I;I;F;A;G;C;E;E;K;L;I;I;D;D;L;L;L;B;E;E;E;E;B;B;B;E;D;H;L;L;F;A;I;J;L;L;H;B;L;H;G;L;I;J;D;B;I;I;L;L;L;L;C;L;L;I;I;C;L;L;B;L;L;L;L;L;I;I;I;G;I;A;I;G;D;D;A;A;A;I;I;B;D;I;I;G;E;D;B;I;I;I;F;B;D;F;F;D;D;B;B;B;E;E;E;E;B;B;A;D;L;A;C;C;F;C;E;E;G;G;C;D;D;D;D;L;L;C;C;C;H;I;I;J;B;B;A;H;L;A;A;A;A;F;L;I;I;L;G;D;E;L;L;F;B;D;B;B;I;I;C;D;M;B;B;E;H;G;J;B;B;I;A;I;G;F;F;B;G;L;B;D;B;B;B;B;B;K;I;F;F;L;L;L;D;E;L;K;C;C;D;D;L;E;G;G;G;C;B;D;E;E;E;B;E;L;A;A;A;C;E;E;E;H;I;E;E;L;L;I;F;I;F;I;A;A;D;I;D;B;B;I;B;B;I;I;E;E;L;E;E;C;E;C;B;L;L;L;G;G;L;E;L;L;B;B;F;L;I;E;B;A;A;A;A;A;I;I;I;I;I;J;G;E;G;C;H;A;E;A;B;B;I;I;B;D;B;L;I;L;B;I;A;L;L;L;L;F;B;H;I;B;K;B;I;H;B;B;A;A;J;B;B;A;A;A;I;L;H;H;H;H;B;C;B;A;A;A;A;A;D;L;G;B;F;B;B;B;H;E;C;C;C;C;B;H;B;L;L;A;I;I;I;A;B;E;E;J;B;E;E;E;E;L;E;E;A;K;K;K;A;A;L;L;E;E;F;F;F;F;D;H;E;L;L;E;E;G;G;H;I;I;I;I;F;I;I;I;B;B;B;B;L;F;A;A;E;G;C;M;E;C;C;B;B;E;G;H;G;L;L;L;L;L;L;L;F;F;C;B;L;I;L;L;I;A;G;C;E;E;E;C;A;B;B;L;E;C;I;L;I;C;F;G;L;A;B;G;H;L;B;E;E;L;L;L;L;L;B;B;I;C;D;L;E;C;C;B;E;E;C;L;D;D;A;E;C;B;B;B;B;A;F;A;A;A;H;H;H;B;C;C;B;D;C;F;F;A;E;E;E;F;F;L;E;I;C;B;I;I;L;L;F;B;A;B;I;D;B;B;B;A;B;B;B;L;L;A;A;E;C;C;L;L;L;L;L;L;K;A;I;C;C;B;G;B;A;A;G;G;G;L;L;B;B;A;A;A;A;A;I;I;I;I;I;B;B;D;L;L;H;C;B;B;A;D;E;B;I;G;F;F;B;D;L;B;A;I;I;I;I;I;E;E;C;C;C;C;B;H;H;E;L;E;J;E;F;D;D;I;C;C;H;B;G;B;C;C;A;L;A;C;D;B;B;B;B;B;B;I;I;A;L;I;H;I;C;E;E;E;L;B;E;C;B;B;H;F;B;B;B;A;A;A;A;I;I;G;B;B;C;H;B;B;G;A;G;I;D;D;D;E;G;C;C;C;C;C;B;B;E;E;E;E;E;I;K;K;D;G;E;E;E;I;E;E;D;I;I;I;I;I;E;E;A;C;C;C;F;D;C;B;E;J;I;L;F;F;I;I;B;G;K;K;B;B;E;E;L;L;L;L;L;D;G;B;L;I;C;C;L;C;A;I;I;I;H;E;C;C;B;L;F;F;F;G;B;B;I;G;C;B;B;K;K;L;L;L;L;I;I;I;C;I;B;B;I;I;G;F;F;F;F;F;F;F;F;L;I;I;L;G;L;L;H;H;B;B;L;E;B;D;A;F;G;B;B;J;B;G;C;C;C;C;D;D;D;D;D;D;D;D;D;D;L;L;F;E;F;E;B;A;B;B;A;A;A;A;A;I;I;I;I;I;G;I;I;B;L;I;E;E;H;H;B;A;B;B;B;B;E;E;E;E;E;E;E;E;E;J;G;I;I;I;E;E;L;D;F;H;H;D;G;E;B;A;G;C;A;B;B;B;B;L;I;E;E;H;H;H;F;B;D;D;B;B;B;E;E;E;G;C;G;L;B;B;E;G;J;B;A;B;E;I;B;D;C;C;G;C;C;D;D;E;E;E;E;I;G;C;D;D;I;B;G;B;D;D;D;C;I;I;F;B;E;B;B;B;B;C;C;C;I;F;C;E;E;F;B;B;B;I;E;B;A;E;A;I;D;D;C;F;L;L;L;L;L;L;L;L;C;L;I;B;E;A;A;L;A;A;A;D;D;E;B;B;I;C;C;L;J;B;D;E;E;A;A;L;L;L;L;I;C;D;L;L;B;B;B;A;H;H;A;B;B;B;B;B;D;D;H;A;L;H;B;E;D;D;D;L;L;L;E;A;E;E;E;C;C;B;F;F;E;E;B;B;E;E;J;D;B;I;I;I;F;K;B;L;D;D;B;C;C;B;G;B;F;B;C;C;C;C;C;C;C;B;B;B;H;H;H;H;B;D;D;B;B;B;E;E;E;F;I;B;E;E;G;B;E;E;H;H;F;B;D;A;B;B;L;I;E;E;E;E;E;E;C;L;L;C;E;E;E;I;I;I;B;L;L;L;L;E;E;H;I;B;D;D;D;E;E;E;H;B;J;B;B;B;B;E;B;L;C;B;B;L;L;L;L;H;H;E;B;B;B;L;L;D;D;L;L;L;E;D;L;B;L;L;L;C;C;B;B;B;B;F;B;B;G;F;F;F;B;E;L;H;B;B;F;F;B;L;A;F;D;D;E;E;B;E;A;K;K;C;E;C;C;C;C;C;C;D;D;D;D;D;M;D;D;A;C;B;L;L;L;L;L;L;E;E;B;E;I;A;A;H;I;I;E;B;C;C;C;C;L;L;L;L;L;L;F;E;E;A;I;I;L;D;E;E;E;E;E;F;G;F;G;G;I;A;J;G;D;L;A;C;B;E;E;E;A;C;C;C;F;F;F;F;F;F;H;H;H;H;B;D;D;B;B;B;E;E;E;B;B;B;J;I;L;E;G;G;F;F;F;A;A;A;A;A;A;I;I;B;I;D;B;F;L;L;L;L;L;C;B;B;L;I;I;F;C;A;E;A;A;D;F;H;L;I;J;B;I;C;E;E;E;E;B;B;F;E;C;C;C;H;L;L;C;C;C;C;C;C;C;C;D;I;H;H;C;C;F;K;A;A;A;C;B;B;E;E;E;C;C;C;A;J;B;F;G;I;C;B;A;I;L;D;B;B;C;L;I;I;H;B;L;L;A;I;B;E;E;I;I;C;C;C;C;B;C;L;L;A;A;L;B;B;B;B;A;I;I;I;I;I;I;I;G;B;B;I;F;B;A;C;C;B;F;B;H;L;I;B;L;L;H;B;I;I;L;L;L;E;G;B;L;E;C;B;B;G;B;L;H;L;L;L;L;K;A;G;D;D;A;A;A;B;I;I;I;L;J;L;L;L;L;L;L;E;E;L;C;B;L;I;D;D;D;D;E;E;E;I;C;G;C;A;F;F;L;D;K;K;K;D;D;D;D;D;E;E;K;K;B;I;F;L;L;I;I;A;L;L;H;C;C;C;B;B;A;B;H;D;I;G;A;G;E;L;I;I;I;I;H;B;L;L;K;B;D;B;B;B;E;B;B;E;D;C;K;B;H;G;B;E;I;J;D;B;I;H;I;I;B;B;I;I;E;E;E;E;E;H;D;B;B;H;H;E;H;B;G;C;I;H;H;C;C;C;C;B;B;F;B;B;D;F;G;C;F;D;D;H;E;F;E;B;G;F;D;B;B;C;L;B;D;L;H;L;A;A;G;G;C;C;G;G;F;F;B;G;I;I;B;B;A;A;A;A;A;B;K;K;B;L;I;I;I;I;A;B;L;I;B;K;B;A;D;M;G;G;G;C;C;C;C;C;G;C;G;G;E;H;K;B;L;L;H;L;L;B;I;B;B;L;L;H;H;C;B;E;B;B;B;G;C;G;G;L;L;L;L;L;L;L;L;F;D;D;D;D;D;E;E;L;E;E;G;D;B;B;B;B;L;E;F;B;L;L;L;L;L;I;I;I;C;B;B;L;B;I;I;J;B;C;C;E;C;L;H;G;G;B;D;B;L;E;I;B;B;C;D;B;A;L;L;C;G;B;C;B;B;H;D;D;D;I;I;L;L;L;L;L;L;E;E;C;F;D;A;A;C;G;D;H;L;F;B;C;B;C;C;L;I;I;I;I;D;L;L;E;F;D;F;C;C;C;G;C;C;C;G;D;D;D;D;D;D;D;D;D;B;B;B;B;B;B;F;K;K;K;K;K;K;K;K;K;C;B;D;D;D;D;B;D;D;D;D;H;L;L;L;E;E;E;H;H;B;D;D;E;F;A;D;D;D;D;L;H;F;H;F;L;L;L;L;L;I;I;I;I;G;F;G;L;A;E;E;B;A;B;E;L;L;F;F;F;F;I;B;B;B;E;E;E;H;A;H;I;I;B;B;G;D;C;F;B;G;B;L;B;L;D;B;B;D;E;G;C;B;A;L;I;I;D;I;D;B;I;F;I;I;C;H;H;C;B;I;B;C;C;B;B;M;H;H;H;H;G;F;F;D;E;E;B;B;B;B;E;C;L;H;F;A;A;B;G;F;C;F;E;A;B;B;B;B;B;C;E;C;E;I;I;I;G;C;B;B;E;G;K;B;J;L;L;I;I;H;I;E;B;B;K;B;B;B;B;B;I;D;L;B;H;C;F;L;L;L;L;H;L;C;C;C;C;B;F;G;F;B;G;G;A;A;B;B;B;F;I;C;C;C;C;D;A;I;I;G;B;B;B;B;H;H;H;F;H;H;H;H;F;G;I;I;A;A;A;A;A;C;C;C;C;C;B;I;I;I;I;F;H;B;D;D;D;D;B;I;C;B;J;H;I;E;E;E;B;D;D;D;D;D;D;H;H;H;H;H;H;H;C;L;B;B;B;F;B;E;E;I;B;B;L;G;B;L;B;K;L;L;F;F;I;I;I;F;G;B;B;B;E;G;L;B;L;G;K;K;K;I;I;I;H;G;G;D;I;G;A;C;C;D;E;C;H;H;H;A;A;A;A;L;A;L;F;F;F;F;G;B;B;B;B;B;H;H;H;H;H;C;A;A;L;L;L;C;C;C;D;I;I;E;L;E;E;E;E;E;E;E;I;L;B;L;C;B;I;C;B;B;D;E;A;I;L;I;F;F;B;E;E;A;A;A;A;M;L;L;L;L;L;L;A;H;H;H;H;H;G;F;F;F;F;F;A;A;F;B;F;J;A;A;B;L;I;I;I;H;A;B;K;B;H;F;G;B;I;I;G;F;H;L;I;I;I;I;B;B;B;B;I;I;B;E;D;A;A;A;L;I;B;E;J;F;F;F;I;I;L;A;B;L;L;L;G;D;A;L;L;F;B;B;B;I;G;E;G;B;L;L;H;C;I;I;I;G;F;F;C;A;A;A;H;H;H;E;J;G;C;L;I;G;B;L;L;L;H;I;I;I;I;I;C;C;F;C;D;L;L;L;E;F;F;L;B;C;C;F;G;L;L;A;G;G;I;H;G;I;H;F;B;B;B;B;B;B;L;L;L;I;L;B;B;A;G;A;A;A;A;L;A;C;E;E;B;B;I;L;L;L;L;L;L;I;E;G;C;E;G;B;D;C;L;A;H;A;A;A;B;H;C;D;M;B;B;F;F;C;C;L;L;E;E;E;E;E;E;E;H;H;H;B;B;L;K;E;L;G;A;A;E;E;I;H;L;L;L;A;H;B;B;D;A;B;I;I;I;H;B;G;C;D;L;F;E;B;G;F;B;H;L;L;G;B;F;C;B;D;I;I;I;H;I;J;L;H;C;C;B;B;B;B;B;B;L;L;L;L;L;L;I;I;G;E;E;I;I;I;F;L;G;A;A;L;E;B;L;A;A;A;B;E;I;I;H;H;B;I;C;C;H;H;D;D;B;B;L;F;L;A;E;E;F;E;L;D;B;C;C;L;L;I;L;L;K;I;B;C;L;L;E;A;H;B;B;B;B;A;D;D;I;I;A;A;D;I;F;C;C;B;B;B;B;B;F;F;L;L;F;F;H;C;I;I;D;B;B;K;K;K;K;K;B;F;I;I;I;L;K;K;E;E;E;H;H;G;B;B;D;H;D;L;L;B;B;L;I;B;I;I;G;B;L;L;L;I;B;C;C;D;D;H;H;I;E;I;J;B;B;E;B;E;F;B;I;I;L;J;L;K;K;F;I;H;H;D;D;D;D;D;D;D;D;D;E;E;E;E;E;E;E;H;D;A;C;C;I;G;I;D;D;L;E;I;G;D;L;L;B;B;B;G;B;K;A;B;E;I;I;I;E;L;L;L;C;C;C;C;C;G;G;C;B;D;B;B;C;B;B;C;F;F;F;F;L;I;B;F;L;L;L;B;F;F;I;I;A;A;G;G;C;C;G;G;F;F;E;G;B;B;I;I;F;B;C;B;B;L;L;L;L;I;A;C;E;G;B;B;B;B;A;I;L;B;L;B;B;C;G;B;L;H;C;L;E;E;E;E;E;I;I;I;L;H;D;D;D;D;D;E;E;E;E;E;K;I;H;D;B;B;E;B;E;I;I;C;D;D;L;G;I;L;A;I;D;C;F;L;I;I;I;I;L;E;L;E;H;G;C;C;C;I;E;L;L;L;L;H;G;B;E;E;E;B;E;E;E;F;F;G;C;C;E;B;F;F;J;L;A;D;C;C;D;B;L;G;L;L;L;E;L;L;B;B;B;L;I;G;E;F;G;G;C;C;C;B;E;C;C;G;G;I;I;E;A;C;C;I;I;I;L;A;A;H;A;A;L;H;K;K;K;K;A;G;B;L;L;L;L;L;L;L;A;A;C;B;B;E;E;A;B;I;I;L;A;C;C;I;J;L;J;E;H;G;F;L;H;H;L;L;L;L;L;L;L;H;H;B;L;B;L;L;A;F;B;I;I;F;B;F;H;D;D;L;L;I;B;F;B;B;I;B;B;I;A;B;J;F;F;B;I;F;F;I;A;A;A;B;F;E;B;I;B;A;C;C;C;D;D;D;D;D;D;D;B;L;L;E;E;F;H;I;B;I;H;D;D;D;E;D;D;G;I;C;H;A;E;E;C;B;D;B;B;I;I;I;I;A;B;A;D;D;F;E;B;D;D;D;B;I;I;I;I;G;D;D;E;B;F;I;B;B;C;B;I;D;L;L;E;E;F;F;A;A;C;E;E;E;E;E;F;J;A;E;D;I;A;B;C;C;L;I;I;I;C;D;C;C;C;B;B;L;L;E;I;C;B;L;L;L;B;L;L;L;I;E;E;E;E;E;A;G;E;E;E;C;C;L;L;L;D;F;F;G;D;B;B;B;B;M;A;A;A;G;L;F;C;B;L;E;B;C;G;B;L;L;E;F;I;I;E;G;F;B;B;I;J;B;C;C;B;D;D;E;A;A;B;A;G;B;B;B;B;E;G;L;L;L;L;L;L;L;E;F;F;F;A;C;C;F;L;H;K;K;K;K;K;K;K;K;K;K;C;D;D;D;B;B;L;L;L;L;L;L;L;L;H;B;I;I;G;I;F;B;B;G;K;K;B;H;H;E;B;I;A;B;E;G;B;B;K;K;L;L;L;E;E;A;L;C;C;C;C;C;B;I;H;H;B;L;L;L;H;B;C;K;C;C;I;H;H;D;D;B;B;B;L;E;E;F;A;B;A;I;F;C;D;D;D;D;B;B;E;C;B;E;C;C;C;D;D;D;D;D;M;D;D;D;D;F;F;H;C;I;I;I;B;B;B;H;D;D;L;C;B;B;L;L;L;L;B;B;B;F;D;B;C;L;L;L;I;G;A;B;L;A;I;G;H;E;B;E;E;E;I;E;D;B;L;B;B;E;E;E;E;K;H;B;D;A;A;I;E;I;I;I;F;H;C;G;M;K;H;G;L;I;K;A;B;C;C;D;L;H;F;I;I;I;I;B;F;F;D;H;L;D;B;A;B;B;H;K;B;I;L;E;E;L;I;L;E;E;I;B;B;L;D;L;A;B;A;A;A;A;G;B;B;B;D;D;C;B;A;I;D;D;D;E;E;H;A;A;A;A;A;I;C;B;B;B;A;A;H;H;F;F;F;L;A;B;K;F;F;D;B;I;B;I;B;C;B;L;L;H;B;I;I;A;F;G;B;B;B;B;L;L;L;H;H;B;B;F;E;B;C;C;H;C;D;B;L;L;L;L;L;E;E;E;E;A;B;B;C;B;B;B;B;A;A;A;A;B;G;G;F;F;F;E;E;E;E;L;A;J;F;B;B;F;J;F;E;F;B;A;B;L;B;B;G;F;B;B;G;G;G;C;B;B;E;E;E;D;D;D;D;D;K;I;C;L;L;E;E;E;A;I;B;J;E;H;F;I;I;D;L;L;H;E;C;C;B;B;L;L;H;I;I;B;I;C;C;C;D;D;B;B;B;B;B;B;E;E;C;B;L;I;E;B;A;A;A;E;E;E;E;E;F;F;F;F;E;G;G;A;C;I;F;I;K;C;H;G;G;B;D;B;B;L;E;J;E;E;F;C;D;F;F;F;H;L;H;D;D;B;B;A;E;I;B;G;E;H;L;I;L;I;I;I;B;D;L;H;G;G;E;F;F;E;C;I;I;I;I;C;I;E;D;C;C;B;L;L;L;L;F;L;L;G;I;F;M;M;L;I;E;E;C;D;A;B;I;B;B;B;B;A;A;B;I;I;I;B;B;B;I;I;J;I;E;E;L;B;A;B;B;L;E;D;I;G;F;A;A;F;C;B;E;E;G;B;E;G;C;C;C;D;E;L;L;L;L;E;H;E;E;A;I;I;L;I;L;L;I;B;E;E;B;E;E;A;A;B;A;A;A;I;A;E;E;E;H;B;L;L;E;I;I;I;L;E;E;F;F;F;I;J;J;B;B;L;B;E;B;F;C;E;B;E;E;A;A;B;B;I;F;F;D;F;H;F;B;B;B;B;B;G;I;F;C;E;H;B;B;D;D;G;B;L;L;L;H;H;C;J;B;H;I;C;E;E;F;F;A;C;F;F;H;B;H;C;D;B;B;L;L;L;L;E;A;I;J;L;E;C;G;A;I;I;I;E;G;E;L;E;J;D;D;L;L;B;L;A;E;E;B;A;A;B;B;B;B;B;H;I;F;I;I;L;I;A;A;A;A;D;H;H;H;G;I;B;L;E;C;B;G;D;D;L;C;F;F;G;C;C;G;M;M;F;B;C;L;L;L;L;L;L;E;E;B;D;L;D;J;E;F;F;F;L;L;H;L;L;A;A;I;I;E;A;B;H;H;H;H;B;D;D;B;B;B;E;E;E;K;F;H;C;C;B;B;B;B;B;B;L;L;L;L;L;L;I;I;F;F;B;G;B;D;D;I;A;L;I;K;K;L;L;L;E;E;C;B;A;D;H;G;E;E;E;E;E;E;E;B;B;B;B;B;I;J;B;L;E;E;I;B;B;B;L;F;B;H;E;E;B;B;B;D;K;K;K;K;K;K;D;B;I;H;I;G;I;I;C;L;L;L;B;L;L;G;F;F;L;B;B;E;G;L;G;C;C;C;F;L;A;B;B;B;E;E;E;L;L;L;E;I;E;D;H;I;A;I;B;L;L;I;I;L;L;L;H;B;B;I;F;C;C;C;G;C;C;C;G;D;D;D;D;D;D;D;D;D;B;B;B;B;B;B;F;D;E;F;A;F;B;B;B;G;B;L;A;B;I;I;B;E;B;A;B;B;L;L;I;I;I;D;L;B;A;H;E;B;B;F;L;B;E;E;E;H;H;B;I;I;B;B;B;B;B;F;L;L;L;E;E;L;L;E;H;B;L;L;I;B;L;I;L;H;B;B;B;L;L;E;E;D;C;F;H;L;L;L;L;A;E;B;A;A;C;E;F;I;F;C;H;G;G;C;I;A;G;B;B;B;E;G;D;D;A;A;A;B;E;E;E;A;I;I;I;D;F;C;C;C;C;C;C;D;D;D;D;M;M;D;D;I;A;B;F;G;B;G;C;I;B;B;B;B;D;E;I;G;L;L;L;D;D;D;D;A;B;D;I;J;B;B;B;K;C;L;L;L;L;I;H;H;H;H;E;I;B;B;K;A;L;L;B;B;A;I;B;L;J;M;M;B;B;C;H;H;B;B;E;E;B;B;B;A;F;F;B;E;D;H;I;H;K;M;M;B;D;D;E;B;B;B;B;I;I;I;I;F;E;A;B;B;A;A;A;A;A;B;L;L;H;D;A;C;C;F;I;I;A;A;G;C;A;I;H;H;H;H;H;H;E;H;C;H;F;I;I;F;L;H;G;C;C;C;C;E;I;L;H;L;I;B;F;B;B;B;B;I;E;F;B;G;D;D;D;F;B;B;J;K;K;A;A;A;E;E;E;L;B;B;L;I;F;C;L;E;I;A;E;B;D;L;H;L;E;L;L;L;L;E;L;I;C;B;A;B;L;B;B;B;B;B;L;L;F;I;L;C;C;A;D;L;I;B;I;E;E;L;B;C;L;I;I;E;A;A;A;A;A;G;B;B;B;E;F;I;E;C;H;G;A;B;H;C;C;B;B;B;B;B;B;L;L;L;L;L;L;I;I;E;E;B;C;C;B;L;I;I;E;B;C;C;H;H;E;C;B;B;C;B;I;I;G;L;L;A;F;F;F;F;F;C;D;D;L;B;B;B;I;G;B;I;I;B;H;H;H;H;D;H;L;A;G;I;G;B;B;J;I;E;L;E;L;L;I;H;L;G;B;E;H;H;L;E;E;E;E;D;I;I;F;L;H;H;I;H;E;E;E;G;H;K;I;C;C;C;C;C;C;C;C;B;B;F;F;A;A;I;D;L;L;E;E;E;E;B;B;C;H;F;F;E;C;K;B;B;L;E;B;L;D;L;E;E;B;A;I;I;I;L;E;I;D;A;A;I;I;A;L;C;I;I;K;C;D;D;D;D;M;L;L;L;L;L;L;L;E;D;B;B;E;E;E;A;A;D;C;G;M;L;I;I;G;E;E;E;B;G;L;L;L;L;L;H;H;H;B;B;L;E;A;B;G;B;B;I;I;B;J;L;B;E;E;I;H;A;A;C;E;I;I;C;H;I;H;I;I;I;I;I;B;C;A;L;L;B;I;C;C;F;F;A;C;B;G;B;L;E;H;I;H;H;H;H;H;H;H;H;H;H;H;H;H;H;E;G;B;K;E;H;I;D;D;D;A;A;A;A;D;L;L;H;J;E;D;D;M;B;I;D;L;L;I;F;D;G;G;L;D;L;L;C;C;C;C;E;E;E;L;L;D;D;E;E;B;E;D;D;F;F;C;D;L;L;L;L;L;L;B;G;C;A;A;A;I;C;B;E;A;I;I;I;D;L;L;A;A;L;L;L;L;L;L;E;E;I;G;H;H;F;F;L;I;B;I;L;I;L;K;K;L;L;E;L;E;E;A;A;A;A;B;B;B;B;B;L;I;I;I;F;C;E;H;B;I;I;I;I;D;L;L;L;L;D;I;E;C;I;L;L;F;A;B;I;C;F;F;F;F;L;I;D;I;I;I;I;I;B;A;G;G;B;B;F;A;A;A;D;D;D;L;B;I;J;L;H;E;G;C;B;B;I;A;L;L;L;B;F;F;F;H;I;B;F;F;G;I;I;B;H;H;H;F;D;D;D;B;B;B;E;E;E;I;D;D;H;B;I;A;B;B;E;E;C;C;C;C;B;B;B;B;B;B;L;L;C;B;L;I;E;E;E;E;K;K;A;A;A;E;E;E;C;D;L;F;B;D;L;E;E;L;L;I;G;B;B;B;B;L;L;L;H;H;J;B;C;D;L;L;L;K;E;E;E;E;D;C;B;L;L;H;E;E;F;L;L;E;B;E;E;E;E;F;B;G;B;L;L;I;E;E;K;G;G;H;I;B;E;E;E;E;E;F;E;B;B;B;B;B;I;G;A;B;C;C;B;B;B;C;A;I;F;E;E;I;G;B;B;A;A;L;E;K;I;B;B;G;G;I;A;E;E;E;E;E;E;E;E;E;E;E;E;E;L;I;H;H;C;M;H;B;B;J;L;A;A;A;I;B;B;B;B;B;B;B;G;C;E;D;D;D;G;B;A;A;D;E;E;E;E;E;F;F;F;F;E;A;L;L;E;E;B;K;I;E;E;E;E;B;E;E;C;F;B;B;B;B;F;F;F;I;L;L;L;B;B;G;B;I;L;L;L;L;I;I;I;F;I;E;E;E;C;F;L;L;E;E;E;L;I;I;L;L;F;C;H;B;B;L;E;K;K;E;E;C;C;D;D;D;D;D;M;D;D;D;D;F;F;H;C;C;F;B;B;L;I;F;F;B;B;B;I;H;L;B;F;F;H;B;C;F;F;F;B;B;B;B;L;L;L;L;L;L;F;C;E;E;E;E;J;J;J;C;H;F;F;I;D;L;A;A;A;I;I;E;L;L;L;C;K;K;K;I;I;I;H;L;F;F;F;F;F;B;H;F;L;L;L;A;A;H;A;B;B;I;A;B;I;C;A;B;B;L;L;L;L;E;E;H;I;E;E;I;B;C;C;L;L;L;L;L;L;L;L;L;L;I;H;D;B;B;H;H;F;I;G;A;L;L;D;E;B;H;A;A;C;H;L;L;L;B;I;F;F;H;H;G;L;L;H;F;F;I;C;C;D;D;D;D;D;M;D;D;D;D;F;F;H;E;C;C;C;B;F;F;F;F;E;F;F;K;B;G;G;B;B;I;I;I;F;C;C;C;C;B;H;C;L;B;B;B;B;I;L;L;E;D;D;L;E;C;K;K;K;F;I;L;L;L;D;G;H;D;L;L;A;H;A;A;L;L;E;B;I;I;L;F;H;G;A;A;G;B;I;I;L;I;I;F;F;D;A;A;C;C;C;B;D;D;L;L;E;E;L;H;H;H;L;L;L;L;L;L;F;F;H;E;J;I;I;B;B;I;L;C;E;E;E;H;B;B;L;L;F;E;E;B;E;F;F;K;H;F;H;I;H;J;I;J;I;C;G;H;H;A;B;C;B;B;L;A;E;D;D;I;L;B;G;D;B;E;E;C;F;F;H;H;E;E;L;B;I;E;C;L;L;L;H;H;C;I;B;B;B;I;F;A;A;A;I;H;C;C;C;A;E;G;I;B;D;D;D;D;I;I;J;L;E;B;B;F;C;L;H;B;B;L;L;H;L;A;E;I;F;F;I;B;E;B;I;H;H;C;C;B;B;B;B;B;B;L;L;L;L;L;L;I;I;C;D;D;D;D;I;B;B;G;G;G;G;B;C;L;H;A;E;G;F;B;E;L;I;F;L;A;J;H;F;I;I;I;K;A;F;F;H;H;F;F;F;I;I;F;I;L;I;E;G;A;B;B;B;C;C;L;L;C;A;I;C;F;H;H;F;F;I;B;B;B;B;A;A;I;D;B;D;D;D;D;D;B;L;C;L;C;C;I;A;B;B;I;I;F;I;I;D;J;J;F;F;B;L;L;L;L;C;H;H;H;C;C;D;D;L;L;L;C;D;D;D;D;D;D;D;L;L;L;L;L;L;L;A;D;E;C;C;D;D;D;D;D;M;D;D;D;D;F;F;H;C;C;I;B;F;H;H;H;H;H;H;H;H;H;H;H;H;H;H;H;B;H;L;E;C;F;F;C;L;L;L;H;G;G;B;I;B;G;I;F;G;F;G;B;L;E;L;F;F;B;L;L;L;L;H;E;E;E;E;F;I;D;E;E;A;E;E;F;F;B;L;B;G;C;B;B;B;I;C;A;B;B;I;B;B;I;I;F;F;F;H;H;L;L;F;G;F;G;I;I;B;G;I;B;C;L;I;B;I;C;C;L;L;L;L;L;L;L;L;L;L;B;B;I;I;I;M;G;G;G;B;B;B;B;F;F;E;L;L;L;I;E;E;J;B;A;A;I;C;B;B;L;L;L;L;B;I;B;B;D;D;H;H;H;H;F;B;A;H;G;E;E;E;E;D;B;L;C;D;M;B;B;B;H;I;F;F;F;G;A;G;L;K;E;L;F;B;I;H;H;H;H;I;I;I;I;I;F;F;B;G;C;B;I;F;F;B;E;I;F;I;F;E;E;I;I;B;L;H;H;F;F;C;B;B;L;F;B;D;A;M;G;G;G;C;C;C;C;C;G;C;G;E;K;A;E;E;C;F;F;L;C;J;A;B;B;B;I;I;I;H;H;L;E;B;I;L;B;L;A;A;I;B;K;E;H;H;B;B;B;I;C;E;E;I;A;H;C;D;D;E;E;B;D;K;A;I;H;A;C;G;C;C;G;B;I;L;A;A;H;B;I;I;L;L;L;L;L;B;B;D;J;A;A;F;J;B;B;I;B;F;F;B;D;D;D;D;D;D;B;I;D;G;C;C;D;A;A;B;L;E;E;A;A;A;M;F;A;C;D;F;C;L;L;A;B;B;H;G;F;F;F;E;E;A;A;H;F;B;G;L;H;B;A;C;C;C;C;B;H;L;A;A;A;F;B;I;L;L;L;L;L;C;D;D;L;D;E;B;L;B;E;E;E;E;E;E;H;B;B;B;I;A;A;A;B;K;E;B;L;I;F;B;I;I;I;G;C;L;L;I;I;I;E;C;G;I;L;E;C;C;D;D;B;L;K;D;C;C;C;C;E;E;E;A;G;B;B;B;B;I;D;C;E;D;E;I;F;F;B;L;L;L;L;B;A;L;E;B;I;B;I;I;B;B;B;F;L;L;L;E;L;L;E;E;A;D;A;C;E;D;A;E;H;H;K;K;G;G;G;C;G;H;K;L;G;L;L;L;L;L;L;L;H;H;H;H;G;D;D;E;E;E;L;L;A;H;H;E;H;A;C;B;H;L;E;I;H;E;E;C;L;L;L;L;L;L;L;L;L;L;L;F;I;B;I;D;A;A;A;A;A;L;L;L;L;L;B;C;E;E;G;C;G;C;K;K;K;E;F;H;H;H;B;C;E;I;H;C;D;D;D;L;L;L;L;C;C;L;L;E;E;E;E;E;E;E;H;H;H;A;A;I;A;D;F;B;B;G;G;B;I;C;D;F;C;B;C;D;L;I;C;C;L;L;I;D;D;E;I;C;C;C;C;C;C;D;D;D;D;D;M;D;D;B;C;H;E;E;G;B;B;B;L;I;E;E;E;E;C;C;D;D;E;E;E;E;E;G;I;D;I;I;A;I;I;B;B;L;D;B;E;A;E;L;L;L;L;L;C;C;F;F;C;E;L;L;E;B;I;A;A;C;B;B;I;E;B;F;F;C;H;C;B;B;L;L;G;L;B;C;C;D;D;D;D;D;M;D;D;D;D;F;F;H;L;L;L;L;A;A;H;I;J;I;E;E;E;D;D;B;C;L;K;I;E;F;B;B;A;H;J;B;I;I;B;A;L;G;F;C;I;I;B;L;L;H;H;L;C;E;E;E;D;D;D;B;L;L;L;L;L;B;C;C;C;C;C;G;G;C;B;D;B;G;E;I;E;E;C;G;C;I;L;L;L;L;L;L;L;L;F;B;F;I;D;B;I;C;B;C;B;L;L;H;E;J;F;L;E;A;H;D;C;C;B;B;M;H;H;H;H;G;C;C;C;C;D;D;D;D;D;D;D;D;D;D;L;L;F;L;L;C;A;C;C;L;L;L;C;E;E;B;B;F;G;G;I;I;F;F;F;E;I;F;F;J;A;B;C;C;B;B;B;A;C;C;I;B;L;B;L;L;L;L;C;C;C;D;D;H;H;B;I;I;C;L;D;G;B;C;C;L;L;L;L;L;H;H;E;G;C;B;B;C;C;L;F;A;A;I;I;I;B;L;A;A;B;K;A;A;I;F;B;A;B;L;L;L;E;E;B;B;I;E;E;F;F;B;B;I;L;L;D;L;F;B;E;C;F;H;H;C;E;E;E;C;L;L;L;K;E;E;B;L;H;E;D;G;I;I;I;F;F;E;B;L;L;L;B;L;E;E;B;L;L;B;I;E;H;F;D;E;B;L;B;B;C;I;I;I;D;F;C;B;L;L;L;K;K;K;K;K;K;K;K;K;K;C;D;D;D;B;B;L;L;L;L;L;L;L;L;H;A;G;B;B;F;F;H;I;L;L;I;A;C;D;B;B;B;B;B;B;I;I;F;F;B;L;I;I;I;H;I;H;K;C;C;C;D;I;I;I;I;L;L;L;E;C;I;E;I;I;I;E;H;E;E;D;D;D;D;D;E;E;L;E;E;B;L;L;L;L;L;C;A;I;B;E;E;E;A;B;C;G;G;A;A;E;K;C;C;C;D;I;I;I;I;B;C;B;B;L;C;D;E;L;F;E;E;E;A;I;C;L;L;L;L;L;F;G;J;C;B;B;L;L;L;L;E;I;A;B;C;E;E;E;F;F;B;E;B;C;C;D;D;F;G;G;B;A;A;A;A;I;I;G;D;B;I;E;B;B;B;B;B;B;E;B;E;H;G;C;H;G;F;H;K;L;B;I;I;I;I;I;G;F;F;B;G;E;H;C;B;A;C;C;B;F;C;D;L;I;B;L;H;G;C;D;B;L;E;H;D;D;F;I;F;F;I;C;D;D;D;H;E;H;A;A;B;G;D;B;H;E;E;L;L;E;E;E;E;K;K;K;K;K;K;C;E;B;L;L;L;L;L;L;A;E;H;C;D;B;B;B;B;F;F;H;H;C;E;E;C;C;C;D;B;G;D;L;L;C;B;E;A;E;I;D;D;B;A;E;E;L;A;I;I;L;L;B;I;B;A;A;M;G;G;G;C;C;C;C;C;G;C;G;B;L;L;I;D;L;L;G;G;G;L;L;G;I;I;C;F;L;E;I;I;C;E;I;I;I;I;A;A;A;C;I;I;F;H;H;B;B;B;B;L;L;L;B;L;I;I;I;I;I;B;D;I;I;I;L;E;D;E;G;B;F;C;L;D;D;H;H;H;H;G;G;E;D;A;G;F;I;I;C;C;G;C;B;B;C;F;D;J;E;B;B;L;L;L;L;L;L;B;L;B;I;L;B;L;L;I;B;G;B;I;H;C;A;B;A;E;F;D;C;C;L;L;L;A;C;C;C;L;E;I;I;B;B;E;E;E;A;A;I;I;F;A;C;E;C;H;G;B;E;A;C;F;A;C;C;C;C;E;E;E;B;B;B;B;A;A;I;E;D;L;C;L;L;G;G;G;C;D;D;B;H;H;B;E;H;E;E;F;F;E;C;B;L;B;A;C;H;D;E;L;A;I;G;G;C;C;G;G;B;A;C;D;L;B;I;L;B;B;B;B;L;D;L;I;I;A;A;A;A;A;D;D;D;B;L;L;L;L;L;L;I;I;I;L;E;E;E;E;E;E;E;E;I;A;B;E;A;E;L;E;E;B;B;B;B;E;E;E;J;L;F;B;L;L;E;E;G;B;I;G;I;B;L;E;E;J;B;B;B;I;I;K;D;I;D;C;L;L;L;E;E;A;G;I;I;A;B;L;I;F;D;E;L;G;B;A;I;I;C;B;B;L;B;C;B;B;L;L;E;A;H;D;L;L;L;L;L;L;E;E;H;H;B;C;C;J;J;E;H;C;C;C;C;C;C;C;D;B;A;F;I;C;L;L;E;E;E;D;B;L;B;B;B;B;L;L;B;G;G;D;G;B;L;L;G;B;B;E;B;E;L;I;F;C;L;L;L;L;L;L;E;E;B;I;I;I;H;K;L;A;I;G;A;F;B;I;I;E;E;D;D;D;L;L;F;C;B;I;A;A;I;I;C;A;C;H;D;B;B;H;H;B;H;H;H;A;A;A;A;A;A;A;A;D;B;B;E;L;L;L;E;F;E;B;L;E;F;L;L;F;A;B;I;F;B;H;I;C;G;D;L;L;C;C;E;B;B;J;B;D;D;G;F;A;C;L;L;L;L;K;C;D;B;E;F;F;B;F;F;L;F;F;E;B;B;E;D;D;D;E;B;D;D;L;H;A;I;I;B;B;I;J;E;E;E;E;L;E;E;E;E;E;E;B;B;H;E;E;E;I;I;I;C;C;B;B;I;F;I;I;I;C;C;C;L;H;C;C;G;J;L;F;I;L;E;L;F;I;K;K;K;K;K;K;F;C;C;C;B;B;B;B;B;B;B;I;I;I;I;I;F;H;E;I;L;L;E;B;B;B;L;L;H;B;L;B;E;C;F;H;H;D;D;E;A;L;E;E;F;B;B;B;B;C;B;G;D;D;A;L;A;I;I;F;J;I;H;F;D;C;C;C;D;D;D;D;D;D;D;B;L;L;E;E;F;H;F;I;A;L;L;L;L;L;L;L;L;L;L;F;F;B;L;L;L;L;D;B;H;B;E;B;D;J;I;C;C;G;C;L;E;E;C;A;L;I;C;C;L;A;A;A;A;A;L;A;I;I;B;D;B;I;A;A;B;A;A;A;G;B;B;F;F;J;C;C;L;L;D;B;I;C;G;E;E;E;E;J;J;C;E;B;I;I;B;B;B;B;B;I;A;I;B;E;E;I;A;L;H;B;B;B;H;D;B;B;F;I;I;B;L;E;K;L;C;E;E;E;H;I;I;B;I;I;F;F;D;D;G;F;F;F;E;E;A;D;C;E;E;E;J;E;E;J;E;E;C;B;B;L;E;J;J;H;H;G;D;D;D;D;D;L;L;E;A;B;H;H;H;A;A;A;A;D;K;K;G;G;G;C;I;B;L;L;H;H;C;C;G;D;D;L;B;B;B;I;I;G;F;G;C;C;B;I;I;I;I;D;H;I;B;F;I;C;C;B;E;F;B;D;B;D;L;E;B;L;L;L;L;A;E;C;C;E;E;B;B;B;L;H;H;H;H;I;D;H;G;G;B;E;F;G;M;A;A;L;L;L;L;I;H;H;A;B;B;E;A;G;B;B;B;B;D;D;D;A;A;A;D;D;A;A;A;A;G;B;B;B;B;B;B;B;B;B;B;B;L;I;I;L;A;E;L;L;L;L;L;L;A;H;H;H;H;H;B;C;C;F;B;L;C;I;C;B;B;B;F;E;B;I;E;I;I;I;I;G;B;F;F;F;C;L;L;L;L;L;I;A;A;G;G;C;C;G;G;F;F;A;B;B;B;H;E;G;G;C;C;K;K;A;K;B;I;E;G;C;L;L;I;F;F;F;J;E;F;L;L;H;B;G;D;H;H;H;H;H;L;C;C;L;B;B;B;B;B;B;I;I;F;L;L;L;L;L;H;I;E;D;C;L;A;L;G;B;D;B;I;B;B;E;B;B;G;I;D;H;E;I;E;E;E;E;E;B;I;I;G;B;F;L;I;I;F;F;E;G;C;B;H;B;L;A;G;C;C;L;L;L;L;I;I;A;L;L;A;A;A;A;A;B;L;I;H;F;F;F;F;F;F;F;F;B;E;A;G;G;B;G;I;I;I;E;E;E;E;E;A;C;C;C;C;B;H;H;C;C;D;D;H;H;C;L;L;F;I;L;E;D;D;D;L;L;L;A;B;D;B;L;B;A;H;L;L;E;B;A;A;M;G;G;G;C;C;C;C;C;G;C;G;B;D;D;E;E;B;I;B;D;E;H;C;C;C;C;B;B;I;B;B;K;K;K;K;K;K;C;E;B;L;L;L;L;L;L;A;E;H;C;E;E;E;H;B;L;L;H;H;B;B;E;D;E;B;C;L;L;G;I;A;A;E;F;B;B;B;E;I;C;D;L;I;I;I;I;L;A;E;B;D;D;E;E;E;E;E;B;C;C;C;C;G;D;D;D;D;D;D;D;D;D;D;K;B;I;H;B;I;B;I;C;G;G;G;E;H;A;A;A;A;B;I;D;C;C;C;J;B;L;L;E;E;E;E;D;L;E;F;H;K;B;F;B;I;I;L;L;G;C;C;C;C;C;B;B;E;E;E;E;E;A;A;C;D;B;B;B;B;B;B;I;I;B;I;B;D;B;C;A;E;E;C;D;D;L;L;C;D;F;D;E;B;L;B;E;E;E;I;J;H;H;I;A;F;F;C;C;B;B;I;F;C;C;C;C;G;D;D;D;D;D;D;D;D;D;D;G;C;A;B;B;L;L;E;E;E;H;K;L;I;I;C;D;D;C;D;F;I;A;L;B;A;E;I;B;G;D;L;F;B;G;G;E;L;G;B;F;C;C;C;C;C;G;C;G;C;D;B;B;L;I;I;B;E;B;L;L;B;B;B;B;G;C;C;E;E;B;B;B;B;F;F;B;I;B;J;B;C;I;I;F;B;L;H;C;C;C;H;C;D;D;D;D;B;E;E;E;A;G;E;E;J;C;C;D;D;D;A;D;L;L;I;C;C;C;C;L;L;E;E;E;B;J;F;A;A;A;D;L;E;D;K;B;I;B;F;D;D;D;L;E;E;A;F;C;C;C;B;I;E;B;I;C;C;D;I;D;D;D;D;D;E;E;L;E;E;H;H;H;C;D;D;L;B;B;I;I;J;J;L;I;G;G;E;E;I;F;A;B;C;G;C;E;G;G;B;G;L;L;L;L;L;K;B;J;C;B;F;C;I;I;B;B;L;G;B;B;B;B;L;L;L;H;H;E;G;B;J;L;A;E;E;E;E;E;F;B;B;I;E;E;C;L;L;L;I;I;I;I;F;A;A;A;C;D;L;D;D;G;B;K;A;I;F;E;E;E;E;G;A;A;E;I;D;A;G;B;B;G;E;A;J;C;H;H;H;K;C;B;J;F;F;F;B;I;C;F;A;A;G;G;C;C;G;G;F;F;K;I;L;L;E;E;E;E;F;B;H;H;E;E;E;D;L;L;E;B;D;B;L;H;H;H;H;A;B;L;G;E;E;E;L;E;F;F;F;F;G;C;E;F;I;B;F;F;F;A;E;E;H;I;I;I;A;E;B;C;B;E;D;L;K;G;G;H;B;L;L;I;H;L;B;B;L;A;B;B;H;B;B;E;E;D;L;E;L;L;M;B;B;B;G;G;C;C;G;G;B;A;E;H;C;L;L;L;L;D;E;L;I;F;L;M;G;G;G;A;A;B;B;B;A;I;G;G;C;C;B;B;C;C;F;L;L;L;L;L;L;L;F;F;F;F;L;L;B;B;A;B;B;L;L;I;D;H;E;I;I;I;I;B;C;J;B;E;E;L;C;C;D;B;L;L;G;B;L;L;I;I;C'.split(';')))


In [186]:
pd.to_datetime(6461807460000, unit='ms')

Timestamp('2174-10-07 10:31:00')

In [187]:
class_code_mapper = {

    "AmpC_Producers": "A",
    "Non_ESBL_Enterobacterales": "B",
    "Enterococcus_VRE": "C",
    "ESBL_Enterobacterales": "D",
    "MRSA": "E",
    "Low_Significance": "F",
    "Enterococcus_VSE": "G",
    "Other_NonFermenters": "H",
    "MSSA": "I",
    "Streptococcus_pneumoniae": "J",
    "Acinetobacter": "K",
    "Pseudomonas": "L",
    "Carbopenam Resistant Enterobacterales": "M"
}

In [188]:
reverse_class_code_mapper = {v: k for k, v in class_code_mapper.items()}

In [189]:
mapper_df["final_class"] = mapper_df["mapped_letter"].map(reverse_class_code_mapper)

In [190]:
mapper_df

,index,stay_id,org_name,storetime,AST_PATTERN,mapped_letter,final_class
0,0,30000153,ENTEROBACTER CLOACAE,2174-10-07 10:31:00,"['ENTEROBACTER CLOACAE', 'TRIMETHOPRIM/SULFA', 'S']['ENTEROBACTER CLOACAE', 'GENTAMICIN', 'S']['ENTEROBACTER CLOACAE', 'TOBRAMYCIN', 'S']['ENTEROBACTER CLOACAE', 'CEFTAZIDIME', 'S']['ENTEROBACTER CLOACAE', 'CEFTRIAXONE', 'S']['ENTEROBACTER CLOACAE', 'CIPROFLOXACIN', 'S']['ENTEROBACTER CLOACAE', 'PIPERACILLIN', 'S']['ENTEROBACTER CLOACAE', 'CEFEPIME', 'S']['ENTEROBACTER CLOACAE', 'MEROPENEM', 'S']",A,AmpC_Producers
1,1,30000153,KLEBSIELLA PNEUMONIAE,2174-10-07 10:31:00,"['KLEBSIELLA PNEUMONIAE', 'CEFAZOLIN', 'S']['KLEBSIELLA PNEUMONIAE', 'TRIMETHOPRIM/SULFA', 'S']['KLEBSIELLA PNEUMONIAE', 'GENTAMICIN', 'S']['KLEBSIELLA PNEUMONIAE', 'TOBRAMYCIN', 'S']['KLEBSIELLA PNEUMONIAE', 'CEFTAZIDIME', 'S']['KLEBSIELLA PNEUMONIAE', 'CEFTRIAXONE', 'S']['KLEBSIELLA PNEUMONIAE', 'CIPROFLOXACIN', 'S']['KLEBSIELLA PNEUMONIAE', 'AMPICILLIN/SULBACTAM', 'S']['KLEBSIELLA PNEUMONIAE', 'CEFUROXIME', 'S']['KLEBSIELLA PNEUMONIAE', 'PIPERACILLIN/TAZO', 'S']['KLEBSIELLA PNEUMONIAE', 'CEFEPIME', 'S']['KLEBSIELLA PNEUMONIAE', 'MEROPENEM', 'S']",B,Non_ESBL_Enterobacterales
2,2,30000484,ENTEROCOCCUS SP.,2136-01-21 11:25:00,"['ENTEROCOCCUS SP.', 'PENICILLIN G', 'R']['ENTEROCOCCUS SP.', 'AMPICILLIN', 'R']['ENTEROCOCCUS SP.', 'VANCOMYCIN', 'R']['ENTEROCOCCUS SP.', 'LINEZOLID', 'S']",C,Enterococcus_VRE
3,3,30003598,KLEBSIELLA PNEUMONIAE,2189-03-25 10:53:00,"['KLEBSIELLA PNEUMONIAE', 'CEFAZOLIN', 'R']['KLEBSIELLA PNEUMONIAE', 'TRIMETHOPRIM/SULFA', 'R']['KLEBSIELLA PNEUMONIAE', 'NITROFURANTOIN', 'R']['KLEBSIELLA PNEUMONIAE', 'GENTAMICIN', 'S']['KLEBSIELLA PNEUMONIAE', 'TOBRAMYCIN', 'S']['KLEBSIELLA PNEUMONIAE', 'CEFTAZIDIME', 'R']['KLEBSIELLA PNEUMONIAE', 'CEFTRIAXONE', 'R']['KLEBSIELLA PNEUMONIAE', 'CIPROFLOXACIN', 'R']['KLEBSIELLA PNEUMONIAE', 'AMPICILLIN/SULBACTAM', 'R']['KLEBSIELLA PNEUMONIAE', 'CEFUROXIME', 'R']['KLEBSIELLA PNEUMONIAE', 'PIPERACILLIN/TAZO', 'R']['KLEBSIELLA PNEUMONIAE', 'CEFEPIME', 'R']['KLEBSIELLA PNEUMONIAE', 'MEROPENEM', 'S']",D,ESBL_Enterobacterales
4,4,30003598,KLEBSIELLA PNEUMONIAE,2189-03-27 10:35:00,"['KLEBSIELLA PNEUMONIAE', 'CEFAZOLIN', 'R']['KLEBSIELLA PNEUMONIAE', 'TRIMETHOPRIM/SULFA', 'R']['KLEBSIELLA PNEUMONIAE', 'NITROFURANTOIN', 'R']['KLEBSIELLA PNEUMONIAE', 'GENTAMICIN', 'S']['KLEBSIELLA PNEUMONIAE', 'TOBRAMYCIN', 'S']['KLEBSIELLA PNEUMONIAE', 'CEFTAZIDIME', 'R']['KLEBSIELLA PNEUMONIAE', 'CEFTRIAXONE', 'R']['KLEBSIELLA PNEUMONIAE', 'CIPROFLOXACIN', 'R']['KLEBSIELLA PNEUMONIAE', 'AMPICILLIN/SULBACTAM', 'R']['KLEBSIELLA PNEUMONIAE', 'CEFUROXIME', 'R']['KLEBSIELLA PNEUMONIAE', 'PIPERACILLIN/TAZO', 'R']['KLEBSIELLA PNEUMONIAE', 'CEFEPIME', 'R']['KLEBSIELLA PNEUMONIAE', 'MEROPENEM', 'S']",D,ESBL_Enterobacterales
...,...,...,...,...,...,...,...
18544,18544,39999168,PSEUDOMONAS AERUGINOSA,2190-03-13 10:23:00,"['PSEUDOMONAS AERUGINOSA', 'GENTAMICIN', 'S']['PSEUDOMONAS AERUGINOSA', 'TOBRAMYCIN', 'S']['PSEUDOMONAS AERUGINOSA', 'CEFTAZIDIME', 'S']['PSEUDOMONAS AERUGINOSA', 'CIPROFLOXACIN', 'R']['PSEUDOMONAS AERUGINOSA', 'PIPERACILLIN/TAZO', 'S']['PSEUDOMONAS AERUGINOSA', 'CEFEPIME', 'S']['PSEUDOMONAS AERUGINOSA', 'MEROPENEM', 'S']",L,Pseudomonas
18545,18545,39999168,PSEUDOMONAS AERUGINOSA,2190-04-08 12:45:00,"['PSEUDOMONAS AERUGINOSA', 'GENTAMICIN', 'S']['PSEUDOMONAS AERUGINOSA', 'TOBRAMYCIN', 'S']['PSEUDOMONAS AERUGINOSA', 'CEFTAZIDIME', 'S']['PSEUDOMONAS AERUGINOSA', 'CIPROFLOXACIN', 'R']['PSEUDOMONAS AERUGINOSA', 'PIPERACILLIN/TAZO', 'S']['PSEUDOMONAS AERUGINOSA', 'CEFEPIME', 'S']['PSEUDOMONAS AERUGINOSA', 'MEROPENEM', 'S']",L,Pseudomonas
18546,18546,39999168,STAPH AUREUS COAG +,2190-03-13 10:23:00,"['STAPH AUREUS COAG +', 'ERYTHROMYCIN', 'R']['STAPH AUREUS COAG +', 'CLINDAMYCIN', 'R']['STAPH AUREUS COAG +', 'TRIMETHOPRIM/SULFA', 'S']['STAPH AUREUS COAG +', 'TETRACYCLINE', 'S']['STAPH AUREUS COAG +', 'GENTAMICIN', 'S']['STAPH AUREUS COAG +', 'OXACILLIN', 'S']['STAPH AUREUS COAG +', 'LEVOFLOXACIN', 'S']",I,MSSA

In [196]:
# hh.parq(mapper_df,'mapper_ab_org_llm_df_')

File saved at: mapper_ab_org_llm_df_27Feb26_1644.parquet


In [192]:
# mapper_df_x= hh.load_data('./parq/mapper_ab_org_llm_df_27Feb26_1609.parquet')

In [95]:
mapper_df= hh.load_data('./parq/mapper_ab_org_llm_df_27Feb26_1644.parquet')
hh.dxx(mapper_df, pid_col= 'stay_id')

7.8k Unique Patient IDs (7838)
Admission ID column not found.
7.8k Unique ICU Stay IDs (7838)
18.5k Rows, shape: (18549, 7)



,index,stay_id,org_name,storetime,AST_PATTERN,mapped_letter,final_class
dtype,int64,int64,object,datetime64[ns],object,object,object
NotNA | NA,18549 | 0,18549 | 0,18549 | 0,18549 | 0,18549 | 0,18549 | 0,18549 | 0
nunique,18549,7838,153,10891,3595,13,13
0,0,30000153,ENTEROBACTER CLOACAE,2174-10-07 10:31:00,"['ENTEROBACTER CLOACAE', 'TRIMETHOPRIM/SULFA', 'S']['ENTEROBACTER CLOACAE', 'GENTAMICIN', 'S']['ENTEROBACTER CLOACAE', 'TOBRAMYCIN', 'S']['ENTEROBACTER CLOACAE', 'CEFTAZIDIME', 'S']['ENTEROBACTER CLOACAE', 'CEFTRIAXONE', 'S']['ENTEROBACTER CLOACAE', 'CIPROFLOXACIN', 'S']['ENTEROBACTER CLOACAE', 'PIPERACILLIN', 'S']['ENTEROBACTER CLOACAE', 'CEFEPIME', 'S']['ENTEROBACTER CLOACAE', 'MEROPENEM', 'S']",A,AmpC_Producers
1,1,30000153,KLEBSIELLA PNEUMONIAE,2174-10-07 10:31:00,"['KLEBSIELLA PNEUMONIAE', 'CEFAZOLIN', 'S']['KLEBSIELLA PNEUMONIAE', 'TRIMETHOPRIM/SULFA', 'S']['KLEBSIELLA PNEUMONIAE', 'GENTAMICIN', 'S']['KLEBSIELLA PNEUMONIAE', 'TOBRAMYCIN', 'S']['KLEBSIELLA PNEUMONIAE', 'CEFTAZIDIME', 'S']['KLEBSIELLA PNEUMONIAE', 'CEFTRIAXONE', 'S']['KLEBSIELLA PNEUMONIAE', 'CIPROFLOXACIN', 'S']['KLEBSIELLA PNEUMONIAE', 'AMPICILLIN/SULBACTAM', 'S']['KLEBSIELLA PNEUMONIAE', 'CEFUROXIME', 'S']['KLEBSIELLA PNEUMONIAE', 'PIPERACILLIN/TAZO', 'S']['KLEBSIELLA PNEUMONIAE', 'CEFEPIME', 'S']['KLEBSIELLA PNEUMONIAE', 'MEROPENEM', 'S']",B,Non_ESBL_Enterobacterales
2,2,30000484,ENTEROCOCCUS SP.,2136-01-21 11:25:00,"['ENTEROCOCCUS SP.', 'PENICILLIN G', 'R']['ENTEROCOCCUS SP.', 'AMPICILLIN', 'R']['ENTEROCOCCUS SP.', 'VANCOMYCIN', 'R']['ENTEROCOCCUS SP.', 'LINEZOLID', 'S']",C,Enterococcus_VRE


In [ ]:
# Checks

In [199]:
mapper_df_x.loc[17000]

index                                                                                                                                                                                                                                                                                                                                                                                                             17000
stay_id                                                                                                                                                                                                                                                                                                                                                                                                        39224051
org_name                                                                                                                                                                                

In [200]:
mapper_df.loc[17000]

index                                                                                                                                                                                                                                                                                                                                                                                                             17000
stay_id                                                                                                                                                                                                                                                                                                                                                                                                        39224051
org_name                                                                                                                                                                                

In [273]:
hh.search_in_df(all_micro_icu_df,column='org_name',search_list=[['KLEBSIELLA']])[['stay_id','org_name','ab_name','interpretation']].org_name.value_counts()

org_name
KLEBSIELLA PNEUMONIAE                      20020
KLEBSIELLA OXYTOCA                          2619
KLEBSIELLA (RAOULTELLA) PLANTICOLA           127
KLEBSIELLA (RAOULTELLA) ORNITHINOLYTICA       20
Name: count, dtype: int64

### Mapping 

In [96]:
hh.dxx(all_ast_df)

4.7k Unique Patient IDs (4680)
6.0k Unique Admission IDs (5971)
7.8k Unique ICU Stay IDs (7838)
160.4k Rows, shape: (160393, 20)



,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los,microevent_id,chartdate,charttime,spec_type_desc,storetime,test_name,org_name,ab_name,interpretation,dilution_value,comments,combined_col
dtype,int64,int64,int64,object,object,datetime64[ns],datetime64[ns],float64,int64,datetime64[ns],datetime64[ns],object,datetime64[ns],object,object,object,object,float64,object,object
NotNA | NA,160393 | 0,160393 | 0,160393 | 0,160393 | 0,160393 | 0,160393 | 0,160393 | 0,160393 | 0,160393 | 0,160393 | 0,160393 | 0,160393 | 0,160301 | 92,160393 | 0,160393 | 0,160393 | 0,160393 | 0,152509 | 7884,91872 | 68521,160393 | 0
nunique,4680,5971,7838,14,14,7838,7838,7811,110133,8564,10774,36,10892,15,153,47,6,26,47,2085
1,10005817,20626031,32604416,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2132-12-15 09:29:01,2132-12-17 18:06:07,2.359097,2591,2132-12-12 00:00:00,2132-12-12 02:08:00,URINE,2132-12-17 14:25:00,URINE CULTURE,"STAPHYLOCOCCUS, COAGULASE NEGATIVE",ERYTHROMYCIN,R,8.000000,None,"['STAPHYLOCOCCUS, COAGULASE NEGATIVE', 'ERYTHROMYCIN', 'R']"
2,10005817,20626031,32604416,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2132-12-15 09:29:01,2132-12-17 18:06:07,2.359097,2592,2132-12-12 00:00:00,2132-12-12 02:08:00,URINE,2132-12-17 14:25:00,URINE CULTURE,"STAPHYLOCOCCUS, COAGULASE NEGATIVE",NITROFURANTOIN,S,16.000000,None,"['STAPHYLOCOCCUS, COAGULASE NEGATIVE', 'NITROFURANTOIN', 'S']"
3,10005817,20626031,32604416,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2132-12-15 09:29:01,2132-12-17 18:06:07,2.359097,2593,2132-12-12 00:00:00,2132-12-12 02:08:00,URINE,2132-12-17 14:25:00,URINE CULTURE,"STAPHYLOCOCCUS, COAGULASE NEGATIVE",TETRACYCLINE,S,2.000000,None,"['STAPHYLOCOCCUS, COAGULASE NEGATIVE', 'TETRACYCLINE', 'S']"


In [201]:
hh.dxx(mapper_df, pid_col='stay_id')

7.8k Unique Patient IDs (7838)
Admission ID column not found.
7.8k Unique ICU Stay IDs (7838)
18.5k Rows, shape: (18549, 7)



,index,stay_id,org_name,storetime,AST_PATTERN,mapped_letter,final_class
dtype,int64,int64,object,datetime64[ns],object,object,object
NotNA | NA,18549 | 0,18549 | 0,18549 | 0,18549 | 0,18549 | 0,18549 | 0,18549 | 0
nunique,18549,7838,153,10891,3595,13,13
0,0,30000153,ENTEROBACTER CLOACAE,2174-10-07 10:31:00,"['ENTEROBACTER CLOACAE', 'TRIMETHOPRIM/SULFA', 'S']['ENTEROBACTER CLOACAE', 'GENTAMICIN', 'S']['ENTEROBACTER CLOACAE', 'TOBRAMYCIN', 'S']['ENTEROBACTER CLOACAE', 'CEFTAZIDIME', 'S']['ENTEROBACTER CLOACAE', 'CEFTRIAXONE', 'S']['ENTEROBACTER CLOACAE', 'CIPROFLOXACIN', 'S']['ENTEROBACTER CLOACAE', 'PIPERACILLIN', 'S']['ENTEROBACTER CLOACAE', 'CEFEPIME', 'S']['ENTEROBACTER CLOACAE', 'MEROPENEM', 'S']",A,AmpC_Producers
1,1,30000153,KLEBSIELLA PNEUMONIAE,2174-10-07 10:31:00,"['KLEBSIELLA PNEUMONIAE', 'CEFAZOLIN', 'S']['KLEBSIELLA PNEUMONIAE', 'TRIMETHOPRIM/SULFA', 'S']['KLEBSIELLA PNEUMONIAE', 'GENTAMICIN', 'S']['KLEBSIELLA PNEUMONIAE', 'TOBRAMYCIN', 'S']['KLEBSIELLA PNEUMONIAE', 'CEFTAZIDIME', 'S']['KLEBSIELLA PNEUMONIAE', 'CEFTRIAXONE', 'S']['KLEBSIELLA PNEUMONIAE', 'CIPROFLOXACIN', 'S']['KLEBSIELLA PNEUMONIAE', 'AMPICILLIN/SULBACTAM', 'S']['KLEBSIELLA PNEUMONIAE', 'CEFUROXIME', 'S']['KLEBSIELLA PNEUMONIAE', 'PIPERACILLIN/TAZO', 'S']['KLEBSIELLA PNEUMONIAE', 'CEFEPIME', 'S']['KLEBSIELLA PNEUMONIAE', 'MEROPENEM', 'S']",B,Non_ESBL_Enterobacterales
2,2,30000484,ENTEROCOCCUS SP.,2136-01-21 11:25:00,"['ENTEROCOCCUS SP.', 'PENICILLIN G', 'R']['ENTEROCOCCUS SP.', 'AMPICILLIN', 'R']['ENTEROCOCCUS SP.', 'VANCOMYCIN', 'R']['ENTEROCOCCUS SP.', 'LINEZOLID', 'S']",C,Enterococcus_VRE


In [89]:
# hh.df_sample(mapper_df,by_col='final_class').head(5)

In [79]:
# hh.search_in_df(all_ast_df,column='org_name',search_list=[['influ']])

In [97]:
all_ast_dff=all_ast_df.merge(mapper_df,
    on=['stay_id','org_name','storetime'],
    how='inner')

In [80]:
# hh.df_sample(all_ast_df,by_col='stay_id',item=34236022)

In [102]:
hh.dx(all_ast_dff)

4.7k Unique Patient IDs (4680)
6.0k Unique Admission IDs (5971)
7.8k Unique ICU Stay IDs (7838)
160.3k Rows, shape: (160301, 24)



In [118]:
resp_index_ast_df= all_ast_dff.merge(index_ast_df,  on=['subject_id', 'hadm_id','stay_id','org_name','microevent_id'],
    how='inner')

In [106]:
all_ast_dff.columns


Index(['subject_id', 'hadm_id', 'stay_id', 'first_careunit', 'last_careunit',
       'intime', 'outtime', 'los', 'microevent_id', 'chartdate', 'charttime',
       'spec_type_desc', 'storetime', 'test_name', 'org_name', 'ab_name',
       'interpretation', 'dilution_value', 'comments', 'combined_col', 'index',
       'AST_PATTERN', 'mapped_letter', 'final_class'],
      dtype='object')

In [105]:
index_ast_df.columns

Index(['subject_id', 'hadm_id', 'stay_id', 'first_careunit', 'last_careunit',
       'intime', 'outtime', 'los', 'microevent_id', 'chartdate', 'charttime',
       'spec_type_desc', 'storetime', 'test_name', 'org_name', 'ab_name',
       'interpretation', 'dilution_value', 'comments', 'combined_col',
       'first_ast_time'],
      dtype='object')

In [119]:
hh.dxx(index_ast_df)

3.6k Unique Patient IDs (3588)
3.9k Unique Admission IDs (3882)
5.2k Unique ICU Stay IDs (5213)
51.7k Rows, shape: (51688, 21)



,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los,microevent_id,chartdate,charttime,spec_type_desc,storetime,test_name,org_name,ab_name,interpretation,dilution_value,comments,combined_col,first_ast_time
dtype,int64,int64,int64,object,object,datetime64[ns],datetime64[ns],float64,int64,datetime64[ns],datetime64[ns],object,datetime64[ns],object,object,object,object,float64,object,object,datetime64[ns]
NotNA | NA,51688 | 0,51688 | 0,51688 | 0,51688 | 0,51688 | 0,51688 | 0,51688 | 0,51688 | 0,51688 | 0,51688 | 0,51688 | 0,51688 | 0,51688 | 0,51688 | 0,51688 | 0,51688 | 0,51688 | 0,49495 | 2193,34314 | 17374,51688 | 0,51688 | 0
nunique,3588,3882,5213,14,14,5213,5213,5203,38682,3649,3882,29,3929,14,96,40,5,24,44,1320,3882
0,10005817,28661809,31316840,Medical/Surgical Intensive Care Unit (MICU/SICU),Medical/Surgical Intensive Care Unit (MICU/SICU),2135-01-03 21:55:32,2135-01-19 21:16:23,15.972812,2605,2135-01-08 00:00:00,2135-01-08 15:30:00,MRSA SCREEN,2135-01-12 10:47:00,MRSA SCREEN,STAPH AUREUS COAG +,ERYTHROMYCIN,R,nan,None,"['STAPH AUREUS COAG +', 'ERYTHROMYCIN', 'R']",2135-01-08 15:30:00
1,10005817,28661809,31316840,Medical/Surgical Intensive Care Unit (MICU/SICU),Medical/Surgical Intensive Care Unit (MICU/SICU),2135-01-03 21:55:32,2135-01-19 21:16:23,15.972812,2606,2135-01-08 00:00:00,2135-01-08 15:30:00,MRSA SCREEN,2135-01-12 10:47:00,MRSA SCREEN,STAPH AUREUS COAG +,CLINDAMYCIN,R,nan,None,"['STAPH AUREUS COAG +', 'CLINDAMYCIN', 'R']",2135-01-08 15:30:00
2,10005817,28661809,31316840,Medical/Surgical Intensive Care Unit (MICU/SICU),Medical/Surgical Intensive Care Unit (MICU/SICU),2135-01-03 21:55:32,2135-01-19 21:16:23,15.972812,2607,2135-01-08 00:00:00,2135-01-08 15:30:00,MRSA SCREEN,2135-01-12 10:47:00,MRSA SCREEN,STAPH AUREUS COAG +,TRIMETHOPRIM/SULFA,S,0.500000,None,"['STAPH AUREUS COAG +', 'TRIMETHOPRIM/SULFA', 'S']",2135-01-08 15:30:00


In [121]:
hh.dx(resp_index_ast_df)

3.6k Unique Patient IDs (3588)
3.9k Unique Admission IDs (3882)
5.2k Unique ICU Stay IDs (5213)
51.7k Rows, shape: (51688, 40)



In [122]:
hh.parq(resp_index_ast_df, 'resp_index_ast_df')

File saved at: resp_index_ast_df13Apr26_0549.parquet


In [124]:
resp_index_ast_df= hh.load_data('./parq/resp_index_ast_df13Apr26_0549.parquet')
hh.dx(resp_index_ast_df)

3.6k Unique Patient IDs (3588)
3.9k Unique Admission IDs (3882)
5.2k Unique ICU Stay IDs (5213)
51.7k Rows, shape: (51688, 40)



In [ ]:
pd.set_option('display.max_colwidth',None)

In [81]:
# all_micro_icu_df.groupby('org_name').stay_id.nunique().sort_values(ascending= False)[0:50]


In [88]:
# ast_df.org_name.unique()

In [86]:
# ast_df.org_name.value_counts().sort_values(ascending=False)[0:50].index

### The AST Time is should be the first ast taken in the first ICU stay of the patient
- It should be the taken in ICU
- It should be taken during the window of that stay for that disease

In [85]:
# ast_df.head()

In [83]:
# ast_df.columns

In [84]:
# resp_inf_cohort.head()